<a href="https://colab.research.google.com/github/bogdanparvu18/msc-graduate-project/blob/main/notebooks/phase3/phase3_01_terminology_mapping.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Phase 3 — Sector 1: Terminology Mapping and Evidence Contract

This notebook is a terminology preparation stage.

**Current scope:** discover and pin the published Phase 2 S8 source; retain its metadata integrity checks; derive the actual normalized classes; import source terminology; prepare traceable mapping candidates and reviewed synonyms; enrich the task/evidence/variant contract. No QA pairs, LLM calls, generation estimates or full QA audits are produced.


## 1. Configuration and verification of the published Phase 2 source

The RAG produced evidence in JSON is fetched separately from GitHub `main` using `RAG_GITHUB_CONFIG`. Its manifest supplies the expected file SHA-256 and byte count, file hashes and source URLs are recorded for every run. This JSON does not require a Drive input file.

The default Drive input folder is `<storage_root>/inputs/terminology`. The notebook looks for `mst30_tables.xls`, `cest_package.json` plus its referenced document, and one SNOMED CT ZIP. Discovery accepts `SnomedCT*.zip`, `SNOMED_CT*.zip`, and `SNOMED-CT*.zip` case-insensitively, including `SNOMED_CT_CA_RF2_Release_20260831.zip`. When an input is absent, its `downloads` entry is used; default downloaded filenames are `mst30_tables.xls`, `cest_package.json`, `cest_source.pdf`, and `SnomedCT_RF2.zip`. An explicit review override is optional; the default is the RAG evidence package fetched from GitHub. Exact Drive paths can be configured; an extracted RF2 directory needs an explicit path. If multiple matching files exist, set `snomed_rf2_path` to select the intended release; the notebook never picks one by filename order. Filename discovery locates an input; the existing hash verification and RF2 Snapshot parser still validate it.

For a protected source, configure `url_secret_name` with the name of a Colab Secret containing the authorized direct download URL. Optional `header_secret_names`, e.g. `{"Authorization": "SNOMED_AUTH_HEADER"}`, map a request header to a secret containing its full value. Alternatively provide a public URL directly under `url`. Credentials and signed URL query parameters are excluded from exported configuration. Existing Drive files never trigger secret lookup or external download. Missing access is reported as `source_configuration_required`.

`inputs/terminology/ontology_manifest.json` stores source versions, byte counts, hashes, acquisition status and reusable read identifiers. SNOMED entries include release date, edition tag, module IDs, language refset IDs and RF2 component paths/hashes; official MLDS package/version/file IDs are retained when present in the configured download endpoint. These identifiers are tied to an exact artifact hash. The immutable per-version references remain in `_source_registry/<source>/<sha256>.json`. Working copies go to the VM under `WORK/terminology_cache`; directories use a canonical manifest of all file paths/sizes/SHA-256 values. A new Colab VM reuses the Drive files and manifest, then verifies a fresh local copy.


In [1]:
from __future__ import annotations

# Dependency bootstrap: install only missing or below-minimum packages.
import sys
import site
import subprocess
import importlib
import importlib.util
import base64
from urllib.error import HTTPError, URLError
from importlib.metadata import PackageNotFoundError, version

requirements = (("pyarrow", "pyarrow", 14), ("pandas", "pandas", 2), ("xlrd", "xlrd", 2))
missing = []
for distribution, module, minimum in requirements:
    try:
        installed_major = int(version(distribution).split(".")[0])
    except PackageNotFoundError:
        installed_major = 0
    if importlib.util.find_spec(module) is None or installed_major < minimum:
        missing.append(f"{distribution}>={minimum}")
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
    site.addsitedir(site.getusersitepackages())
    importlib.invalidate_caches()

# All application imports are centralized here.
from collections import defaultdict, Counter
from dataclasses import dataclass
from datetime import date, datetime, timezone
from time import monotonic
from html import escape
from pathlib import Path, PurePosixPath
from typing import Any, Callable, Iterable, Mapping, Sequence
from urllib.parse import quote, unquote, urlparse, urlsplit, urlunsplit
from urllib.request import Request, urlopen, HTTPRedirectHandler, build_opener
import copy
import fnmatch
import gzip
import gc
import hashlib
import io
import json
import math
import os
import re
import shutil
import sqlite3
import tarfile
import tempfile

import pandas as pd
import xlrd
import csv
import unicodedata
import zipfile
import urllib.request
from contextlib import contextmanager
import pyarrow as pa
import pyarrow.parquet as pq
try:
    from IPython.display import display, Markdown, HTML
except ImportError:  # Local verification outside a notebook.
    display = print
    Markdown = HTML = str
try:
    from google.colab import drive, userdata
    IN_COLAB = True
except ImportError:
    drive = userdata = None
    IN_COLAB = False

print("Python:", sys.version.split()[0], "| pyarrow:", version("pyarrow"))
print("Colab:", IN_COLAB)


# Load the complete verified metadata corpus. No image staging or QA execution.
CODE_VERSION = "phase3-sector1-terminology-v1.4.6"
CONFIG = {
    "code_version": CODE_VERSION,
    "notebook_base_commit": "35ffcb93020f84fde4bab7896c1a75e6ea322f7e",
    "repository": "bogdanparvu18/msc-graduate-project",
    "ref": None,  # Discover the repository default branch; optionally override with a branch, tag or commit.
    "work_root": "/content/mmvqa_phase3_sector1" if IN_COLAB else os.environ.get("MMVQA_WORK_ROOT", "./mmvqa_phase3_sector1"),
    "storage_root": None,  # Use the verified S8 config unless storage was explicitly relocated.
    "mount_google_drive": True,
    "read_colab_secret": True,
    "demo_variants": None,  # Derive the declared same-cohort demo variants from S8 metadata.
    "save_reports_to_drive": IN_COLAB,
    "metadata_batch_size": 8192,
    "progress": True,
    "expected_class_count": 14,
}

TERMINOLOGY_CONFIG = {
    "input_root": None,  # Google Drive: <S8 storage_root>/inputs/terminology.
    "mst_source_path": None,  # Drive path; otherwise discover mst30_tables.xls in input_root.
    "cest_package_path": None,  # Drive JSON; its source_file points to the original document.
    "snomed_rf2_path": None,  # Drive ZIP or extracted directory; discover SNOMED CT ZIP naming variants.
    "snomed_release_date": None,  # Resolve a unique date from RF2 Snapshot filenames.
    "snomed_edition": "",  # Optional label; identifiers/modules come from RF2.
    "snomed_language_code": "en",  # Language of the imported description component.
    "snomed_language_refset_ids": None,  # Discover all active refsets in that component; optional explicit subset.
    "snomed_component_paths": {},  # Explicit components if the archive is ambiguous.
    "mapping_review_path": None,  # Optional explicit override; otherwise use the frozen AI-assisted RAG review.
    "vm_copy_attempts": 3,  # Retry a failed transfer/hash check; reference stays fixed.
    # Existing Drive sources always take priority; only absent inputs are downloaded.
    # Signed URLs/tokens should be stored in Colab Secrets (or environment variables locally).
    "downloads": {
        "MST": {"kind": "xls", "url": "https://www.worldendo.org/assets-craft/pdf/resources/mst30_tables.xls"},
        "CEST_PACKAGE": {"kind": "json", "url": None, "url_secret_name": None, "header_secret_names": {}},
        "CEST_DOCUMENT": {"kind": "document", "url": None, "url_secret_name": None, "header_secret_names": {}},
        "SNOMED_CT": {"kind": "zip", "url": None, "url_secret_name": None, "header_secret_names": {}},
    },
    "save_to_drive": IN_COLAB,
}

RAG_GITHUB_CONFIG = {
    "repository": "bogdanparvu18/msc-graduate-project",
    "ref": "main",  # Read current main on every run; no RAG commit pin.
    "manifest_path": "outputs/phase3/sector1/terminology/rag_evidence_manifest.json",
    "download_attempts": 3,
}

WORK = Path(CONFIG["work_root"]).expanduser().resolve()
if IN_COLAB and (not WORK.is_relative_to(Path("/content")) or WORK.is_relative_to(Path("/content/drive"))):
    raise ValueError("work_root must be on the Colab VM under /content, outside the Drive mount")
RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ")
RUN_DIR = REPORT_DIR = WORK / "outputs" / "phase3" / "sector1" / "runs" / RUN_ID
RUN_DIR.mkdir(parents=True, exist_ok=True)
GITHUB_TOKEN = os.environ.get("GITHUB_TOKEN") or None
if not GITHUB_TOKEN and IN_COLAB and CONFIG["read_colab_secret"]:
    try:
        GITHUB_TOKEN = userdata.get("GITHUB_TOKEN") or None
    except Exception:
        print("Optional GitHub secret unavailable; using public access.")

DEFAULT_REPOSITORY = CONFIG["repository"]
S8_ROOT = "outputs/phase2/sector8"
INDEX_PATH = "configs/sector8_github_storage.json"
CATALOG_PATH = "configs/sector8_archive_state.json"
_BLOCK_SIZE = 1024 * 1024
_SHA256 = re.compile(r"[0-9a-f]{64}\Z")
_COMMIT = re.compile(r"[0-9a-f]{40}\Z")


class IntegrityError(ValueError):
    """The publication, its layout, or an artifact failed verification."""


def safe_relative_path(value: object) -> str:
    """Accept only canonical portable paths beneath a publication root."""
    if not isinstance(value, str) or not value or "\\" in value or ":" in value:
        raise IntegrityError(f"Invalid relative path: {value!r}")
    path = PurePosixPath(value)
    if (path.is_absolute() or any(p in {".", ".."} for p in path.parts)
            or path.as_posix() != value or value == "."
            or any(ord(c) < 32 for c in value)):
        raise IntegrityError(f"Unsafe or noncanonical relative path: {value!r}")
    return value


def file_digest(path: Path, on_bytes=None) -> tuple[int, str]:
    digest = hashlib.sha256()
    size = 0
    with path.open("rb") as stream:
        for block in iter(lambda: stream.read(_BLOCK_SIZE), b""):
            size += len(block)
            digest.update(block)
            if on_bytes is not None:
                on_bytes(len(block))
    return size, digest.hexdigest()


def verify_file(path: Path, size: int, sha256: str) -> None:
    actual_size, actual_hash = file_digest(path)
    if actual_size != size or actual_hash != sha256:
        raise IntegrityError(
            f"Integrity check failed for {path.name}: expected {size} bytes / "
            f"{sha256}; received {actual_size} bytes / {actual_hash}"
        )


def _size(value: object, field: str) -> int:
    if isinstance(value, bool) or not isinstance(value, int) or value < 0:
        raise IntegrityError(f"{field} must be a nonnegative integer")
    return value


def _hash(value: object, field: str) -> str:
    if not isinstance(value, str) or not _SHA256.fullmatch(value):
        raise IntegrityError(f"{field} must be a lowercase SHA-256")
    return value


class S8Store:
    """Verified S8 metadata loader.

    ``fetch`` is an optional transport taking ``(url, destination_path)`` and
    writing exact response bytes. Normal Colab runs fetch from GitHub directly.
    Credentials are kept only in memory and never included in snapshots.
    """

    def __init__(
        self,
        repo: str = DEFAULT_REPOSITORY,
        ref: str | None = None,
        cache_dir: str | Path = ".cache/mmvqa-phase3",
        *,
        token: str | None = None,
        fetch: Callable[[str, Path], None] | None = None,
        timeout: float = 120,
    ) -> None:
        if (not re.fullmatch(r"[A-Za-z0-9_.-]+/[A-Za-z0-9_.-]+", repo)
                or any(part in {".", ".."} for part in repo.split("/"))):
            raise ValueError("repo must have the form owner/repository")
        if ref is not None and (not isinstance(ref, str) or not ref or any(ord(c) < 32 for c in ref)):
            raise ValueError("ref must be None or a nonempty branch, tag, or commit")
        self.repo, self.ref = repo, ref
        self.selected_ref: str | None = None
        self.repository_default_branch: str | None = None
        self.resolved_branch: str | None = None
        self.cache_dir = Path(cache_dir).expanduser().resolve()
        self.token, self.fetch = token, fetch
        self.timeout = timeout
        self.commit: str | None = None
        self.index: dict = {}
        self.entries: dict[str, dict] = {}
        self.consumed: dict[str, dict] = {}
        self._index_identity: dict = {}
        self._root: Path | None = None

    def _download(self, url: str, destination: Path) -> None:
        parsed = urlparse(url)
        if self.fetch is not None:
            self.fetch(url, destination)
            return
        headers = {"User-Agent": "MMVQA-Phase3", "Accept": "application/vnd.github+json"}
        if self.token and parsed.netloc in {"api.github.com", "raw.githubusercontent.com"}:
            headers["Authorization"] = f"Bearer {self.token}"
        with urlopen(Request(url, headers=headers), timeout=self.timeout) as response:
            with destination.open("wb") as output:
                shutil.copyfileobj(response, output, _BLOCK_SIZE)

    def _raw_url(self, relative_path: str) -> str:
        path = safe_relative_path(relative_path)
        return f"https://raw.githubusercontent.com/{self.repo}/{self.commit}/{S8_ROOT}/{quote(path, safe='/')}"

    @staticmethod
    def _temp(parent: Path) -> Path:
        parent.mkdir(parents=True, exist_ok=True)
        fd, name = tempfile.mkstemp(prefix=".s8-", suffix=".tmp", dir=parent)
        os.close(fd)
        return Path(name)

    def _cache_path(self, namespace: str, relative_path: str) -> Path:
        assert self._root is not None
        relative_path = safe_relative_path(relative_path)
        base = self._root / namespace
        result = base.joinpath(*PurePosixPath(relative_path).parts)
        # Refuse cache symlinks that could redirect a publication outside its root.
        if not result.resolve().is_relative_to(self._root.resolve()):
            raise IntegrityError("Cache path escapes the pinned publication root")
        return result

    def resolve(self) -> str:
        """Discover the default branch if needed, then pin one immutable commit."""
        if self.commit is not None and self.entries:
            return self.commit
        self.cache_dir.mkdir(parents=True, exist_ok=True)
        temporary = self._temp(self.cache_dir)
        try:
            selected_ref = self.ref
            if selected_ref is None:
                self._download(f"https://api.github.com/repos/{self.repo}", temporary)
                repository = json.loads(temporary.read_bytes())
                default_branch = repository.get("default_branch") if isinstance(repository, dict) else None
                if (not isinstance(default_branch, str) or not default_branch
                        or any(ord(character) < 32 for character in default_branch)):
                    raise IntegrityError("GitHub repository metadata has no valid default_branch")
                self.repository_default_branch = self.resolved_branch = default_branch
                selected_ref = default_branch
            self.selected_ref = selected_ref
            self._download(
                f"https://api.github.com/repos/{self.repo}/commits/{quote(selected_ref, safe='')}",
                temporary,
            )
            result = json.loads(temporary.read_bytes())
            commit = result.get("sha") if isinstance(result, dict) else None
            if not isinstance(commit, str) or not _COMMIT.fullmatch(commit):
                raise IntegrityError("GitHub did not return an immutable 40-character commit SHA")
            if _COMMIT.fullmatch(selected_ref) and commit != selected_ref:
                raise IntegrityError("GitHub resolved a different commit than the requested SHA")
            self.commit = commit
            self._root = self.cache_dir / self.repo.replace("/", "__") / commit
            target = self._cache_path("publication", INDEX_PATH)
            target.parent.mkdir(parents=True, exist_ok=True)
            self._download(self._raw_url(INDEX_PATH), temporary)
            index = json.loads(temporary.read_bytes())
            entries = self._validate_index(index)
            byte_count, digest = file_digest(temporary)
            # Copy atomically in the target directory (cache can span mounts).
            staged = self._temp(target.parent)
            try:
                shutil.copyfile(temporary, staged)
                os.replace(staged, target)
            finally:
                staged.unlink(missing_ok=True)
            self.index, self.entries = index, entries
            self._index_identity = {
                "relative_path": INDEX_PATH, "bytes": byte_count, "sha256": digest,
            }
            return commit
        except Exception:
            self.commit = None
            self.selected_ref = self.repository_default_branch = self.resolved_branch = None
            self.entries = {}
            raise
        finally:
            temporary.unlink(missing_ok=True)

    @staticmethod
    def _validate_index(index: object) -> dict[str, dict]:
        if (not isinstance(index, dict) or type(index.get("schema_version")) is not int
                or index.get("schema_version") != 1):
            raise IntegrityError("Unsupported S8 storage-index schema")
        if index.get("mode") != "exact_byte_parts":
            raise IntegrityError("Unsupported S8 storage-index mode")
        catalog_hash = _hash(index.get("source_catalog_sha256"), "source_catalog_sha256")
        files = index.get("files")
        if not isinstance(files, list) or not files:
            raise IntegrityError("Storage index has no files")
        entries = {}
        for raw in files:
            if not isinstance(raw, dict):
                raise IntegrityError("Storage entry is not an object")
            entry = dict(raw)
            logical = safe_relative_path(entry.get("relative_path"))
            if logical in entries:
                raise IntegrityError(f"Duplicate logical storage path: {logical}")
            total = _size(entry.get("bytes"), "bytes")
            _hash(entry.get("sha256"), "sha256")
            storage = entry.get("storage")
            if storage == "direct":
                safe_relative_path(entry.get("github_relative_path"))
            elif storage == "parts":
                parts = entry.get("parts")
                if not isinstance(parts, list) or not parts:
                    raise IntegrityError(f"No parts declared for {logical}")
                for part in parts:
                    if not isinstance(part, dict):
                        raise IntegrityError("Part entry is not an object")
                    _size(part.get("ordinal"), "ordinal")
                parts = sorted(parts, key=lambda part: part["ordinal"])
                offset = 0
                for ordinal, part in enumerate(parts):
                    if part["ordinal"] != ordinal:
                        raise IntegrityError(f"Noncontiguous or duplicate part ordinals: {logical}")
                    if _size(part.get("offset"), "offset") != offset:
                        raise IntegrityError(f"Noncontiguous part offsets: {logical}")
                    safe_relative_path(part.get("relative_path"))
                    _hash(part.get("sha256"), "part sha256")
                    part_size = _size(part.get("bytes"), "part bytes")
                    if part_size == 0:
                        raise IntegrityError(f"Empty part in {logical}")
                    offset += part_size
                if offset != total:
                    raise IntegrityError(f"Part sizes do not match logical size: {logical}")
                entry["parts"] = parts
            else:
                raise IntegrityError(f"Unsupported storage kind for {logical}: {storage}")
            entries[logical] = entry
        if CATALOG_PATH not in entries or entries[CATALOG_PATH]["sha256"] != catalog_hash:
            raise IntegrityError("Storage index does not bind its source catalog")
        return entries

    def _ensure(self, target: Path, source: str, size: int, sha256: str) -> None:
        if target.is_file():
            try:
                verify_file(target, size, sha256)
                return
            except IntegrityError:
                pass  # Repair a stale or interrupted cache only with verified bytes.
        temporary = self._temp(target.parent)
        try:
            self._download(self._raw_url(source), temporary)
            verify_file(temporary, size, sha256)
            os.replace(temporary, target)
        finally:
            temporary.unlink(missing_ok=True)

    def load(self, logical_path: str) -> Path:
        """Return a verified logical file, reconstructing ordered parts as needed."""
        self.resolve()
        logical_path = safe_relative_path(logical_path)
        if logical_path not in self.entries:
            raise KeyError(f"Logical path is absent from the pinned S8 index: {logical_path}")
        entry = self.entries[logical_path]
        target = self._cache_path("logical", logical_path)
        valid = False
        if target.is_file():
            try:
                verify_file(target, entry["bytes"], entry["sha256"])
                valid = True
            except IntegrityError:
                pass
        if not valid:
            if entry["storage"] == "direct":
                self._ensure(target, entry["github_relative_path"], entry["bytes"], entry["sha256"])
            else:
                temporary = self._temp(target.parent)
                try:
                    with temporary.open("wb") as output:
                        for part in entry["parts"]:
                            cached = self._cache_path("parts", part["sha256"] + ".bin")
                            self._ensure(cached, part["relative_path"], part["bytes"], part["sha256"])
                            with cached.open("rb") as source:
                                shutil.copyfileobj(source, output, _BLOCK_SIZE)
                    verify_file(temporary, entry["bytes"], entry["sha256"])
                    os.replace(temporary, target)
                finally:
                    temporary.unlink(missing_ok=True)
        self.consumed[logical_path] = {
            "relative_path": logical_path, "bytes": entry["bytes"],
            "sha256": entry["sha256"], "storage": entry["storage"],
            **({"parts": [dict(p) for p in entry["parts"]]} if "parts" in entry else {}),
        }
        return target

    def read_json(self, logical_path: str) -> dict:
        value = json.loads(self.load(logical_path).read_bytes())
        if not isinstance(value, dict):
            raise IntegrityError(f"Expected a JSON object: {logical_path}")
        return value

    def catalog(self) -> dict:
        """Load the catalog selected by the verified index."""
        return self.read_json(CATALOG_PATH)

    def snapshot(self) -> dict:
        """Serializable provenance, including every consumed logical artifact."""
        return {
            "schema_version": 1,
            "repository": self.repo,
            "requested_ref": self.ref,
            "ref_selection": "repository_default_branch" if self.ref is None else "explicit_override",
            "selected_ref": self.selected_ref,
            "repository_default_branch": self.repository_default_branch,
            "resolved_branch": self.resolved_branch,
            "resolved_commit": self.commit,
            "s8_root": S8_ROOT,
            "storage_index": dict(self._index_identity),
            "resolution_status": "verified" if self.commit and self.entries else "not_resolved",
            "source_catalog_sha256": self.index.get("source_catalog_sha256"),
            "consumed_artifacts": [self.consumed[key] for key in sorted(self.consumed)],
        }

DEFAULT_VARIANTS = None  # Resolve from the verified S8 declaration, never a fixed cohort list.
_HASH = re.compile(r"[0-9a-f]{64}\Z")


def _canonical(value):
    return (json.dumps(value, sort_keys=True, indent=2, ensure_ascii=False,
                       allow_nan=False) + "\n").encode("utf-8")


def _identity(record):
    return {key: record.get(key) for key in ("relative_path", "sha256", "bytes")}


def _records(rows, description):
    if not isinstance(rows, list):
        raise IntegrityError(f"{description} must be a list")
    result = {}
    for record in rows:
        if not isinstance(record, dict):
            raise IntegrityError(f"{description} contains a non-object entry")
        path = safe_relative_path(record.get("relative_path"))
        if path in result:
            raise IntegrityError(f"Duplicate {description} path: {path}")
        size = record.get("bytes")
        if type(size) is not int or size < 0 or not _HASH.fullmatch(str(record.get("sha256"))):
            raise IntegrityError(f"Invalid {description} identity: {path}")
        result[path] = record
    return result


def _clean(value):
    if isinstance(value, float) and not math.isfinite(value):
        return None
    if isinstance(value, dict):
        return {key: _clean(item) for key, item in value.items()}
    if isinstance(value, (tuple, list)):
        return [_clean(item) for item in value]
    return value


def _decoded(row):
    result = _clean(row)
    for key in list(result):
        if key.endswith("_json") and isinstance(result[key], str):
            result[key[:-5]] = json.loads(result.pop(key))
    return result




class S8Benchmark:
    """Discover and verify the active S8 publication and its metadata contracts."""

    def __init__(self, store):
        self.store = store
        self.store.resolve()
        self.catalog = self.store.catalog()
        if (self.catalog.get("schema_version") != 1 or self.catalog.get("phase") != 2
                or self.catalog.get("sector") != 8 or self.catalog.get("complete") is not True):
            raise IntegrityError("The selected catalog is not a complete S8 publication")
        self.transport = _records(self.catalog.get("artifacts"), "transport artifact")
        self.logical = _records(self.catalog.get("logical_artifacts"), "logical artifact")
        signature = self.catalog.get("processing_signature")
        if not isinstance(signature, str) or not _HASH.fullmatch(signature):
            raise IntegrityError("S8 catalog has no valid processing signature")
        recipe_paths = [path for path in self.transport
                        if PurePosixPath(path).parts[-2:] == ("benchmark", "model_inputs.jsonl.gz")]
        if len(recipe_paths) != 1:
            raise IntegrityError("The S8 catalog must identify exactly one ordered recipe JSONL artifact")
        self.recipes_path = self._bind(recipe_paths[0])
        self.config_snapshot_path = self._bind(self.catalog.get("config_snapshot"))
        self.source_config = self.store.read_json(self.config_snapshot_path)
        self.summary_path = self._bind(self.catalog.get("summary"))
        self.summary = self.store.read_json(self.summary_path)
        config_variants = self.source_config.get("same_cohort_variants")
        summary_variants = self.summary.get("benchmark_coverage", {}).get("same_cohort_variants")
        if config_variants is not None and summary_variants is not None and config_variants != summary_variants:
            raise IntegrityError("S8 config and summary disagree on same-cohort variants")
        declared_variants = config_variants if config_variants is not None else summary_variants
        if (not isinstance(declared_variants, list) or not declared_variants
                or any(not isinstance(name, str) or not name for name in declared_variants)
                or len(declared_variants) != len(set(declared_variants))):
            raise IntegrityError("S8 metadata does not declare a valid same-cohort demo variant list")
        self.demo_variants = tuple(declared_variants)
        delivery = self.catalog.get("archive_delivery", {})
        if delivery.get("mode") != "tar_per_product_folder":
            raise IntegrityError("Unsupported S8 archive delivery mode")
        manifest_path = delivery.get("manifest_relative_path")
        self._bind(manifest_path)
        self.archive_manifest = self.store.read_json(manifest_path)
        manifest = self.archive_manifest
        semantic = self.transport.get("configs/sector8_state.json", {})
        self._bind("configs/sector8_state.json")
        if (manifest.get("schema_version") != 1 or manifest.get("mode") != "tar_per_product_folder"
                or manifest.get("source_catalog_sha256") != semantic.get("sha256")
                or manifest.get("logical_artifacts_sha256") != hashlib.sha256(
                    _canonical(self.catalog["logical_artifacts"])).hexdigest()):
            raise IntegrityError("Archive manifest belongs to a different logical S8 run")
        archives = manifest.get("archives")
        archive_records = _records(archives, "archive")
        if manifest.get("archive_count") != len(archive_records):
            raise IntegrityError("Archive count differs from the manifest")
        total_members = 0
        all_members = set()
        self.members = {}
        for archive in archives:
            if _identity(archive) != _identity(self.transport.get(archive["relative_path"], {})):
                raise IntegrityError("Archive identity differs from the transport catalog")
            members = _records(archive.get("members"), "archive member")
            if archive.get("member_count") != len(members):
                raise IntegrityError("Archive member count differs from the manifest")
            for path, record in members.items():
                if path in all_members or _identity(record) != _identity(self.logical.get(path, {})):
                    raise IntegrityError("Archive member differs from the logical catalog")
                all_members.add(path)
                self.members[path] = record
            total_members += len(members)
        if manifest.get("member_count") != total_members:
            raise IntegrityError("Total archive member count differs from the manifest")
        self._selected = {}
        self._tables = {}

    def _bind(self, path, descriptor=None):
        path = safe_relative_path(path)
        indexed = self.store.entries.get(path)
        declared = self.transport.get(path)
        if indexed is None or declared is None or _identity(indexed) != _identity(declared):
            raise IntegrityError(f"Storage index and catalog disagree: {path}")
        if descriptor is not None and _identity(descriptor) != _identity(declared):
            raise IntegrityError(f"Table descriptor and catalog disagree: {path}")
        if path in self.logical and _identity(self.logical[path]) != _identity(declared):
            raise IntegrityError(f"Logical and transport artifacts disagree: {path}")
        return path

    def _table(self, name, filters=None):
        descriptor = self.catalog.get("tables", {}).get(name)
        if not isinstance(descriptor, dict):
            raise IntegrityError(f"Missing S8 table descriptor: {name}")
        path = self._bind(descriptor.get("relative_path"), descriptor)
        schema_path = self._bind(descriptor.get("schema_path"))
        # Revalidate cached bytes whenever they are consumed; a filesystem cache
        # must not become a second, unchecked source of truth.
        local = self.store.load(path)
        if name not in self._tables:
            schema = self.store.read_json(schema_path)
            footer = pq.ParquetFile(local)
            if (footer.metadata.num_rows != descriptor.get("rows")
                    or footer.metadata.num_rows != schema.get("rows")
                    or footer.schema_arrow.names != schema.get("columns")):
                raise IntegrityError(f"Parquet row count or columns differ from S8 schema: {name}")
            self._tables[name] = local
        # Filtering avoids loading whole manifests into Python objects.
        return [_decoded(row) for row in pq.read_table(local, filters=filters).to_pylist()]

    @staticmethod
    def _validate_recipe(row):
        for key in ("example_id", "subject_id", "ground_truth_ref", "shared_source_group_id"):
            if not isinstance(row.get(key), str) or not row[key]:
                raise IntegrityError(f"Recipe has no {key}")
        items = row.get("ordered_inputs")
        if not isinstance(items, list) or not items or row.get("image_count") != len(items):
            raise IntegrityError("Recipe input list/image_count is inconsistent")
        if row.get("pixels_available") is not True:
            raise IntegrityError("Selected recipe has no complete pixel evidence")
        for order, item in enumerate(items):
            if type(item.get("input_order")) is not int or item["input_order"] != order:
                raise IntegrityError("Recipe input order must be contiguous and zero based")
            safe_relative_path(item.get("relative_path"))
            if item.get("storage") not in {"original", "derived"}:
                raise IntegrityError("Recipe has an unsupported image storage kind")
            if not _HASH.fullmatch(str(item.get("sha256"))):
                raise IntegrityError("Recipe image has no valid SHA-256")
            if not isinstance(item.get("artifact_id"), str) or not item["artifact_id"]:
                raise IntegrityError("Recipe image has no artifact identity")
            if not isinstance(item.get("image_role"), str) or not item["image_role"]:
                raise IntegrityError("Recipe image has no role")
            if any(type(item.get(key)) is not int or item[key] <= 0 for key in ("width", "height")):
                raise IntegrityError("Recipe image dimensions must be positive integers")
        if row.get("variant") == "temporal_multi":
            frames = [item.get("frame_index") for item in items]
            if (len(items) < 2 or any(type(frame) is not int or frame < 0 for frame in frames)
                    or any(left >= right for left, right in zip(frames, frames[1:]))
                    or any(type(item.get("is_target")) is not bool for item in items)
                    or sum(item["is_target"] for item in items) != 1):
                raise IntegrityError("Temporal inputs require increasing frames and exactly one target")
            for item in items:
                expected_role = "target" if item["is_target"] else "temporal_context"
                offset = item.get("time_offset_seconds")
                if (item["image_role"] != expected_role or isinstance(offset, bool)
                        or not isinstance(offset, (int, float)) or not math.isfinite(offset)):
                    raise IntegrityError("Temporal role/offset metadata is inconsistent")






class ColabS8Store(S8Store):
    def _download(self, url, destination):
        if CONFIG.get("progress", True):
            print("GitHub read:", urlparse(url).path, flush=True)
        return super()._download(url, destination)

    def load(self, logical_path):
        if logical_path not in self.consumed and CONFIG.get("progress", True):
            print("Verify metadata:", logical_path, flush=True)
        return super().load(logical_path)


def write_json(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    fd, temporary = tempfile.mkstemp(prefix=".phase3-", dir=path.parent)
    try:
        with os.fdopen(fd, "w", encoding="utf-8") as stream:
            json.dump(value, stream, indent=2, ensure_ascii=False, allow_nan=False)
            stream.write("\n")
        os.replace(temporary, path)
    finally:
        Path(temporary).unlink(missing_ok=True)



STORE = store = ColabS8Store(repo=CONFIG["repository"], ref=CONFIG["ref"],
                            cache_dir=WORK / "cache", token=GITHUB_TOKEN)
RESOLVED_COMMIT = STORE.resolve()
BENCHMARK = benchmark = S8Benchmark(STORE)
if CONFIG["demo_variants"] is None:
    CONFIG["demo_variants"] = list(BENCHMARK.demo_variants)
if CONFIG["storage_root"] is None:
    source_storage_root = BENCHMARK.source_config.get("storage_root")
    if not isinstance(source_storage_root, str) or not source_storage_root:
        raise IntegrityError("S8 config has no storage_root; set an explicit storage relocation in CONFIG")
    CONFIG["storage_root"] = source_storage_root
STORAGE_ROOT = Path(CONFIG["storage_root"]).expanduser()
if not STORAGE_ROOT.is_absolute():
    raise ValueError("The storage_root must be an absolute path")
ORIGINAL_ROOTS = [STORAGE_ROOT]
ARCHIVE_ROOT = STORAGE_ROOT / safe_relative_path(BENCHMARK.source_config.get("sector8_relative_root", S8_ROOT))
SOURCE_COMMIT_REPORT = {
    "requested_ref": STORE.ref,
    "ref_selection": "repository_default_branch" if STORE.ref is None else "explicit_override",
    "selected_ref": STORE.selected_ref,
    "repository_default_branch": STORE.repository_default_branch,
    "resolved_branch": STORE.resolved_branch,
    "resolved_commit": RESOLVED_COMMIT,
    "processing_signature": BENCHMARK.catalog["processing_signature"],
    "catalog_complete": BENCHMARK.catalog["complete"],
    "catalog_and_transport_verified": True,
    "all_metadata_loaded": False,
    "all_pixels_reverified_now": False,
    "training_ready": False,
}
write_json(RUN_DIR / "source_commit_check.json", SOURCE_COMMIT_REPORT)
write_json(RUN_DIR / "configuration.json", CONFIG)
print("Source ref:", STORE.selected_ref, "(repository default branch)" if STORE.ref is None else "(explicit override)")
print("Pinned S8 commit:", RESOLVED_COMMIT)
print("Active S8 processing signature:", BENCHMARK.catalog["processing_signature"])
print("Verified publication metadata. This terminology stage does not reload image pixels or generate QA.")

"""Load the current RAG evidence bundle from GitHub with manifest SHA-256 checks.

The manifest binds exact file bytes and parsed canonical content. Hashes verify
the declared repository artifact; they do not establish clinical validity.
"""


RAG_GITHUB_MANIFEST_SCHEMA = "mmvqa-rag-github-manifest-v1"
RAG_GITHUB_BUNDLE_SCHEMA = "mmvqa-rag-evidence-v1"


def _rag_github_path(value):
    if (not isinstance(value, str) or not value or "\\" in value
            or any(ord(char) < 32 for char in value)
            or any(part in {"", ".", ".."} for part in value.split("/"))
            or PurePosixPath(value).is_absolute()
            or PurePosixPath(value).as_posix() != value):
        raise ValueError("RAG GitHub paths must be safe repository-relative paths")
    return value


def _rag_github_sha(value, name):
    if not isinstance(value, str) or not re.fullmatch(r"[0-9a-f]{64}", value):
        raise ValueError(f"RAG manifest {name} must be a lowercase SHA-256 digest")
    return value


def _rag_github_json(payload):
    def pairs(items):
        result = {}
        for key, value in items:
            if key in result:
                raise ValueError("Duplicate key in RAG JSON document")
            result[key] = value
        return result

    def constant(value):
        raise ValueError("Non-finite number in RAG JSON document")

    return json.loads(payload.decode("utf-8"), object_pairs_hook=pairs, parse_constant=constant)


def _rag_github_validate_manifest(payload):
    if len(payload) > 1024 * 1024:
        raise ValueError("RAG GitHub manifest exceeds the supported size")
    manifest = _rag_github_json(payload)
    if not isinstance(manifest, dict) or manifest.get("schema_version") != RAG_GITHUB_MANIFEST_SCHEMA:
        raise ValueError("Unsupported RAG GitHub manifest schema")
    artifact = manifest.get("artifact")
    if not isinstance(artifact, dict):
        raise ValueError("RAG GitHub manifest requires one artifact record")
    _rag_github_path(artifact.get("path"))
    if type(artifact.get("bytes")) is not int or not 0 < artifact["bytes"] <= 32 * 1024 * 1024:
        raise ValueError("RAG artifact bytes must be a positive integer within 32 MiB")
    _rag_github_sha(artifact.get("sha256"), "artifact.sha256")
    _rag_github_sha(manifest.get("bundle_content_sha256"), "bundle_content_sha256")
    if artifact.get("schema_version") != RAG_GITHUB_BUNDLE_SCHEMA:
        raise ValueError("Unsupported RAG evidence schema in GitHub manifest")
    review_date = manifest.get("bundle_review_date")
    if not isinstance(review_date, str):
        raise ValueError("RAG manifest requires bundle_review_date")
    if date.fromisoformat(review_date).isoformat() != review_date:
        raise ValueError("RAG bundle_review_date must use YYYY-MM-DD")
    reviewed_commit = manifest.get("phase2_source_commit_at_review")
    if reviewed_commit is not None and (not isinstance(reviewed_commit, str) or not re.fullmatch(r"[0-9a-f]{40}", reviewed_commit)):
        raise ValueError("RAG review provenance requires an immutable Phase 2 commit")
    return manifest


def _rag_github_validate_bundle(payload, manifest):
    artifact = manifest["artifact"]
    if len(payload) != artifact["bytes"]:
        raise ValueError("RAG evidence byte count differs from the GitHub manifest")
    if hashlib.sha256(payload).hexdigest() != artifact["sha256"]:
        raise ValueError("RAG evidence SHA-256 differs from the GitHub manifest")
    bundle = _rag_github_json(payload)
    if not isinstance(bundle, dict) or bundle.get("schema_version") != artifact["schema_version"]:
        raise ValueError("RAG evidence JSON has an unsupported schema")
    if bundle.get("review_date") != manifest["bundle_review_date"]:
        raise ValueError("RAG evidence review date differs from the GitHub manifest")
    for key in ("sources", "passages", "profiles", "target_decisions"):
        if not isinstance(bundle.get(key), list):
            raise ValueError(f"RAG evidence requires a {key} list")
    canonical = json.dumps(bundle, ensure_ascii=False, sort_keys=True, separators=(",", ":"), allow_nan=False)
    if hashlib.sha256(canonical.encode("utf-8")).hexdigest() != manifest["bundle_content_sha256"]:
        raise ValueError("RAG evidence canonical content hash differs from the GitHub manifest")
    return bundle


def _rag_github_local_path(root, relative):
    path = root.joinpath(*PurePosixPath(relative).parts)
    if path.is_symlink() or not path.resolve().is_relative_to(root.resolve()):
        raise ValueError("RAG cache path escapes the repository/ref directory")
    return path


def _rag_github_temporary(parent):
    parent.mkdir(parents=True, exist_ok=True)
    fd, name = tempfile.mkstemp(prefix=".rag-", suffix=".partial", dir=parent)
    os.close(fd)
    return Path(name)


def load_github_verified_artifact(config, *, manifest_validator, artifact_validator):
    """Load current GitHub bytes with one shared retry/cache/integrity workflow.

    This function performs network and filesystem I/O through the supplied
    transport. The manifest and artifact validators remain separate pure
    functions. Artifact-specific cache paths, messages and provenance fields
    are configured by the compatibility wrappers below.

    Every attempt checks a fresh manifest before and after artifact validation;
    an unstable reference restarts the attempt without publishing its cache.
    """
    repository, ref = config["repository"], config["ref"]
    manifest_path, cache_root = config["manifest_path"], config["cache_root"]
    fetch, attempts = config["fetch"], config.get("attempts", 3)
    label, artifact_label = config["label"], config["artifact_label"]
    namespace = config.get("cache_namespace")
    if (not isinstance(repository, str) or not re.fullmatch(r"[A-Za-z0-9_.-]+/[A-Za-z0-9_.-]+", repository)
            or any(part in {".", ".."} for part in repository.split("/"))):
        raise ValueError(f"{label} repository must have the form owner/repository")
    if (not isinstance(ref, str) or not ref or len(ref) > 255
            or not re.fullmatch(r"[A-Za-z0-9._/-]+", ref)
            or any(part in {"", ".", ".."} for part in ref.split("/"))):
        raise ValueError(f"{label} GitHub ref must be a safe branch, tag, or commit name")
    if type(attempts) is not int or not 1 <= attempts <= 10:
        raise ValueError(f"{label} download attempts must be an integer from 1 to 10")
    if not callable(fetch):
        raise TypeError(f"{label} fetch must be a callable transport")
    manifest_path = _rag_github_path(manifest_path)
    base = Path(cache_root).expanduser().resolve()
    root = base / repository.replace("/", "__") / hashlib.sha256(ref.encode("utf-8")).hexdigest()[:16]
    if namespace:
        root = root.joinpath(*PurePosixPath(_rag_github_path(namespace)).parts)
    if not root.resolve().is_relative_to(base):
        raise ValueError(f"{label} cache root escapes its repository/ref directory")
    root.mkdir(parents=True, exist_ok=True)

    def raw_url(path):
        return f"https://raw.githubusercontent.com/{repository}/{quote(ref, safe='')}/{quote(path, safe='/')}"

    attempt_log = []
    artifact_downloads = 0
    manifest_downloads = 0
    for number in range(1, attempts + 1):
        temporary_paths = []
        log = {"attempt": number, "manifest_sha256": None, "confirmed_manifest_sha256": None,
               "artifact_sha256": None, "status": "started"}
        attempt_log.append(log)
        try:
            manifest_temp = _rag_github_temporary(root)
            temporary_paths.append(manifest_temp)
            manifest_downloads += 1
            fetch(raw_url(manifest_path), manifest_temp)
            manifest_payload = manifest_temp.read_bytes()
            log["manifest_sha256"] = hashlib.sha256(manifest_payload).hexdigest()
            manifest = manifest_validator(manifest_payload)
            artifact = manifest["artifact"]
            artifact_path = artifact["path"]
            if artifact_path == manifest_path:
                raise ValueError(f"{artifact_label} and its manifest must use distinct paths")
            log["artifact_sha256"] = artifact["sha256"]
            # Content-keyed caches cannot accidentally reuse an older main artifact.
            relative = "artifacts/" + artifact["sha256"] + "/" + PurePosixPath(artifact_path).name
            destination = _rag_github_local_path(root, relative)
            document = None
            if destination.is_file():
                try:
                    document = artifact_validator(destination.read_bytes(), manifest)
                except (OSError, ValueError, UnicodeError):
                    pass
            artifact_temp = None
            cache_status = "reused_verified" if document is not None else "downloaded_and_verified"
            if document is None:
                artifact_temp = _rag_github_temporary(destination.parent)
                temporary_paths.append(artifact_temp)
                artifact_downloads += 1
                fetch(raw_url(artifact_path), artifact_temp)
                document = artifact_validator(artifact_temp.read_bytes(), manifest)
            confirm_temp = _rag_github_temporary(root)
            temporary_paths.append(confirm_temp)
            manifest_downloads += 1
            fetch(raw_url(manifest_path), confirm_temp)
            confirmed_payload = confirm_temp.read_bytes()
            log["confirmed_manifest_sha256"] = hashlib.sha256(confirmed_payload).hexdigest()
            if confirmed_payload != manifest_payload:
                log["status"] = "manifest_changed_retry"
                continue
            # No candidate becomes the accepted cache until its reference is stable.
            if artifact_temp is not None:
                os.replace(artifact_temp, destination)
            manifest_destination = _rag_github_local_path(root, "manifests/" + log["manifest_sha256"] + ".json")
            manifest_destination.parent.mkdir(parents=True, exist_ok=True)
            os.replace(manifest_temp, manifest_destination)
            actual_sha = hashlib.sha256(destination.read_bytes()).hexdigest()
            if actual_sha != artifact["sha256"]:
                raise ValueError(f"{label} local artifact changed after verification")
            log["status"] = "verified"
            provenance = {
                "schema_version": config["provenance_schema"],
                "repository": repository, "repository_ref": ref,
                "manifest_path": manifest_path, "manifest_url": raw_url(manifest_path),
                "manifest_sha256": log["manifest_sha256"], "manifest_bytes": len(manifest_payload),
                "manifest_local_path": str(manifest_destination),
                "artifact_path": artifact_path, "artifact_url": raw_url(artifact_path),
                "artifact_bytes": artifact["bytes"], "source_sha256": artifact["sha256"], "vm_sha256": actual_sha,
                **{key: manifest.get(key) for key in config["provenance_fields"]},
                "local_path": str(destination), "cache_status": cache_status,
                "attempts_used": number, "manifest_downloads": manifest_downloads,
                "artifact_downloads": artifact_downloads, "attempt_log": attempt_log,
                "verification_status": "verified", "reference_stability_check": "manifest_identical_before_and_after_artifact_verification",
                "integrity_scope": "GitHub manifest reference and matching file bytes/content; not clinical validation or a pinned main commit",
            }
            return document, provenance
        except (OSError, ValueError, UnicodeError):
            log["status"] = "download_or_verification_failed"
        finally:
            for temporary in temporary_paths:
                temporary.unlink(missing_ok=True)
    raise RuntimeError(f"{artifact_label} at {repository}/{ref} could not be verified against a stable GitHub manifest after {attempts} attempts; the VM did not redefine the reference") from None


def load_github_rag_evidence(repository, ref, manifest_path, cache_root, fetch, attempts=3):
    """Return the RAG bundle and its existing provenance from the current ref."""
    return load_github_verified_artifact(
        {"repository": repository, "ref": ref, "manifest_path": manifest_path,
         "cache_root": cache_root, "fetch": fetch, "attempts": attempts,
         "label": "RAG", "artifact_label": "RAG evidence",
         "provenance_schema": "mmvqa-rag-github-source-v1",
         "provenance_fields": ("bundle_content_sha256", "bundle_review_date",
                               "phase2_source_commit_at_review")},
        manifest_validator=_rag_github_validate_manifest,
        artifact_validator=_rag_github_validate_bundle,
    )


# RAG inputs follow main on every run; the GitHub manifest supplies the expected SHA-256.
RAG_EVIDENCE_PACKAGE, RAG_SOURCE_PROVENANCE = load_github_rag_evidence(
    repository=RAG_GITHUB_CONFIG["repository"],
    ref=RAG_GITHUB_CONFIG["ref"],
    manifest_path=RAG_GITHUB_CONFIG["manifest_path"],
    cache_root=WORK / "rag_github_cache",
    fetch=STORE._download,
    attempts=RAG_GITHUB_CONFIG["download_attempts"],
)
write_json(RUN_DIR / "rag_source_provenance.json", RAG_SOURCE_PROVENANCE)
print("RAG source:", RAG_GITHUB_CONFIG["repository"], RAG_GITHUB_CONFIG["ref"])
print("RAG JSON SHA-256 on the VM matches the GitHub manifest.")
if RAG_EVIDENCE_PACKAGE.get("policy_version") != "concept-bound-mapping-v2":
    raise ValueError("This notebook requires the concept-bound-mapping-v2 RAG package and its matching GitHub manifest; rerun section 1 after publication.")
display(pd.DataFrame([{key: RAG_SOURCE_PROVENANCE[key] for key in (
    "repository_ref", "source_sha256", "vm_sha256", "verification_status", "cache_status", "attempts_used"
)}]))


Python: 3.13.16 | pyarrow: 23.0.1
Colab: True
GitHub read: /repos/bogdanparvu18/msc-graduate-project
GitHub read: /repos/bogdanparvu18/msc-graduate-project/commits/main
GitHub read: /bogdanparvu18/msc-graduate-project/fd5c5c1ab660bc1518d18b1500882ab458e6419f/outputs/phase2/sector8/configs/sector8_github_storage.json
Verify metadata: configs/sector8_archive_state.json
GitHub read: /bogdanparvu18/msc-graduate-project/fd5c5c1ab660bc1518d18b1500882ab458e6419f/outputs/phase2/sector8/metadata/_github_chunks/8513359d8f914f9ecbfb8946e657da604ecd84c9f3420a502e539477b0c83286.bin
GitHub read: /bogdanparvu18/msc-graduate-project/fd5c5c1ab660bc1518d18b1500882ab458e6419f/outputs/phase2/sector8/metadata/_github_chunks/14bbba0f9d01140cd25099568e46d87ba2b17d89492ffd904a4ea3e981b6f3d0.bin
Verify metadata: metadata/5d2f2d49883814ea4c8226b409c6139a125b95bd02cf3bbe29bcb5f83f80c865/configs/sector8_config.json
GitHub read: /bogdanparvu18/msc-graduate-project/fd5c5c1ab660bc1518d18b1500882ab458e6419f/outputs/p

,repository_ref,source_sha256,vm_sha256,verification_status,cache_status,attempts_used
0,main,1bce1d283eb6f204cb0e5b484766d9a59009195db52e42...,1bce1d283eb6f204cb0e5b484766d9a59009195db52e42...,verified,downloaded_and_verified,1


## 2. Load and cross-check the complete S8 metadata corpus

Retain the original seven-table contract, schema and row-count checks, annotation/source joins and recipe provenance verification. The full class inventory is derived from these verified annotations, not from existing QA or a manually maintained list. No physical image reread is claimed by this metadata check.


In [2]:
S8_TABLE_NAMES = (
    "source_evidence_manifest", "visual_manifest", "visual_source_manifest",
    "benchmark_ground_truth_manifest", "shared_source_groups",
    "skipped_products", "benchmark_input_manifest",
)


def s8_semantic_hash(value):
    """Use the same canonical JSON convention as the published S8 generator."""
    return hashlib.sha256(_canonical(value)).hexdigest()


def _frame_key(row):
    video, frame = row.get("video_key"), row.get("frame_index")
    if not isinstance(video, str) or not video:
        raise IntegrityError("Missing video identity")
    if isinstance(frame, bool) or not isinstance(frame, (int, float)) or not math.isfinite(frame) or frame != int(frame) or frame < 0:
        raise IntegrityError("Missing or invalid frame identity")
    return video, int(frame)


class Corpus:
    """Complete metadata corpus. Images are resolved separately and on demand.

    Tables are read once in bounded Arrow batches, then joined through indexes.
    Recipe JSONL is streamed per pass; ordered input lists are not duplicated in
    input_by_id. No same-cohort, bounding-box, class or task filter is applied.
    """

    def __init__(self, benchmark, batch_size=8192):
        if type(batch_size) is not int or batch_size < 1:
            raise ValueError("metadata_batch_size must be a positive integer")
        self.benchmark, self.store, self.batch_size = benchmark, benchmark.store, batch_size
        self.source_config = copy.deepcopy(benchmark.source_config)
        self.schemas, self.table_counts = {}, {}
        self.sources_by_path, self.visual_by_artifact = {}, {}
        self.truths_by_subject, self.truths_by_annotation, self.truths_by_ref = {}, {}, {}
        self.truths_by_frame, self.links_by_artifact = defaultdict(list), defaultdict(list)
        self.groups_by_id, self.group_for_path = {}, {}
        self.input_by_id, self._recipe_hashes = {}, {}
        self.recipe_artifact_ids, self.ordered_input_slots = set(), 0
        self.skipped_products = []
        self.available_variants_by_subject = defaultdict(set)
        declared = set(benchmark.catalog.get("tables", {}))
        if declared != set(S8_TABLE_NAMES):
            raise IntegrityError(f"S8 table contract changed: missing={set(S8_TABLE_NAMES)-declared}, additional={declared-set(S8_TABLE_NAMES)}")
        signature = benchmark.catalog["processing_signature"]
        self.summary = benchmark.summary
        if self.summary.get("complete") is not True or self.summary.get("training_ready") is not False:
            raise IntegrityError("S8 summary is not preparation-complete and explicitly untrained")
        self._load_sources()
        self._load_visuals()
        self._load_links()
        self._load_truths()
        self._load_groups()
        self.skipped_products = list(self._rows("skipped_products"))
        self._load_input_index()
        self._recipes_local = self.store.load(benchmark._bind(benchmark.recipes_path))
        self._recipes_descriptor = dict(self.store.entries[benchmark.recipes_path])
        recipe_count = self._validate_recipe_stream()
        self.source_snapshot = self.store.snapshot()
        self.validation_report = {
            "status": "complete_metadata_verified", "table_counts": dict(self.table_counts),
            "source_files": len(self.sources_by_path), "visual_artifacts": len(self.visual_by_artifact),
            "visual_artifacts_referenced_in_recipes": len(self.recipe_artifact_ids),
            "visual_artifacts_without_recipe": sorted(set(self.visual_by_artifact) - self.recipe_artifact_ids),
            "ordered_input_slots": self.ordered_input_slots,
            "subjects": len(self.truths_by_subject), "recipes": recipe_count,
            "variants": dict(sorted(Counter(r["variant"] for r in self.input_by_id.values()).items())),
            "subjects_with_all_demo_variants": sum(set(CONFIG["demo_variants"]).issubset(v) for v in self.available_variants_by_subject.values()),
            "unexplained_subjects_without_recipes": 0, "recipes_without_subjects": 0,
            "recipes_without_manifest_rows": 0, "unresolved_identity_contradictions": 0,
            "all_recipe_metadata_verified": True, "all_pixels_reverified_now": False,
            "pixel_authority": "Completed S8 snapshot; no image pixels are reverified in the terminology stage.",
            "training_ready": False,
        }
        expected = {
            "subjects": len(self.truths_by_subject), "benchmark_examples": recipe_count,
            "source_files_in_catalog": len(self.sources_by_path), "skipped": len(self.skipped_products),
        }
        for field, count in expected.items():
            if self.summary.get(field) != count:
                raise IntegrityError(f"Published S8 summary disagrees with loaded {field}: {self.summary.get(field)} != {count}")

    @property
    def all_subject_ids(self):
        return set(self.truths_by_subject)

    @property
    def all_recipe_ids(self):
        return set(self.input_by_id)

    def _rows(self, name):
        """Verify table bytes, footer, schema and every row without a full table copy."""
        descriptor = self.benchmark.catalog["tables"][name]
        relative = self.benchmark._bind(descriptor["relative_path"], descriptor)
        schema_path = self.benchmark._bind(descriptor["schema_path"])
        local = self.store.load(relative)
        schema = self.store.read_json(schema_path)
        parquet = pq.ParquetFile(local)
        if (parquet.metadata.num_rows != descriptor.get("rows")
                or parquet.metadata.num_rows != schema.get("rows")
                or parquet.schema_arrow.names != schema.get("columns")):
            raise IntegrityError(f"Parquet footer disagrees with schema/catalog: {name}")
        self.schemas[name] = schema
        count = 0
        for batch in parquet.iter_batches(batch_size=self.batch_size):
            for row in batch.to_pylist():
                count += 1
                yield _decoded(row)
        if count != schema["rows"]:
            raise IntegrityError(f"Incomplete table read: {name}")
        self.table_counts[name] = count
        if CONFIG.get("progress", True):
            print(f"Complete metadata table: {name}: {count:,} rows", flush=True)

    @staticmethod
    def _put_unique(index, key, row, description):
        if not isinstance(key, str) or not key or key in index:
            raise IntegrityError(f"Missing or duplicate {description}: {key!r}")
        index[key] = row

    def _load_sources(self):
        for row in self._rows("source_evidence_manifest"):
            path = safe_relative_path(row.get("image_relpath"))
            _hash(row.get("sha256"), "source SHA-256")
            if row.get("source_sha256") != row["sha256"]:
                raise IntegrityError(f"Source hash fields disagree: {path}")
            for dimension in ("width", "height"):
                value = row.get(dimension)
                if isinstance(value, bool) or not isinstance(value, (int, float)) or value != int(value) or value <= 0:
                    raise IntegrityError(f"Invalid source dimension: {path}/{dimension}")
                if row.get("source_" + dimension) != value:
                    raise IntegrityError(f"Source dimension fields disagree: {path}/{dimension}")
            _frame_key(row)
            if not isinstance(row.get("physical_frame_id"), str) or not row["physical_frame_id"]:
                raise IntegrityError(f"Missing physical-frame identity: {path}")
            self._put_unique(self.sources_by_path, path, row, "source path")

    def _load_visuals(self):
        for row in self._rows("visual_manifest"):
            aid = row.get("artifact_id")
            _hash(aid, "artifact identity")
            _hash(row.get("sha256"), "visual SHA-256")
            path = safe_relative_path(row.get("relative_path"))
            if row.get("storage") == "source":
                source = self.sources_by_path.get(path)
                if source is None or source["sha256"] != row["sha256"]:
                    raise IntegrityError(f"Original visual/source identity mismatch: {aid}")
                if any(row.get(k) != source[k] for k in ("width", "height")):
                    raise IntegrityError(f"Original visual/source dimensions disagree: {aid}")
            elif row.get("storage") == "derived":
                if row.get("complete") is not True:
                    raise IntegrityError(f"Incomplete S8 derivative record: {aid}")
                member = self.benchmark.members.get(path)
                if member is None or _identity(member) != _identity(row):
                    raise IntegrityError(f"Derived visual/archive identity mismatch: {aid}")
            else:
                raise IntegrityError(f"Unknown visual storage kind: {row.get('storage')}")
            self._put_unique(self.visual_by_artifact, aid, row, "visual artifact")

    def _load_links(self):
        for row in self._rows("visual_source_manifest"):
            aid, path = row.get("artifact_id"), row.get("image_relpath")
            source = self.sources_by_path.get(path)
            if aid not in self.visual_by_artifact or source is None:
                raise IntegrityError(f"Unresolved visual/source link: {aid}/{path}")
            if row.get("source_sha256") != source["sha256"]:
                raise IntegrityError(f"Provenance source SHA-256 mismatch: {aid}/{path}")
            for field, expected in (("source_width", source["width"]), ("source_height", source["height"]),
                                    ("physical_frame_id", source["physical_frame_id"])):
                if row.get(field) != expected:
                    raise IntegrityError(f"Provenance {field} mismatch: {aid}/{path}")
            if _frame_key(row) != _frame_key(source):
                raise IntegrityError(f"Provenance video/frame mismatch: {aid}/{path}")
            if not set(row.get("annotation_ids") or []).issubset(source.get("annotation_ids") or []):
                raise IntegrityError(f"Provenance annotations not present in source: {aid}/{path}")
            self.links_by_artifact[aid].append(row)
        if set(self.links_by_artifact) != set(self.visual_by_artifact):
            raise IntegrityError("Every visual must have source provenance")
        for aid, links in self.links_by_artifact.items():
            links.sort(key=lambda row: row["source_index"])
            if [row["source_index"] for row in links] != list(range(len(links))):
                raise IntegrityError(f"Duplicate or noncontiguous source order: {aid}")
            declared = self.visual_by_artifact[aid].get("source_refs")
            if declared is None and self.visual_by_artifact[aid]["storage"] == "source":
                # S8 original reference records omit source_refs; the dedicated
                # visual_source_manifest is authoritative for these references.
                if len(links) != 1 or links[0]["image_relpath"] != self.visual_by_artifact[aid]["relative_path"]:
                    raise IntegrityError(f"Original provenance coverage mismatch: {aid}")
            else:
                if not isinstance(declared, list) or len(declared) != len(links):
                    raise IntegrityError(f"Visual source_refs coverage mismatch: {aid}")
                for ref, link in zip(declared, links):
                    if any(link.get(key) != value for key, value in ref.items()):
                        raise IntegrityError(f"Visual source_refs differ from provenance links: {aid}")

    def _load_truths(self):
        for row in self._rows("benchmark_ground_truth_manifest"):
            sid, annotation, path = row.get("subject_id"), row.get("annotation_id"), row.get("image_relpath")
            source = self.sources_by_path.get(path)
            if source is None or annotation not in (source.get("annotation_ids") or []):
                raise IntegrityError(f"Annotation evidence lacks a verified source: {sid}")
            if _frame_key(row) != _frame_key(source):
                raise IntegrityError(f"Subject/source frame disagreement: {sid}")
            if sid != s8_semantic_hash({"annotation_id": annotation, "image_relpath": path}):
                raise IntegrityError(f"Subject identity hash mismatch: {sid}")
            if row.get("ground_truth_ref") != s8_semantic_hash({"source_annotation_evidence": sid}):
                raise IntegrityError(f"Ground-truth identity hash mismatch: {sid}")
            if row.get("question") is not None or row.get("answer") is not None:
                raise IntegrityError("S8 annotation evidence unexpectedly contains authored QA")
            self._put_unique(self.truths_by_subject, sid, row, "subject")
            self._put_unique(self.truths_by_annotation, annotation, row, "annotation")
            self._put_unique(self.truths_by_ref, row["ground_truth_ref"], row, "ground-truth reference")
            self.truths_by_frame[_frame_key(row)].append(row)
        for rows in self.truths_by_frame.values():
            rows.sort(key=lambda row: row["subject_id"])

    def _load_groups(self):
        physical_groups, hash_groups = {}, {}
        for row in self._rows("shared_source_groups"):
            gid = row.get("shared_source_group_id")
            self._put_unique(self.groups_by_id, gid, row, "shared-source group")
            paths = row.get("source_image_relpaths")
            if not isinstance(paths, list) or not paths or len(set(paths)) != len(paths):
                raise IntegrityError(f"Malformed group membership: {gid}")
            tokens, physical, hashes = set(), set(), set()
            for path in paths:
                source = self.sources_by_path.get(path)
                if source is None or path in self.group_for_path:
                    raise IntegrityError(f"Missing or multiply grouped source: {path}")
                self.group_for_path[path] = gid
                frame, sha = source["physical_frame_id"], source["sha256"]
                if physical_groups.setdefault(frame, gid) != gid or hash_groups.setdefault(sha, gid) != gid:
                    raise IntegrityError("Identical frames/bytes cross shared-source groups")
                physical.add(frame)
                hashes.add(sha)
                tokens.update(("frame:" + frame, "sha256:" + sha))
            if set(row.get("physical_frame_ids") or []) != physical or set(row.get("source_sha256_values") or []) != hashes:
                raise IntegrityError(f"Group declared identities differ from its sources: {gid}")
            if gid != s8_semantic_hash({"physical_or_byte_shared_source_component": sorted(tokens)}):
                raise IntegrityError(f"Shared-source group identity hash mismatch: {gid}")
        if set(self.group_for_path) != set(self.sources_by_path):
            raise IntegrityError("Source inventory is not completely partitioned into shared-source groups")

    def _validate_recipe(self, row):
        self.benchmark._validate_recipe(row)
        sid, eid = row["subject_id"], row["example_id"]
        truth = self.truths_by_subject.get(sid)
        if truth is None or row["ground_truth_ref"] != truth["ground_truth_ref"]:
            raise IntegrityError(f"Recipe subject/evidence mismatch: {eid}")
        if eid != s8_semantic_hash({"subject_id": sid, "variant": row["variant"],
                                    "ordered_artifact_ids": [i["artifact_id"] for i in row["ordered_inputs"]]}):
            raise IntegrityError(f"Recipe identity hash mismatch: {eid}")
        groups = set()
        for order, item in enumerate(row["ordered_inputs"]):
            aid = item["artifact_id"]
            visual = self.visual_by_artifact.get(aid)
            if visual is None:
                raise IntegrityError(f"Recipe references unknown visual: {eid}/{aid}")
            if any(item.get(field) != visual.get(field) for field in ("relative_path", "sha256", "width", "height")):
                raise IntegrityError(f"Recipe visual identity/dimensions mismatch: {eid}/{aid}")
            expected_storage = "original" if visual["storage"] == "source" else "derived"
            if item["storage"] != expected_storage:
                raise IntegrityError(f"Recipe/visual storage mismatch: {eid}/{aid}")
            expected_input_id = s8_semantic_hash({"example_subject": sid, "variant": row["variant"], "order": order, "artifact_id": aid})
            if item.get("input_id") != expected_input_id:
                raise IntegrityError(f"Input identity hash mismatch: {eid}/{order}")
            links = self.links_by_artifact[aid]
            if expected_storage == "original":
                if len(links) != 1 or links[0]["image_relpath"] != item["relative_path"]:
                    raise IntegrityError(f"Original recipe/source provenance mismatch: {eid}/{aid}")
                if "frame_index" in item and item["frame_index"] != links[0]["frame_index"]:
                    raise IntegrityError(f"Temporal recipe/source frame mismatch: {eid}/{aid}")
                if row["variant"] == "temporal_multi":
                    is_target = item["relative_path"] == truth["image_relpath"]
                    if item["is_target"] != is_target:
                        raise IntegrityError(f"Temporal target differs from subject frame: {eid}")
            groups.update(self.group_for_path[link["image_relpath"]] for link in links)
        if groups != {row["shared_source_group_id"]}:
            raise IntegrityError(f"Recipe spans an inconsistent shared-source group: {eid}")
        if row.get("annotation_assisted") is not any(self.visual_by_artifact[i["artifact_id"]].get("annotation_assisted") is True for i in row["ordered_inputs"]):
            raise IntegrityError(f"Recipe annotation-assistance flag mismatch: {eid}")
        if row.get("final_split") is not None or row.get("training_ready") is not False:
            raise IntegrityError("S8 recipe readiness/split contract changed; review before generation")

    def _load_input_index(self):
        fields = ("example_id", "subject_id", "variant", "ground_truth_ref", "shared_source_group_id",
                  "annotation_assisted", "image_count", "pixels_available", "final_split", "training_ready")
        for row in self._rows("benchmark_input_manifest"):
            self._validate_recipe(row)
            eid, sid = row["example_id"], row["subject_id"]
            self._put_unique(self.input_by_id, eid, {key: row.get(key) for key in fields}, "benchmark input")
            if row["variant"] in self.available_variants_by_subject[sid]:
                raise IntegrityError(f"Multiple recipes for the same subject/variant: {sid}/{row['variant']}")
            self.available_variants_by_subject[sid].add(row["variant"])
            self.recipe_artifact_ids.update(item["artifact_id"] for item in row["ordered_inputs"])
            self.ordered_input_slots += len(row["ordered_inputs"])
            self._recipe_hashes[eid] = s8_semantic_hash(row)
        if set(self.available_variants_by_subject) != set(self.truths_by_subject):
            missing = set(self.truths_by_subject) - set(self.available_variants_by_subject)
            raise IntegrityError(f"Subjects unexplained by recipe inventory: {len(missing)}")

    def iter_recipes(self):
        """Every published recipe, in published order. No image decoding or filtering."""
        verify_file(self._recipes_local, self._recipes_descriptor["bytes"], self._recipes_descriptor["sha256"])
        with gzip.open(self._recipes_local, "rt", encoding="utf-8") as stream:
            for line_number, line in enumerate(stream, 1):
                if not line.strip():
                    raise IntegrityError(f"Blank recipe record at line {line_number}")
                row = json.loads(line)
                if not isinstance(row, dict):
                    raise IntegrityError(f"Recipe is not an object at line {line_number}")
                yield row

    def input_records(self):
        return self.iter_recipes()

    def _validate_recipe_stream(self):
        seen = set()
        for row in self.iter_recipes():
            eid = row.get("example_id")
            if eid in seen or eid not in self._recipe_hashes:
                raise IntegrityError(f"Duplicate or unindexed JSONL recipe: {eid}")
            if s8_semantic_hash(row) != self._recipe_hashes[eid]:
                raise IntegrityError(f"Recipe JSONL differs from Parquet manifest: {eid}")
            seen.add(eid)
        if seen != set(self.input_by_id):
            raise IntegrityError(f"Recipes missing from JSONL: {len(set(self.input_by_id)-seen)}")
        return len(seen)


CORPUS = Corpus(BENCHMARK, batch_size=CONFIG["metadata_batch_size"])
write_json(RUN_DIR / "corpus_validation.json", CORPUS.validation_report)
write_json(RUN_DIR / "source_snapshot.json", CORPUS.source_snapshot)
print(json.dumps(CORPUS.validation_report, indent=2))
print("Every annotation-based subject is retained, including subjects without bounding boxes.")
print("These are annotation observations, not a patient count. No QA is generated in this stage.")


Verify metadata: metadata/5d2f2d49883814ea4c8226b409c6139a125b95bd02cf3bbe29bcb5f83f80c865/results/source_evidence_manifest.parquet
GitHub read: /bogdanparvu18/msc-graduate-project/fd5c5c1ab660bc1518d18b1500882ab458e6419f/outputs/phase2/sector8/metadata/5d2f2d49883814ea4c8226b409c6139a125b95bd02cf3bbe29bcb5f83f80c865/results/source_evidence_manifest.parquet
Verify metadata: metadata/5d2f2d49883814ea4c8226b409c6139a125b95bd02cf3bbe29bcb5f83f80c865/results/source_evidence_manifest_schema.json
GitHub read: /bogdanparvu18/msc-graduate-project/fd5c5c1ab660bc1518d18b1500882ab458e6419f/outputs/phase2/sector8/metadata/5d2f2d49883814ea4c8226b409c6139a125b95bd02cf3bbe29bcb5f83f80c865/results/source_evidence_manifest_schema.json
Complete metadata table: source_evidence_manifest: 66,127 rows
Verify metadata: metadata/5d2f2d49883814ea4c8226b409c6139a125b95bd02cf3bbe29bcb5f83f80c865/results/visual_manifest.parquet
GitHub read: /bogdanparvu18/msc-graduate-project/fd5c5c1ab660bc1518d18b1500882ab458e64

## 3. Inventory normalized Phase 2 classes and their annotation provenance

Preserve `source_label` and `clinical_group`. Normalization only uses the existing lowercase/whitespace convention. `dataset_category` remains separate from terminology-derived groupings. Each observation retains its annotation, image, video, frame and source commit; terminology import creates no additional clinical observations.


In [3]:
"""Inventory Phase 2 labels from verified S8 annotation evidence; no QA generation."""


def normalize_dataset_label(value):
    """Match the existing Phase 3 case/whitespace convention; add no synonyms."""
    if value is None:
        return ""
    if not isinstance(value, str):
        raise ValueError("Dataset labels and categories must be text, not coerced values")
    return " ".join(value.strip().lower().split())


def build_phase2_class_inventory(truth_rows, source_snapshot, *, source_rows=None,
                                 expected_class_count=None):
    """Build classes, observation provenance and issues from verified S8 rows.

    Caller must load rows through the existing integrity-checked Corpus reader.
    The snapshot records that reader's immutable GitHub source. This function
    does not reinterpret annotations or claim that a technical integrity check
    clinically validates the underlying image label.
    """
    commit = source_snapshot.get("resolved_commit", "")
    if not re.fullmatch(r"[0-9a-f]{40}", commit):
        raise ValueError("The source snapshot must contain an immutable resolved_commit")
    groups, observations, issues = defaultdict(list), [], []
    seen_subjects, seen_annotations, seen_refs = set(), set(), set()
    for row in truth_rows:
        label = normalize_dataset_label(row.get("source_label"))
        if not label:
            raise ValueError("A ground-truth annotation has an empty source_label")
        required = ("subject_id", "annotation_id", "ground_truth_ref", "image_relpath", "video_key")
        if any(not isinstance(row.get(k), str) or not row[k] for k in required):
            raise ValueError("A ground-truth annotation lacks its source identity")
        for key, seen in (("subject_id", seen_subjects), ("annotation_id", seen_annotations),
                          ("ground_truth_ref", seen_refs)):
            if row[key] in seen:
                raise ValueError(f"Duplicate ground-truth {key}: {row[key]}")
            seen.add(row[key])
        frame = row.get("frame_index")
        if isinstance(frame, bool) or not isinstance(frame, (int, float)) or not math.isfinite(frame) or frame < 0 or frame != int(frame):
            raise ValueError("A ground-truth annotation lacks a valid frame_index")
        if row.get("question") is not None or row.get("answer") is not None:
            raise ValueError("Phase 2 evidence unexpectedly contains authored QA")
        category = normalize_dataset_label(row.get("clinical_group"))
        class_id = "kvasir_capsule:" + hashlib.sha256(label.encode("utf-8")).hexdigest()[:24]
        observation = {
            "dataset_class_id": class_id,
            "normalized_label": label,
            "source_label": row["source_label"],
            "dataset_category": category or None,
            "source_clinical_group": row.get("clinical_group"),
            "subject_id": row["subject_id"],
            "annotation_id": row["annotation_id"],
            "ground_truth_ref": row["ground_truth_ref"],
            "image_relpath": row["image_relpath"],
            "video_key": row["video_key"],
            "frame_index": int(frame),
            "has_bbox": bool(row.get("bbox_xyxy")),
            "source_commit": commit,
        }
        observations.append(observation)
        groups[label].append(observation)
    if not observations:
        raise ValueError("No verified annotation evidence was supplied")
    classes = []
    for label, rows in sorted(groups.items()):
        categories = sorted({r["dataset_category"] for r in rows if r["dataset_category"]})
        missing_category = sum(r["dataset_category"] is None for r in rows)
        category_status = "consistent" if len(categories) == 1 and not missing_category else "review_required"
        classes.append({
            "dataset_class_id": rows[0]["dataset_class_id"],
            "normalized_label": label,
            "source_label_forms": sorted({r["source_label"] for r in rows}),
            "dataset_categories": categories,
            "dataset_category_status": category_status,
            "annotations": len(rows),
            "frames": len({(r["video_key"], r["frame_index"]) for r in rows}),
            "videos": len({r["video_key"] for r in rows}),
            "annotations_with_bbox": sum(r["has_bbox"] for r in rows),
            "annotations_without_category": missing_category,
            "source_commit": commit,
            "source_table": "benchmark_ground_truth_manifest",
            "source_field": "source_label",
        })
        if category_status != "consistent":
            issues.append({"dataset_class_id": rows[0]["dataset_class_id"],
                           "code": "source_category_missing_or_conflicting",
                           "detail": json.dumps({"categories": categories, "missing": missing_category}, sort_keys=True)})
    if expected_class_count is not None and len(classes) != expected_class_count:
        raise ValueError(f"Pinned source expected {expected_class_count} classes; observed {len(classes)}")
    if source_rows is not None:
        for row in source_rows:
            labels = row.get("verified_labels") or []
            if not isinstance(labels, (list, tuple)):
                raise ValueError("Source verified_labels must be a decoded list")
            for raw_label in labels:
                label = normalize_dataset_label(raw_label)
                if label and label not in groups:
                    issues.append({"dataset_class_id": None,
                                   "code": "source_label_without_ground_truth_class",
                                   "detail": json.dumps({"source_label": raw_label,
                                                         "image_relpath": row.get("image_relpath")}, sort_keys=True)})
    observations.sort(key=lambda r: (r["normalized_label"], r["subject_id"]))
    return {
        "classes": classes,
        "observations": observations,
        "issues": issues,
        "summary": {
            "class_count": len(classes),
            "annotation_count": len(observations),
            "frame_count": len({(r["video_key"], r["frame_index"]) for r in observations}),
            "source_commit": commit,
            "source_processing_signature": source_snapshot.get("processing_signature"),
            "source_catalog_sha256": source_snapshot.get("source_catalog_sha256"),
            "category_conflict_count": sum(r["dataset_category_status"] != "consistent" for r in classes),
            "issue_count": len(issues),
            "normalization": "lowercase_and_collapsed_whitespace_only",
            "clinical_reannotation_performed": False,
            "qa_generated": 0,
        },
    }

PHASE2_INVENTORY = build_phase2_class_inventory(
    CORPUS.truths_by_subject.values(), CORPUS.source_snapshot,
    source_rows=CORPUS.sources_by_path.values(),
    expected_class_count=CONFIG["expected_class_count"],
)
CLASSES = pd.DataFrame(PHASE2_INVENTORY["classes"])
OBSERVATIONS = pd.DataFrame(PHASE2_INVENTORY["observations"])
display(CLASSES[["normalized_label", "dataset_categories", "annotations", "frames", "annotations_with_bbox"]])
print(json.dumps(PHASE2_INVENTORY["summary"], indent=2))


,normalized_label,dataset_categories,annotations,frames,annotations_with_bbox
0,ampulla of vater,[anatomical_landmark],10,10,10
1,angiectasia,[vascular_lesion],866,866,864
2,blood fresh,[bleeding],446,446,395
3,blood hematin,[bleeding],12,12,12
4,erosion,[mucosal_lesion],507,506,504
5,erythema,[mucosal_lesion],159,159,117
6,foreign body,[foreign_body],776,776,735
7,ileocecal valve,[anatomical_landmark],4189,4189,0
8,lymphangiectasia,[lymphatic_lesion],592,592,592
9,normal clean mucosa,[normal_mucosa],34338,34338,0


{
  "class_count": 14,
  "annotation_count": 47239,
  "frame_count": 47229,
  "source_commit": "fd5c5c1ab660bc1518d18b1500882ab458e6419f",
  "source_processing_signature": null,
  "source_catalog_sha256": "2ccf17c63d568c2d4ba36e390fe90c9418fea5ca1a95b818715ec1920134ea4d",
  "category_conflict_count": 0,
  "issue_count": 0,
  "normalization": "lowercase_and_collapsed_whitespace_only",
  "clinical_reannotation_performed": false,
  "qa_generated": 0
}


## 4. Preserve the task and evidence coverage contract

The original task definitions and A/N/R policies remain authoritative. `applicable` means permitted **only when the stated guards hold**; `review_required` is not an accepted example. The 9 tasks × 11 variant specifications × 9 evidence probes produce 891 expected combinations. Evidence probes may overlap in real inputs.

The historical `implementation` fields are retained for traceability; they do not mean that generators run in this notebook. `observed_status` remains `not_evaluated`. Section 9 adds terminology bindings without changing these expected policy outcomes. `finding_category` continues to require the original source-backed category; ontology categories are not silently substituted.


In [4]:
COVERAGE_SPEC_VERSION = "mmvqa-s1-coverage-v3"

TASK_SPECIFICATION = [
    {"task": "finding_identification", "rule_id": "finding_identification_v1", "requirement": "REQ_FINDING", "reference": "Verified labels attached to the displayed source or indicated region; allow a grounded label set.", "required": True, "implementation": "generator"},
    {"task": "finding_category", "rule_id": "finding_category_v1", "requirement": "REQ_CATEGORY", "reference": "Unambiguous source-backed clinical_group for each applicable finding; no inferred taxonomy.", "required": True, "implementation": "generator"},
    {"task": "evidence_frame_localization", "rule_id": "evidence_frame_v1", "requirement": "REQ_EVIDENCE", "reference": "An explicit positive annotation linked to a displayed frame/panel; geometry is optional.", "required": True, "implementation": "generator"},
    {"task": "optional_spatial_localization", "rule_id": "spatial_localization_v1", "requirement": "REQ_SPATIAL", "reference": "Valid same-class geometric reference in the coordinates of the evaluated image.", "required": True, "implementation": "generator"},
    {"task": "verified_reference_comparison", "rule_id": "comparison_reference_v1", "requirement": "REQ_COMPARISON", "reference": "Explicit verified normal reference, distinct source and visible role-to-panel mapping.", "required": True, "implementation": "generator"},
    {"task": "observed_anchor_order", "rule_id": "temporal_observed_order_v1", "requirement": "REQ_TEMPORAL", "reference": "At least two displayed positive anchors of the queried finding in one video; observed order only.", "required": True, "implementation": "generator"},
    {"task": "temporal_relative_evidence_localization", "rule_id": "temporal_relative_evidence_v1", "requirement": "REQ_TEMPORAL", "reference": "Every supported finding × distinct displayed reference frame × before/after; accept a nonempty set of positively annotated same-video frames in the requested direction. Reference frames may be unlabelled; unknown target frames are not negative.", "required": True, "implementation": "generator", "cardinality": "all_finding_reference_direction_units"},
    {"task": "clinical_visual_quality", "rule_id": "clinical_quality_v1", "requirement": "REQ_QUALITY", "reference": "Independently reviewed clinical visibility target; S8 technical QC is insufficient.", "required": False, "implementation": "review_placeholder"},
    {"task": "guarded_clinical_sufficiency", "rule_id": "guarded_sufficiency_v1", "requirement": "REQ_GUARDED", "reference": "Independently reviewed sufficiency, ambiguity or abstention target.", "required": False, "implementation": "review_placeholder"},
]

EVIDENCE_STATES = [
    "label_without_box", "label_with_box", "multiple_labels", "missing_category",
    "unlabelled_neighbor", "explicit_normal_reference", "multiple_verified_anchors",
    "annotation_assisted_input", "unmapped_or_conflicting_reference",
]

EVIDENCE_STATE_MEANINGS = {
    "label_without_box": "A visible annotated source has a verified finding and category but no box. Other views remain eligible.",
    "label_with_box": "A visible annotated source has a verified finding, category and valid source-coordinate box.",
    "multiple_labels": "More than one verified finding is attached to the visible source; use supported sets or unambiguous referents.",
    "missing_category": "Finding evidence exists but its clinical_group is absent or cannot be mapped unambiguously.",
    "unlabelled_neighbor": "The focused context frame has no positive annotation; absence of annotation is not a negative target.",
    "explicit_normal_reference": "The recipe has an explicitly annotated normal comparison reference distinct from the target.",
    "multiple_verified_anchors": "At least two displayed same-finding positive anchors have verified frame identities in one video.",
    "annotation_assisted_input": "The displayed input contains an annotation-derived crop or overlay; assistance must be declared.",
    "unmapped_or_conflicting_reference": "A required label, geometry, role or provenance reference is unresolved or contradictory.",
}

VARIANT_SPECIFICATION = {
    "full": {"family": "single_source", "assisted_possible": False, "requires_box": False},
    "crop": {"family": "single_region", "assisted_possible": True, "requires_box": True},
    "fullcrop": {"family": "same_source_views", "assisted_possible": True, "requires_box": True},
    "fullcrop_panel": {"family": "same_source_views", "assisted_possible": True, "requires_box": True},
    "overlay": {"family": "single_source", "assisted_possible": True, "requires_box": True},
    "temporal_multi": {"family": "temporal", "assisted_possible": False, "requires_box": False},
    "temporal_collage": {"family": "temporal", "assisted_possible": False, "requires_box": False},
    "progression": {"family": "temporal", "assisted_possible": False, "requires_box": False},
    "qc": {"family": "quality", "assisted_possible": False, "requires_box": False},
    "comparison": {"family": "comparison", "assisted_possible": False, "requires_box": False},
    "combined": {"family": "combined", "assisted_possible": True, "requires_box": False},
}

TASK_EVIDENCE_POLICY = {
    "finding_identification": ("A", "A", "A", "A", "N", "A", "A", "A", "R"),
    "finding_category": ("A", "A", "A", "R", "N", "A", "A", "A", "R"),
    "evidence_frame_localization": ("A", "A", "A", "A", "N", "A", "A", "A", "R"),
    "optional_spatial_localization": ("N", "A", "A", "A", "N", "A", "A", "A", "R"),
    "verified_reference_comparison": ("N", "N", "N", "N", "N", "A", "N", "N", "R"),
    "observed_anchor_order": ("N", "N", "N", "N", "N", "N", "A", "N", "R"),
    "temporal_relative_evidence_localization": ("A", "A", "A", "A", "A", "A", "A", "A", "R"),
    "clinical_visual_quality": ("R", "R", "R", "R", "R", "R", "R", "R", "R"),
    "guarded_clinical_sufficiency": ("R", "R", "R", "R", "R", "R", "R", "R", "R"),
}

S8_HINT_SPECIFICATION = {
    "finding_identification": (["REQ_FINDING", "REQ_CATEGORY"], "source_supported_identification"),
    "visual_quality_assessment": (["REQ_QUALITY"], "clinical_reference_missing"),
    "indicated_region_characterization": (["REQ_FINDING", "REQ_CATEGORY", "REQ_SPATIAL"], "region_scope_checks_required"),
    "global_local_comparison": (["REQ_FINDING", "REQ_EVIDENCE"], "finding_views_supported; diagnostic_comparison_requires_review"),
    "indicated_region_interpretation": (["REQ_FINDING", "REQ_CATEGORY"], "annotation_assistance_retained"),
    "neighbor_frame_comparison": (["REQ_TEMPORAL", "REQ_EVIDENCE"], "observed_positive_anchors_and_exhaustive_relative_evidence_units"),
    "observed_verified_anchor_comparison": (["REQ_TEMPORAL", "REQ_EVIDENCE"], "observed_order_and_relative_positive_evidence_only; disease_progression_not_supported"),
    "verified_reference_image_comparison": (["REQ_COMPARISON"], "explicit_reference_selection_only"),
    "multi_view_evidence_interpretation": (["REQ_FINDING", "REQ_CATEGORY", "REQ_EVIDENCE"], "source_supported_tasks_only; integrative_clinical_reasoning_requires_review"),
}

def _coverage_expectation(task, variant, evidence):
    """Read the independent policies; never run generation code here."""
    if variant not in VARIANT_SPECIFICATION:
        return "review_required", "New S8 variant has no independently reviewed capability policy.", "unknown_variant"
    capability = VARIANT_SPECIFICATION[variant]
    if evidence == "unmapped_or_conflicting_reference":
        return "review_required", "Resolve the conflicting or unmapped evidence before any target can be accepted.", "reference_integrity"
    if evidence == "annotation_assisted_input" and not capability["assisted_possible"]:
        return "not_applicable", "This S8 variant is unassisted by construction; an assisted instance would contradict its contract.", "impossible_state"
    if evidence == "label_without_box" and capability["requires_box"]:
        return "not_applicable", "S8 cannot construct this optional ROI/overlay variant without geometry; retain the subject and its other views.", "impossible_state"
    if evidence == "explicit_normal_reference" and variant != "comparison":
        return "not_applicable", "The paired explicit-normal-reference probe belongs to the comparison recipe; ordinary normal-labelled frames remain finding evidence.", "impossible_state"
    if evidence == "multiple_verified_anchors" and capability["family"] in {"single_source", "single_region", "same_source_views"}:
        return "not_applicable", "Several renderings of one physical frame are not multiple observed anchors.", "impossible_state"
    name = task["task"]
    if name == "evidence_frame_localization" and capability["family"] in {"single_source", "single_region", "same_source_views"}:
        return "not_applicable", "There is only one physical source frame; selecting it supplies no frame-evidence discrimination.", "variant_scope"
    if name == "verified_reference_comparison" and variant != "comparison":
        return "not_applicable", "This recipe does not provide the dedicated target/reference pairing required by this task.", "variant_scope"
    if name in {"observed_anchor_order", "temporal_relative_evidence_localization"} and capability["family"] not in {"temporal", "combined"}:
        return "not_applicable", "This task is restricted to explicit temporal/anchor recipes, not arbitrary ordering of quality or comparison inputs.", "variant_scope"
    policy = TASK_EVIDENCE_POLICY[name]
    code = policy[evidence]["expected_policy_code"] if isinstance(policy, dict) else policy[EVIDENCE_STATES.index(evidence)]
    if code == "R":
        reasons = {
            "clinical_visual_quality": "S8 has technical proxies, not independently validated clinical visibility or diagnostic-quality answers.",
            "guarded_clinical_sufficiency": "S8 has no independently validated sufficiency/abstention targets; do not convert missing data into an answer.",
            "finding_category": "The category cannot be inferred or invented when its source mapping is missing or ambiguous.",
        }
        return "review_required", reasons.get(name, "The available evidence does not justify an unambiguous reference answer."), "missing_validated_reference"
    if code == "N":
        if evidence == "unlabelled_neighbor":
            reason = "An unlabelled context frame supplies no positive finding target and must not be labelled normal or negative."
        elif name == "optional_spatial_localization":
            reason = "No geometric reference is available; this does not block finding, category or frame-evidence questions."
        elif name == "verified_reference_comparison":
            reason = "This probe supplies no explicit normal comparison reference."
        else:
            reason = "This probe supplies fewer than two verified displayed positive anchors; do not infer temporal continuity."
        return "not_applicable", reason, "missing_task_evidence"
    if name == "optional_spatial_localization" and variant != "full":
        return "review_required", "Spatial localization outside the single unassisted full-frame recipe is explicitly deferred: multi-input coordinate selection, transformations and answer leakage need a reviewed policy.", "transformed_spatial_policy"
    if name == "temporal_relative_evidence_localization" and evidence == "unlabelled_neighbor":
        return "applicable", "An unlabelled frame may provide the temporal reference position. A different displayed frame must positively support the queried finding in the requested direction; no unknown target is scored as absent.", "reference_anchor_need_not_have_finding"
    return "applicable", task["reference"], "reference_guards_required"

TEMPORAL_RELATIVE_COMPLETENESS_SPECIFICATION = {
    "rule_id": "temporal_relative_evidence_v1",
    "task_id": "temporal_relative_evidence_localization",
    "variants": ["temporal_multi", "temporal_collage", "progression", "combined"],
    "unit_axes": ["each_supported_finding", "each_distinct_displayed_physical_reference_frame", "before_or_after"],
    "expected_units_formula": "2 * distinct_supported_findings * distinct_displayed_reference_frames",
    "reference_policy": "One reference position per (video_key, frame_index), regardless of repeated crops/overlays; it need not be positively annotated.",
    "target_policy": "Only explicitly positive same-video frames strictly before/after the reference; no bbox or category requirement.",
    "unknown_policy": "Unlabelled frames may be reference positions, never automatically positive or negative targets. Missing annotation support is not clinical absence.",
    "empty_target_policy": "Record not_applicable with a reason; do not author a negative answer or silently drop the unit.",
    "conflict_policy": "Record review_required for unresolved frame order, provenance or source contradictions.",
    "deduplication_policy": "Repeated views of one physical frame do not multiply reference positions. Distinct reference/direction predicates remain separate units; semantic families identify reused evidence across presentation variants.",
    "candidate_limit": None,
    "audit_obligation": "Recompute expected unit keys and known-positive frame sets independently of emitted decisions, then check missing, extra, duplicate and wrongly classified units.",
}

# Keep original contract values; implementation fields describe the historical
# task specification, not execution in this terminology-only notebook.
CONTRACT_EXECUTION_STATUS = "not_executed_terminology_stage"
COVERAGE_CONTRACT = pd.DataFrame([
    {"task": task["task"], "rule_id": task["rule_id"], "variant": variant,
     "evidence_state": state, "expected_status": expectation[0],
     "reason": expectation[1], "reason_code": expectation[2],
     "observed_status": "not_evaluated", "qa_generation_executed": False}
    for task in TASK_SPECIFICATION
    for variant in sorted(VARIANT_SPECIFICATION)
    for state in EVIDENCE_STATES
    for expectation in [_coverage_expectation(task, variant, state)]
])
write_json(RUN_DIR / "task_contract.json", {
    "coverage_spec_version": COVERAGE_SPEC_VERSION,
    "task_specification": TASK_SPECIFICATION,
    "evidence_states": EVIDENCE_STATES,
    "evidence_state_meanings": EVIDENCE_STATE_MEANINGS,
    "variant_specification": VARIANT_SPECIFICATION,
    "task_evidence_policy": TASK_EVIDENCE_POLICY,
    "s8_hint_specification": S8_HINT_SPECIFICATION,
    "temporal_relative_completeness_specification": TEMPORAL_RELATIVE_COMPLETENESS_SPECIFICATION,
    "execution_status": CONTRACT_EXECUTION_STATUS,
})
COVERAGE_CONTRACT.to_parquet(RUN_DIR / "coverage_contract.parquet", index=False)
display(pd.DataFrame(TASK_SPECIFICATION))
display(COVERAGE_CONTRACT.groupby(["task", "expected_status"]).size().unstack(fill_value=0))
print("Coverage contract:", len(COVERAGE_CONTRACT), "expected combinations; observed QA coverage has not been evaluated.")
BASE_CONTRACT = copy.deepcopy({name: globals()[name] for name in (
    "TASK_SPECIFICATION", "EVIDENCE_STATES", "EVIDENCE_STATE_MEANINGS",
    "VARIANT_SPECIFICATION", "TASK_EVIDENCE_POLICY", "S8_HINT_SPECIFICATION",
)})
BASE_COVERAGE_CONTRACT = COVERAGE_CONTRACT.copy(deep=True)


,task,rule_id,requirement,reference,required,implementation,cardinality
0,finding_identification,finding_identification_v1,REQ_FINDING,Verified labels attached to the displayed sour...,True,generator,NaN
1,finding_category,finding_category_v1,REQ_CATEGORY,Unambiguous source-backed clinical_group for e...,True,generator,NaN
2,evidence_frame_localization,evidence_frame_v1,REQ_EVIDENCE,An explicit positive annotation linked to a di...,True,generator,NaN
3,optional_spatial_localization,spatial_localization_v1,REQ_SPATIAL,Valid same-class geometric reference in the co...,True,generator,NaN
4,verified_reference_comparison,comparison_reference_v1,REQ_COMPARISON,"Explicit verified normal reference, distinct s...",True,generator,NaN
5,observed_anchor_order,temporal_observed_order_v1,REQ_TEMPORAL,At least two displayed positive anchors of the...,True,generator,NaN
6,temporal_relative_evidence_localization,temporal_relative_evidence_v1,REQ_TEMPORAL,Every supported finding × distinct displayed r...,True,generator,all_finding_reference_direction_units
7,clinical_visual_quality,clinical_quality_v1,REQ_QUALITY,Independently reviewed clinical visibility tar...,False,review_placeholder,NaN
8,guarded_clinical_sufficiency,guarded_sufficiency_v1,REQ_GUARDED,"Independently reviewed sufficiency, ambiguity ...",False,review_placeholder,NaN


expected_status,applicable,not_applicable,review_required
task,,,
clinical_visual_quality,0,25,74
evidence_frame_localization,32,56,11
finding_category,41,36,22
finding_identification,52,36,11
guarded_clinical_sufficiency,0,25,74
observed_anchor_order,4,84,11
optional_spatial_localization,3,43,53
temporal_relative_evidence_localization,25,63,11
verified_reference_comparison,1,87,11


Coverage contract: 891 expected combinations; observed QA coverage has not been evaluated.


## 5. MST, CEST and capsule nomenclature import functions

MST terms, attributes, values and procedure/organ applicability remain distinct tables. Workbook row identifiers are local source locators, not official ontology codes. Source labels can link to one finding in the same source section by exact text, punctuation-normalized text, a complete slash-separated alternative, or an explicit dotted abbreviation. All words and qualifiers must match that single candidate; multiple candidates remain unresolved. These lexical links connect workbook rows and do not approve new clinical synonyms or Phase 2 class mappings. Accepted links, absent vocabulary, other unresolved labels, and attribute-layout issues are reported separately with source locations and hashes. Original labels and cells remain preserved.

CEST input schema: `mmvqa-cest-transcription-v1`, containing source metadata, a source document path, reviewed transcription identity, `terms`, explicit `synonyms`, and `attributes`. The document hash is computed automatically; an optional supplied `source_sha256` is also checked. The notebook exports an empty schema template; filling it requires the real source. Both original document bytes and transcription bytes are bound to the import identity. Staging preserves the JSON bytes unchanged, including relative or absolute document references.

Sources: [WEO MST 3.0](https://www.worldendo.org/resources/minimal-standard-terminology), [CEST 2005](https://doi.org/10.1055/s-2005-870329), [capsule vascular nomenclature consensus 2019](https://doi.org/10.1055/a-0761-9742). The supplement preserves the distinction between angiectasia, erythematous patch and red spot. It records only explicitly attested alternative names.

The GI supplement is loaded from GitHub `main`, using `outputs/phase3/sector1/gi_supplement_manifest.json` and its referenced JSON artifact. No GI package is embedded in this cell. Exact VM bytes must match the freshly fetched manifest SHA-256; a stable-manifest recheck prevents accepting mismatched files during branch updates. The original source JSON, manifest and retrieval provenance are exported unchanged.


In [5]:
"""Source-faithful MST 3.0 import and reviewed CEST transcription import.

MST row IDs are local identifiers, never official ontology identifiers. No
lexical correspondence is promoted to a clinically approved crosswalk here.
"""


MST_XLS_URL = "https://www.worldendo.org/assets-craft/pdf/resources/mst30_tables.xls"
MST_PDF_URL = "https://www.worldendo.org/assets-craft/pdf/resources/mst-3-0.pdf"
MST_COPYRIGHT_NOTICE = """Copyright 2008 World Endoscopy Organization (formerly known as OMED). All rights reserved.
Permission is hereby granted, without written agreement and without license or royalty fees, to use, copy, or distribute the Minimal Standard Terminology (MST) for any purpose, so long as this copyright notice appears on any copies of the MST and that the following conditions are met.
The notice of WEO (formerly known as OMED) copyright (above) should be displayed on every copy of the MST, on all manuals and other materials used in connection with the MST, including electronic media (disks, CD ROMs, etc.) and should be apparent in text files loaded on these disks or onto the Internet.
The content of the defined core MST fields must not be changed. Users may add list items and sub-classification of items as needed, as long as the diversion from the core MST structure is documented and traceable.
WEO (formerly known as OMED) and the members of the Committee for Standardization and Terminology do not accept liability for any omissions or errors in the MST and all EXPRESS AND IMPLIED WARRANTIES, INCLUDING THOSE RELATING TO MERCHANTABILITY OR FITNESS FOR A PARTICULAR PURPOSE, ARE DISCLAIMED."""


def mst_lookup_key(value):
    """Conservative lookup key; punctuation and qualifiers are not discarded."""
    return " ".join(unicodedata.normalize("NFKC", str(value)).casefold().split())


def _mst_cell(value):
    if value is None:
        return ""
    return str(value).strip()


def _mst_id(sheet, row, column=2):
    return f"MST3:{sheet}:R{row}:C{column}"


def _mst_table(records, columns):
    return pd.DataFrame(records, columns=columns)


MST_TERM_COLUMNS = ["terminology", "version", "term_id", "preferred_term", "term_key",
                "term_type", "section", "organ", "source_url", "source_locator", "source_sha256"]
MST_ATTRIBUTE_COLUMNS = ["term_id", "attribute", "value", "value_kind", "source_locator",
                     "source_url", "source_sha256", "parse_status"]
MST_SYNONYM_COLUMNS = ["term_id", "synonym", "synonym_key", "language", "status",
                   "source_locator", "source_url", "source_sha256"]
MST_APPLICABILITY_COLUMNS = ["term_id", "term_label_raw", "section", "organ", "procedure",
                        "applicability", "source_locator", "source_sha256", "link_status"]
MST_LABEL_ALIGNMENT_COLUMNS = ["section", "source_label", "target_term", "term_id", "match_method",
    "link_status", "reason", "candidate_ids_json", "missing_tokens_json", "source_locator",
    "target_source_locator", "source_sha256"]
MST_LABEL_ALIGNMENT_POLICY = "mst-source-label-alignment-v2"


def _mst_label_tokens(value, retain_dots=False):
    """Whole words only; optional trailing dots identify explicit abbreviations."""
    text = unicodedata.normalize("NFKC", str(value)).casefold()
    tokens = re.findall(r"[^\W_]+\.?", text, flags=re.UNICODE)
    return tuple(tokens if retain_dots else [token.rstrip(".") for token in tokens])


def _mst_source_label_variants(label):
    """Source-spelled full label and slash branches, not asserted synonyms."""
    variants = [str(label)]
    if "/" in str(label):
        branches = [part.strip() for part in str(label).split("/")]
        # Do not split shared qualifiers from constructions such as
        # "Flat/elevated superficial lesion" into the incomplete label "Flat".
        if all(len(_mst_label_tokens(part)) == 1 for part in branches):
            variants.extend(branches)
    return variants


def mst_resolve_source_label(label, section, terms, *, reference_vocabulary=None):
    """Resolve a label between MST sheets without changing clinical mappings.

    A candidate must be a finding in the same section. Earlier tiers win only
    when they identify one distinct source term. Whole slash alternatives and
    explicit dotted prefixes are allowed; unmatched qualifiers are never
    discarded. Words found in unrelated terms do not establish a match.
    """
    records = terms.to_dict("records") if isinstance(terms, pd.DataFrame) else list(terms)
    candidates = [item for item in records if item.get("term_type") == "finding"
                  and mst_lookup_key(item.get("section", "")) == mst_lookup_key(section)]
    query_key, query_tokens = mst_lookup_key(label), _mst_label_tokens(label)
    query_dotted = _mst_label_tokens(label, retain_dots=True)
    vocabulary = ({token for item in records for token in _mst_label_tokens(item.get("preferred_term", ""))}
                  if reference_vocabulary is None else set(reference_vocabulary))

    def token_known(token):
        plain = token.rstrip(".")
        return plain in vocabulary or (token.endswith(".") and len(plain) >= 3
                                      and any(word.startswith(plain) for word in vocabulary))

    missing = sorted({token.rstrip(".") for token in query_dotted if not token_known(token)})
    base = {"section": str(section), "source_label": str(label), "target_term": "", "term_id": "",
            "match_method": "", "link_status": "unresolved_source_label", "reason": "",
            "candidate_ids_json": "[]", "missing_tokens_json": json.dumps(missing, ensure_ascii=False),
            "source_locator": "", "target_source_locator": "", "source_sha256": ""}

    def dotted_match(target):
        target_tokens = _mst_label_tokens(target)
        return bool(query_dotted) and len(query_dotted) == len(target_tokens) and all(
            query.rstrip(".") == target_word or
            (query.endswith(".") and len(query.rstrip(".")) >= 3 and target_word.startswith(query.rstrip(".")))
            for query, target_word in zip(query_dotted, target_tokens)
        )

    tiers = [
        ("exact_source_label", lambda item: query_key == mst_lookup_key(item["preferred_term"])),
        ("normalized_source_label", lambda item: bool(query_tokens) and query_tokens == _mst_label_tokens(item["preferred_term"])),
        ("slash_alternative_source_label", lambda item: bool(query_tokens) and "/" in item["preferred_term"]
         and any(query_tokens == _mst_label_tokens(part) for part in _mst_source_label_variants(item["preferred_term"])[1:])),
        ("abbreviated_source_label", lambda item: any(token.endswith(".") and len(token.rstrip(".")) >= 3 for token in query_dotted)
         and any(dotted_match(part) for part in _mst_source_label_variants(item["preferred_term"]))),
    ]
    for method, predicate in tiers:
        matches = [item for item in candidates if predicate(item)]
        ids = sorted({str(item["term_id"]) for item in matches})
        if not ids:
            continue
        result = {**base, "match_method": method, "candidate_ids_json": json.dumps(ids)}
        if len(ids) != 1:
            return {**result, "reason": "ambiguous_source_label"}
        # A term ID identifies one source row. Sorting makes repeated identical
        # input records irrelevant to the deterministic source selection.
        target = sorted(matches, key=lambda item: (str(item.get("source_locator", "")), str(item["preferred_term"])))[0]
        return {**result, "target_term": target["preferred_term"], "term_id": ids[0], "link_status": method,
                "reason": "unique_source_label_match", "missing_tokens_json": "[]",
                "target_source_locator": target.get("source_locator", ""), "source_sha256": target.get("source_sha256", "")}
    return {**base, "reason": "missing_source_words" if missing else "known_words_without_compatible_term"}


def read_mst_workbook(path, expected_sha256=None):
    """Import luminal findings, anatomy and diagnoses from the official workbook.

    Every nonempty cell from all sheets is retained in raw_cells. The structured
    capsule-oriented view does not import ERCP/EUS attributes as capsule attributes.
    Blank cells never establish synonymy. Ambiguous layout continuations are
    retained in issues instead of guessed. Missing applicability is unknown.
    """
    path = Path(path)
    digest = hashlib.sha256(path.read_bytes()).hexdigest()
    if expected_sha256 and digest != expected_sha256:
        raise ValueError("MST workbook SHA-256 does not match the recorded source snapshot")
    book = xlrd.open_workbook(path)
    required = {"Luminal_findings", "Luminal_findings_organ", "proc_organ", "Anatomy", "Diagnoses"}
    if not required.issubset(set(book.sheet_names())):
        raise ValueError("Unexpected MST workbook sheets")
    terms, attributes, applicability, issues, raw_cells, label_alignment = [], [], [], [], [], []
    for sheet in book.sheets():
        for row in range(sheet.nrows):
            for column in range(sheet.ncols):
                value = _mst_cell(sheet.cell_value(row, column))
                if value:
                    raw_cells.append({"sheet": sheet.name, "row": row + 1, "column": column + 1,
                                      "value": value, "source_sha256": digest})

    def add_term(sheet, row, label, term_type, section="", organ="", column=2):
        item = dict(zip(MST_TERM_COLUMNS, ["MST", "3.0", _mst_id(sheet, row, column), label,
            mst_lookup_key(label), term_type, section, organ, MST_XLS_URL,
            f"{sheet}!R{row}C{column}", digest]))
        terms.append(item)
        return item["term_id"]

    sheet = book.sheet_by_name("Luminal_findings")
    if sheet.row_values(2) != ["Heading", "Term", "Attribute", "Attr values"]:
        raise ValueError("Unexpected Luminal_findings header")
    section, term_id, current_attribute = "", "", ""
    for zero_row in range(4, sheet.nrows):
        row = zero_row + 1
        heading, term, attribute, value = map(_mst_cell, sheet.row_values(zero_row))
        if heading:
            section, term_id, current_attribute = heading, "", ""
        if term:
            term_id = add_term(sheet.name, row, term, "finding", section)
            current_attribute = ""  # never carry an attribute across a new finding
        if attribute:
            current_attribute = attribute
        if not any((heading, term, attribute, value)):
            current_attribute = ""
            continue
        if attribute or value:
            if not term_id or not current_attribute or value.casefold() == "v":
                issues.append({"source_locator": f"{sheet.name}!R{row}", "reason": "ambiguous_attribute_layout",
                               "term_id": term_id, "attribute_raw": attribute, "value_raw": value})
                continue
            value_kind = "unspecified" if not value else "enumerated"
            if "specify" in value.casefold(): value_kind = "free_text_placeholder"
            elif "classification" in value.casefold() or "class for" in value.casefold(): value_kind = "classification_reference"
            elif re.search(r"\b(mm|cm)\b", value.casefold()) or value == "#": value_kind = "measurement_placeholder"
            attributes.append(dict(zip(MST_ATTRIBUTE_COLUMNS, [term_id, current_attribute, value, value_kind,
                f"{sheet.name}!R{row}C3:C4", MST_XLS_URL, digest, "explicit_layout"])))

    # Anatomy uses separate organ/site/modifier columns; a modifier is not a synonym.
    sheet = book.sheet_by_name("Anatomy")
    organ = ""
    for zero_row in range(4, sheet.nrows):
        row = zero_row + 1
        organ_raw, site, modifier = map(_mst_cell, sheet.row_values(zero_row))
        if organ_raw == "ORGAN": organ = ""; continue
        if organ_raw: organ = organ_raw
        if site and site != "SITE": add_term(sheet.name, row, site, "anatomical_site", "Anatomy", organ)
        # The modifier column is a parallel list, not necessarily scoped to the site
        # on the same spreadsheet row. It therefore remains raw only.

    sheet = book.sheet_by_name("Diagnoses")
    section = ""
    for zero_row in range(3, sheet.nrows):
        row = zero_row + 1
        group, label = map(_mst_cell, sheet.row_values(zero_row))
        if group: section = group
        if label and label != "Term": add_term(sheet.name, row, label, "diagnosis", section)

    # Procedure/organ applicability is imported verbatim. Absence of x is recorded
    # as not_listed, never used as a clinical contradiction or patient fact.
    procedures = book.sheet_by_name("proc_organ")
    procedure_headers = [_mst_cell(v) for v in procedures.row_values(2)]
    organ_procedures = {}
    for zero_row in range(3, procedures.nrows):
        values = [_mst_cell(v) for v in procedures.row_values(zero_row)]
        organ_procedures[values[0]] = {procedure_headers[col]: values[col].casefold() == "x"
                                      for col in range(1, len(values))}
    organ_names = {"Esoph": "Esophagus", "Stomach": "Stomach", "Duod": "Duodenum",
                   "Jej": "Jejunum", "Ileum": "Ileum", "Colon": "Colon"}
    # Vocabulary affects diagnostics only. Excluding the query sheet prevents
    # a label from proving its own vocabulary exists elsewhere in the source.
    reference_vocabulary = {token for cell in raw_cells if cell["sheet"] != "Luminal_findings_organ"
                            for token in _mst_label_tokens(cell["value"])}
    sheet = book.sheet_by_name("Luminal_findings_organ")
    headers = [_mst_cell(v) for v in sheet.row_values(3)]
    section = ""
    for zero_row in range(5, sheet.nrows):
        values = [_mst_cell(v) for v in sheet.row_values(zero_row)]
        if values[0]: section = values[0]
        if not values[1]: continue
        alignment = mst_resolve_source_label(values[1], section, terms, reference_vocabulary=reference_vocabulary)
        alignment.update(source_locator=f"{sheet.name}!R{zero_row+1}C2", source_sha256=digest)
        label_alignment.append(alignment)
        term_id, link_status = alignment["term_id"], alignment["link_status"]
        if not term_id:
            issues.append({"source_locator": f"{sheet.name}!R{zero_row+1}", "reason": alignment["reason"],
                           "term_id": "", "attribute_raw": "", "value_raw": values[1]})
        for column in range(2, len(values)):
            organ = organ_names.get(headers[column], headers[column])
            allowed = values[column].casefold() == "x"
            vce = organ_procedures.get(organ, {}).get("VCE", False)
            applicability.append(dict(zip(MST_APPLICABILITY_COLUMNS, [term_id, values[1], section, organ,
                "VCE", "listed" if allowed and vce else "not_listed", f"{sheet.name}!R{zero_row+1}C{column+1};proc_organ",
                digest, link_status])))
    alignment_table = _mst_table(label_alignment, MST_LABEL_ALIGNMENT_COLUMNS)
    issue_table = pd.DataFrame(issues, columns=["source_locator", "reason", "term_id", "attribute_raw", "value_raw"])
    missing_word_rows = alignment_table.loc[alignment_table["reason"].eq("missing_source_words")].copy()
    unresolved_rows = alignment_table.loc[alignment_table["term_id"].eq("") & ~alignment_table["reason"].eq("missing_source_words")].copy()
    return {
        "terms": _mst_table(terms, MST_TERM_COLUMNS), "attributes": _mst_table(attributes, MST_ATTRIBUTE_COLUMNS),
        "synonyms": _mst_table([], MST_SYNONYM_COLUMNS),
        "applicability": _mst_table(applicability, MST_APPLICABILITY_COLUMNS),
        "raw_cells": pd.DataFrame(raw_cells),
        "issues": issue_table,
        "label_alignment": alignment_table,
        "labels_missing_words": missing_word_rows,
        "labels_unresolved": unresolved_rows,
        "attribute_issues": issue_table.loc[issue_table["reason"].eq("ambiguous_attribute_layout")].copy(),
        "manifest": {"terminology": "MST", "version": "3.0", "source_url": MST_XLS_URL,
            "reference_document_url": MST_PDF_URL, "sha256": digest, "byte_size": path.stat().st_size,
            "status": "imported", "scope": "luminal_findings_anatomy_diagnoses",
            "all_sheets_preserved_as_raw_cells": book.sheet_names(), "synonym_policy": "no_synonym_relations_asserted_by_workbook_parser",
            "parse_issue_count": len(issues), "copyright_notice": MST_COPYRIGHT_NOTICE,
            "parser_version": "mst-source-tables-v2",
            "label_alignment_policy_version": MST_LABEL_ALIGNMENT_POLICY,
            "label_alignment_scope": "same_section_findings_only; lexical_source_link_not_clinical_synonym_approval",
            "missing_word_vocabulary_scope": "all_raw_workbook_cells_excluding_query_applicability_sheet; whole_words_and_explicit_dotted_prefixes_only",
            "source_labels_total": len(alignment_table),
            "source_labels_linked": int(alignment_table["term_id"].ne("").sum()),
            "source_labels_with_missing_words": len(missing_word_rows),
            "source_labels_other_unresolved": len(unresolved_rows),
            "source_label_match_counts": {str(k): int(v) for k, v in alignment_table.groupby("link_status").size().items()}}
    }


def read_cest_package(path, source_document_path=None):
    """Read a reviewed JSON transcription of the CEST source, never a guessed list.

    JSON format: {schema_version, metadata, terms:[], synonyms:[], attributes:[]}.
    metadata must identify a local original source document and preserve
    its copyright notice. Table source_locator fields identify source page/table.
    Clinical mappings to Kvasir are reviewed separately from this transcription.
    Optional metadata.source_sha256 is checked against the original document.
    source_document_path can point to its verified VM copy without modifying the
    JSON package bytes or their fingerprint. Output table
    source_sha256 identifies both that document and the exact transcription bytes,
    preventing an approved map from surviving an edited transcription unnoticed.
    """
    path = Path(path)
    package_bytes = path.read_bytes()
    package_hash = hashlib.sha256(package_bytes).hexdigest()
    payload = json.loads(package_bytes.decode("utf-8-sig"))
    if payload.get("schema_version") != "mmvqa-cest-transcription-v1":
        raise ValueError("Unsupported CEST transcription schema")
    metadata = payload.get("metadata", {})
    required = ["version", "citation", "source_url", "source_file",
                "copyright_notice", "transcription_reviewed_by", "transcription_reviewed_at"]
    missing = [key for key in required if not str(metadata.get(key, "")).strip()]
    if missing: raise ValueError("CEST source metadata missing: " + ", ".join(missing))
    original = Path(source_document_path).expanduser() if source_document_path is not None else Path(metadata["source_file"])
    if not original.is_absolute(): original = path.parent / original
    if not original.is_file(): raise FileNotFoundError(f"CEST source document missing: {original}")
    source_hash = hashlib.sha256(original.read_bytes()).hexdigest()
    asserted_hash = str(metadata.get("source_sha256") or "").strip()
    if asserted_hash and source_hash != asserted_hash:
        raise ValueError("CEST source document hash mismatch")
    source_fingerprint = hashlib.sha256(
        b"mmvqa-cest-source-snapshot-v1\0" + bytes.fromhex(source_hash) + bytes.fromhex(package_hash)
    ).hexdigest()
    records = payload.get("terms", [])
    if not records: raise ValueError("CEST transcription contains no terms")
    terms, synonyms, attributes = [], [], []
    for item in records:
        for key in ["term_id", "preferred_term", "term_type", "source_locator"]:
            if not str(item.get(key, "")).strip(): raise ValueError(f"CEST term missing {key}")
        terms.append(dict(zip(MST_TERM_COLUMNS, ["CEST", metadata["version"], item["term_id"],
            item["preferred_term"], mst_lookup_key(item["preferred_term"]), item["term_type"],
            item.get("section", ""), item.get("organ", ""), metadata["source_url"], item["source_locator"], source_fingerprint])))
    term_ids = [t["term_id"] for t in terms]
    if len(set(term_ids)) != len(term_ids): raise ValueError("Duplicate CEST term IDs")
    for item in payload.get("synonyms", []):
        if item.get("term_id") not in term_ids: raise ValueError("CEST synonym references unknown term")
        if not item.get("source_locator") or not item.get("synonym") or not item.get("evidence"):
            raise ValueError("CEST synonym requires source locator, synonym, and source evidence")
        if item.get("relation") != "exact_synonym":
            raise ValueError("Only source-attested exact_synonym relations enter CEST synonyms")
        synonyms.append(dict(zip(MST_SYNONYM_COLUMNS, [item["term_id"], item["synonym"], mst_lookup_key(item["synonym"]),
            item.get("language", "en"), "source_attested_transcription", item["source_locator"], metadata["source_url"], source_fingerprint])))
    for item in payload.get("attributes", []):
        if item.get("term_id") not in term_ids or not item.get("attribute") or not item.get("source_locator"):
            raise ValueError("Invalid CEST attribute reference")
        attributes.append(dict(zip(MST_ATTRIBUTE_COLUMNS, [item["term_id"], item["attribute"], item.get("value", ""),
            item.get("value_kind", "enumerated"), item["source_locator"], metadata["source_url"], source_fingerprint, "reviewed_transcription"])))
    return {"terms": _mst_table(terms, MST_TERM_COLUMNS), "synonyms": _mst_table(synonyms, MST_SYNONYM_COLUMNS),
            "attributes": _mst_table(attributes, MST_ATTRIBUTE_COLUMNS),
            "applicability": _mst_table([], MST_APPLICABILITY_COLUMNS),
            "manifest": {**metadata, "terminology": "CEST", "status": "imported_reviewed_transcription",
                         "source_sha256": source_fingerprint,
                         "source_fingerprint": source_fingerprint,
                         "source_fingerprint_scheme": "sha256(domain_prefix+original_document_digest+package_digest)",
                         "original_document_sha256": source_hash,
                         "declared_original_document_sha256": asserted_hash or None,
                         "declared_original_hash_verified": bool(asserted_hash),
                         "package_sha256": package_hash,
                         "parser_version": "cest-reviewed-transcription-v1"}}


def empty_cest_package_template():
    """An explicitly empty template; it does not pretend to contain CEST."""
    return {"schema_version": "mmvqa-cest-transcription-v1", "metadata": {
        "version": "2005", "citation": "Korman et al. Endoscopy 2005;37:951-959. doi:10.1055/s-2005-870329",
        "source_url": "https://doi.org/10.1055/s-2005-870329", "source_file": "", "source_sha256": "",
        "copyright_notice": "", "transcription_reviewed_by": "", "transcription_reviewed_at": ""},
        "terms": [], "synonyms": [], "attributes": []}

"""Load the GI source supplement from current GitHub bytes and a SHA-256 manifest.

No source terminology is embedded here. Integrity checks establish repository
artifact identity; they do not independently validate medical assertions.
"""



GI_GITHUB_MANIFEST_SCHEMA = "mmvqa-gi-github-manifest-v1"
GI_GITHUB_PACKAGE_SCHEMA = "mmvqa-gi-supplement-v1"


def _gi_github_validate_manifest(payload):
    if len(payload) > 1024 * 1024:
        raise ValueError("GI GitHub manifest exceeds the supported size")
    manifest = _rag_github_json(payload)
    if not isinstance(manifest, dict) or manifest.get("schema_version") != GI_GITHUB_MANIFEST_SCHEMA:
        raise ValueError("Unsupported GI GitHub manifest schema")
    artifact = manifest.get("artifact")
    if not isinstance(artifact, dict):
        raise ValueError("GI GitHub manifest requires one artifact record")
    _rag_github_path(artifact.get("path"))
    if type(artifact.get("bytes")) is not int or not 0 < artifact["bytes"] <= 32 * 1024 * 1024:
        raise ValueError("GI artifact bytes must be a positive integer within 32 MiB")
    _rag_github_sha(artifact.get("sha256"), "artifact.sha256")
    _rag_github_sha(manifest.get("package_content_sha256"), "package_content_sha256")
    if artifact.get("schema_version") != GI_GITHUB_PACKAGE_SCHEMA:
        raise ValueError("Unsupported GI supplement schema in GitHub manifest")
    return manifest


def _gi_github_require_text(record, fields, context):
    if not isinstance(record, dict) or any(
        not isinstance(record.get(key), str) or not record[key].strip() for key in fields
    ):
        raise ValueError("GI supplement requires nonempty text fields for " + context)


def _gi_github_validate_package(payload, manifest):
    artifact = manifest["artifact"]
    if len(payload) != artifact["bytes"]:
        raise ValueError("GI supplement byte count differs from the GitHub manifest")
    if hashlib.sha256(payload).hexdigest() != artifact["sha256"]:
        raise ValueError("GI supplement SHA-256 differs from the GitHub manifest")
    package = _rag_github_json(payload)
    if not isinstance(package, dict) or package.get("schema_version") != GI_GITHUB_PACKAGE_SCHEMA:
        raise ValueError("GI supplement JSON has an unsupported schema")
    _gi_github_require_text(package, (
        "terminology", "version", "source_url", "doi", "source_locator", "citation",
        "scope", "patient_attribute_policy",
    ), "source identity and scope")
    for key in ("terms", "synonyms", "not_synonyms", "excluded_terms"):
        if not isinstance(package.get(key), list):
            raise ValueError("GI supplement requires a " + key + " list")
    if not package["terms"]:
        raise ValueError("GI supplement requires at least one source term")
    term_ids = set()
    for term in package["terms"]:
        _gi_github_require_text(term, (
            "term_id", "preferred_term", "source_label", "term_type", "section", "source_locator",
        ), "term")
        if term["term_id"] in term_ids:
            raise ValueError("GI supplement has duplicate term IDs")
        term_ids.add(term["term_id"])
    for synonym in package["synonyms"]:
        _gi_github_require_text(synonym, (
            "term_id", "synonym", "language", "relation", "source_locator", "evidence",
        ), "source-attested synonym")
        if synonym["term_id"] not in term_ids:
            raise ValueError("GI synonym references an unknown source term")
        if synonym["relation"] != "source_attested_equivalent_name":
            raise ValueError("Unsupported GI synonym relation")
    for pair in package["not_synonyms"]:
        if (not isinstance(pair, list) or len(pair) != 2
                or any(not isinstance(value, str) or value not in term_ids for value in pair)
                or pair[0] == pair[1]):
            raise ValueError("GI non-synonym relation requires two distinct known term IDs")
    for excluded in package["excluded_terms"]:
        _gi_github_require_text(excluded, ("label", "reason"), "excluded source term")
    canonical = json.dumps(package, ensure_ascii=False, sort_keys=True, separators=(",", ":"), allow_nan=False)
    if hashlib.sha256(canonical.encode("utf-8")).hexdigest() != manifest["package_content_sha256"]:
        raise ValueError("GI supplement canonical content hash differs from the GitHub manifest")
    return package


def load_github_gi_supplement(repository, ref, manifest_path, cache_root, fetch, attempts=3):
    """Return the GI package and its existing provenance from the current ref."""
    return load_github_verified_artifact(
        {"repository": repository, "ref": ref, "manifest_path": manifest_path,
         "cache_root": cache_root, "fetch": fetch, "attempts": attempts,
         "label": "GI", "artifact_label": "GI supplement", "cache_namespace": "gi",
         "provenance_schema": "mmvqa-gi-github-source-v1",
         "provenance_fields": ("package_content_sha256",)},
        manifest_validator=_gi_github_validate_manifest,
        artifact_validator=_gi_github_validate_package,
    )


GI_GITHUB_CONFIG = {
    "repository": CONFIG["repository"],
    "ref": "main",
    "manifest_path": "outputs/phase3/sector1/gi_supplement_manifest.json",
    "download_attempts": 3,
}
GI_SUPPLEMENT_PACKAGE, GI_SOURCE_PROVENANCE = load_github_gi_supplement(
    repository=GI_GITHUB_CONFIG["repository"], ref=GI_GITHUB_CONFIG["ref"],
    manifest_path=GI_GITHUB_CONFIG["manifest_path"], cache_root=WORK / "gi_github_cache",
    fetch=STORE._download, attempts=GI_GITHUB_CONFIG["download_attempts"],
)
write_json(RUN_DIR / "gi_source_provenance.json", GI_SOURCE_PROVENANCE)
print("GI supplement SHA-256 on the VM matches the current GitHub manifest.")
display(pd.DataFrame([{key: GI_SOURCE_PROVENANCE[key] for key in (
    "repository_ref", "source_sha256", "vm_sha256", "verification_status", "cache_status"
)}]))

MST_MAPPING_PROPOSALS = json.loads('{\n  "schema_version": "mmvqa-mst-mapping-proposals-v1",\n  "status": "proposals_not_approved",\n  "source_url": "https://www.worldendo.org/assets-craft/pdf/resources/mst30_tables.xls",\n  "rows": [\n    {\n      "class_label": "Ampulla of Vater",\n      "target_terminology": "MST",\n      "target_id": "MST3:Anatomy:R35:C2",\n      "relation": "related",\n      "status": "candidate_for_review",\n      "source_sha256": "f4cf8e2fb50b74d9910295974b96f34c5bcb2813a8eb55796fb9e4ced5b98564",\n      "justification": "MST lists Major papilla in duodenal anatomy. Do not assert exact equivalence between the ampulla and its visible papilla without clinical review.",\n      "required_attribute_constraints": []\n    },\n    {\n      "class_label": "Angiectasia",\n      "target_terminology": "MST",\n      "target_id": "MST3:Luminal_findings:R67:C2",\n      "relation": "exact",\n      "status": "candidate_for_review",\n      "source_sha256": "f4cf8e2fb50b74d9910295974b96f34c5bcb2813a8eb55796fb9e4ced5b98564",\n      "justification": "Same source label in the luminal Flat lesions section.",\n      "required_attribute_constraints": []\n    },\n    {\n      "class_label": "Blood - fresh",\n      "target_terminology": "MST",\n      "target_id": "MST3:Luminal_findings:R34:C2",\n      "relation": "compositional",\n      "status": "candidate_for_review",\n      "source_sha256": "f4cf8e2fb50b74d9910295974b96f34c5bcb2813a8eb55796fb9e4ced5b98564",\n      "justification": "MST Blood with Type=Fresh preserves the class qualifier.",\n      "required_attribute_constraints": [\n        {\n          "attribute": "Type",\n          "value": "Fresh",\n          "source_locator": "Luminal_findings!R34C3:C4"\n        }\n      ]\n    },\n    {\n      "class_label": "Blood - hematin",\n      "target_terminology": "MST",\n      "target_id": "MST3:Luminal_findings:R34:C2",\n      "relation": "compositional",\n      "status": "candidate_for_review",\n      "source_sha256": "f4cf8e2fb50b74d9910295974b96f34c5bcb2813a8eb55796fb9e4ced5b98564",\n      "justification": "MST Blood with Type=Hematin preserves the class qualifier.",\n      "required_attribute_constraints": [\n        {\n          "attribute": "Type",\n          "value": "Hematin",\n          "source_locator": "Luminal_findings!R36C4"\n        }\n      ]\n    },\n    {\n      "class_label": "Erosion",\n      "target_terminology": "MST",\n      "target_id": "MST3:Luminal_findings:R134:C2",\n      "relation": "exact",\n      "status": "candidate_for_review",\n      "source_sha256": "f4cf8e2fb50b74d9910295974b96f34c5bcb2813a8eb55796fb9e4ced5b98564",\n      "justification": "Same source label in the Excavated lesions section.",\n      "required_attribute_constraints": []\n    },\n    {\n      "class_label": "Erythema",\n      "target_terminology": "MST",\n      "target_id": "MST3:Luminal_findings:R46:C2",\n      "relation": "related",\n      "status": "candidate_for_review",\n      "source_sha256": "f4cf8e2fb50b74d9910295974b96f34c5bcb2813a8eb55796fb9e4ced5b98564",\n      "justification": "MST uses Erythematous under Mucosa; lexical morphology and clinical scope need review, not automatic synonym substitution.",\n      "required_attribute_constraints": []\n    },\n    {\n      "class_label": "Foreign body",\n      "target_terminology": "MST",\n      "target_id": "MST3:Luminal_findings:R33:C2",\n      "relation": "exact",\n      "status": "candidate_for_review",\n      "source_sha256": "f4cf8e2fb50b74d9910295974b96f34c5bcb2813a8eb55796fb9e4ced5b98564",\n      "justification": "Same source label in the Contents section; no type value inferred.",\n      "required_attribute_constraints": []\n    },\n    {\n      "class_label": "Ileocecal valve",\n      "target_terminology": "MST",\n      "target_id": "MST3:Anatomy:R68:C2",\n      "relation": "exact",\n      "status": "candidate_for_review",\n      "source_sha256": "f4cf8e2fb50b74d9910295974b96f34c5bcb2813a8eb55796fb9e4ced5b98564",\n      "justification": "Same source label as an anatomical site, not a pathology.",\n      "required_attribute_constraints": []\n    },\n    {\n      "class_label": "Lymphangiectasia",\n      "target_terminology": "MST",\n      "target_id": "MST3:Diagnoses:R92:C2",\n      "relation": "exact",\n      "status": "candidate_for_review",\n      "source_sha256": "f4cf8e2fb50b74d9910295974b96f34c5bcb2813a8eb55796fb9e4ced5b98564",\n      "justification": "Same source label in Enteroscopy diagnoses. Preserve dataset observation role separately from the source diagnosis category.",\n      "required_attribute_constraints": []\n    },\n    {\n      "class_label": "Normal clean mucosa",\n      "target_terminology": "MST",\n      "target_id": "",\n      "relation": "unmapped",\n      "status": "unmapped",\n      "source_sha256": "f4cf8e2fb50b74d9910295974b96f34c5bcb2813a8eb55796fb9e4ced5b98564",\n      "justification": "MST Normal under Lumen is insufficient to assert exact equivalence to normal clean mucosa; keep the dataset label.",\n      "required_attribute_constraints": []\n    },\n    {\n      "class_label": "Polyp",\n      "target_terminology": "MST",\n      "target_id": "MST3:Luminal_findings:R90:C2",\n      "relation": "exact",\n      "status": "candidate_for_review",\n      "source_sha256": "f4cf8e2fb50b74d9910295974b96f34c5bcb2813a8eb55796fb9e4ced5b98564",\n      "justification": "Same source label in Protruding lesions; morphology and histology are not inferred.",\n      "required_attribute_constraints": []\n    },\n    {\n      "class_label": "Pylorus",\n      "target_terminology": "MST",\n      "target_id": "MST3:Anatomy:R22:C2",\n      "relation": "exact",\n      "status": "candidate_for_review",\n      "source_sha256": "f4cf8e2fb50b74d9910295974b96f34c5bcb2813a8eb55796fb9e4ced5b98564",\n      "justification": "Same source label as an anatomical site under Stomach, not a pathology.",\n      "required_attribute_constraints": []\n    },\n    {\n      "class_label": "Reduced mucosal view",\n      "target_terminology": "MST",\n      "target_id": "",\n      "relation": "unmapped",\n      "status": "unmapped",\n      "source_sha256": "f4cf8e2fb50b74d9910295974b96f34c5bcb2813a8eb55796fb9e4ced5b98564",\n      "justification": "No exact term identified in the imported MST tables; this is a visibility condition, not a disease.",\n      "required_attribute_constraints": []\n    },\n    {\n      "class_label": "Ulcer",\n      "target_terminology": "MST",\n      "target_id": "MST3:Luminal_findings:R148:C2",\n      "relation": "exact",\n      "status": "candidate_for_review",\n      "source_sha256": "f4cf8e2fb50b74d9910295974b96f34c5bcb2813a8eb55796fb9e4ced5b98564",\n      "justification": "Same source label in Excavated lesions; grade, site and size are not inferred.",\n      "required_attribute_constraints": []\n    }\n  ]\n}\n')


GitHub read: /bogdanparvu18/msc-graduate-project/main/outputs/phase3/sector1/gi_supplement_manifest.json
GitHub read: /bogdanparvu18/msc-graduate-project/main/outputs/phase3/sector1/gi_supplement_package.json
GitHub read: /bogdanparvu18/msc-graduate-project/main/outputs/phase3/sector1/gi_supplement_manifest.json
GI supplement SHA-256 on the VM matches the current GitHub manifest.


,repository_ref,source_sha256,vm_sha256,verification_status,cache_status
0,main,3774bd3f09b5bfb53b9143d3b2acb0c215a10c69eaf99b...,3774bd3f09b5bfb53b9143d3b2acb0c215a10c69eaf99b...,verified,downloaded_and_verified


## 6. SNOMED CT RF2 Snapshot import functions

Read the verified VM copy of an RF2 Snapshot ZIP or extracted complete release. Import active concepts/descriptions, inferred relationships and language-reference-set acceptability. Keep SCTIDs as strings and retain exact component hashes, edition tags and release dates. Accept language-neutral component filenames, language suffixes such as `-en`, and regional suffixes such as `-en-CA`. For multilingual files, select descriptions by their actual `languageCode` (English by default), then discover usable language refsets through their links to those active descriptions. French-only refsets do not become default English refsets. Preserve dialect-specific preferences separately. An explicit refset subset is optional and must be available for the selected language.

The [SNOMED file naming specification](https://docs.snomed.org/snomed-ct-specifications/snomed-ct-release-file-specification/release-types-packages-and-files/3.3-naming-conventions-for-release-packages-and-files/3.3.2-release-file-naming-convention) makes language suffixes conditional. The [Canadian Edition](https://accelero.infoway-inforoute.ca/en/standards/terminology-standards/snomed-ct-snomed-ct-ca) combines Canadian English/French and International content. Section 7 displays and records the Snapshot component inventory before parsing. Component ambiguity, mismatched editions/dates, invalid schemas and missing dependencies remain errors; the importer does not substitute Full or Delta files.

`refsetId` identifies a language/dialect reference set, not a release or byte-equivalent file. File/directory SHA-256 establishes Drive-to-VM identity. RF2 component hashes, release metadata and discovered refsets separately describe the parsed snapshot.

The eight SNOMED metadata aliases are populated from the imported RF2 content, with no numeric SCTIDs pinned in the notebook. Exact English metadata names identify active concepts; uniqueness and the discovered FSN type are checked before use. Mandatory structural roles must be present; US/GB aliases can be absent without restricting the language refsets discovered for the selected language. Section 7 displays the discovered IDs and stores their description IDs, source component hashes and field-usage counts in the Drive manifest and metadata audit. Semantic labels remain explicit parser anchors; meanings are not inferred from frequency or row order.

Import and mapping eligibility are reported separately. Multiple preferred descriptions for the same concept/type/refset are retained as source conflicts, with exact description/member IDs, terms, modules and effective dates. Such concepts remain in the graph and candidate audit but cannot supply canonical labels or approved class mappings. An explicit review cannot bypass this guard. Multiple active FSNs across dialects are resolved through unique preferred FSNs and the recorded refset display order; unresolved FSN choice is also retained for review. No first-row, newest-date or module-based repair is performed. If a class has a conflicted candidate, competing candidates are not approved by silently ignoring it. Phase 2 classes and annotations are retained.

A preferred or acceptable description is accepted **by the terminology source**; it is not automatically an approved synonym for a Kvasir class. Lexical matches are checked against the independently supported RAG class profiles in section 8 before scoped class-to-concept acceptance. Broader, narrower, related and compositional mappings cannot export interchangeable class aliases. An extension-only package without its dependencies is rejected.

Only the class-related candidate/approved concepts and their ancestors are exported as the local subset. This is not a full GI ontology, an OWL reasoner, an MRCM implementation or a clinical-truth validator. [SNOMED release format](https://docs.snomed.org/snomed-ct-practical-guides/snomed-ct-starter-guide/13-release-schedule-and-file-formats).


In [6]:
"""Local RF2 Snapshot import and reviewed SNOMED terminology mapping.

No network calls; no terminology content is bundled. Read an authorised local
RF2 Snapshot ZIP or directory. Lexical matches remain candidates until reviewed.
This module does not implement SNOMED classification, MRCM or QA generation.
"""



# Notebook compatibility aliases are populated only after a verified RF2 import.
# Import and graph functions use the explicit registry of their own snapshot.
SNOMED_FSN = None
SNOMED_SYNONYM = None
SNOMED_PREFERRED = None
SNOMED_ACCEPTABLE = None
SNOMED_US_ENGLISH = None
SNOMED_GB_ENGLISH = None
SNOMED_INFERRED = None
SNOMED_IS_A = None

SNOMED_METADATA_SPECIFICATION = {
    "SNOMED_FSN": {"fsn": "Fully specified name (core metadata concept)", "component": "descriptions", "field": "typeId", "required": True},
    "SNOMED_SYNONYM": {"fsn": "Synonym (core metadata concept)", "component": "descriptions", "field": "typeId", "required": True},
    "SNOMED_PREFERRED": {"fsn": "Preferred (foundation metadata concept)", "component": "language_refset", "field": "acceptabilityId", "required": True},
    "SNOMED_ACCEPTABLE": {"fsn": "Acceptable (foundation metadata concept)", "component": "language_refset", "field": "acceptabilityId", "required": True},
    "SNOMED_US_ENGLISH": {"fsn": "United States of America English language reference set (foundation metadata concept)", "component": "language_refset", "field": "refsetId", "required": False},
    "SNOMED_GB_ENGLISH": {"fsn": "Great Britain English language reference set (foundation metadata concept)", "component": "language_refset", "field": "refsetId", "required": False},
    "SNOMED_INFERRED": {"fsn": "Inferred relationship (core metadata concept)", "component": "relationships", "field": "characteristicTypeId", "required": True},
    "SNOMED_IS_A": {"fsn": "Is a (attribute)", "component": "relationships", "field": "typeId", "required": True},
}
SNOMED_METADATA_DISCOVERY_VERSION = "snomed-rf2-metadata-discovery-v1"

SNOMED_RF2_COLUMNS = {
    "concepts": ["id", "effectiveTime", "active", "moduleId", "definitionStatusId"],
    "descriptions": ["id", "effectiveTime", "active", "moduleId", "conceptId", "languageCode", "typeId", "term", "caseSignificanceId"],
    "relationships": ["id", "effectiveTime", "active", "moduleId", "sourceId", "destinationId", "relationshipGroup", "typeId", "characteristicTypeId", "modifierId"],
    "language_refset": ["id", "effectiveTime", "active", "moduleId", "refsetId", "referencedComponentId", "acceptabilityId"],
}


def snomed_lexical_key(value):
    """Conservative lexical key; no stemming, synonym invention or tag removal."""
    return " ".join(unicodedata.normalize("NFKC", str(value)).casefold().split())


def snomed_discover_metadata(tables, component_files=None):
    """Discover metadata identifiers from active English RF2 metadata FSNs.

    Metadata names are an explicit semantic specification, not inferred medical
    meanings. Numeric identifiers are read from this edition. Bootstrap looks
    up full labels before knowing the FSN type, then verifies the discovered
    FSN identifier against every matched description, including its own row.
    Optional dialect concepts never constrain actual language-refset discovery.
    """
    components = dict(component_files or {})
    concepts = tables["concepts"].loc[tables["concepts"]["active"].eq("1")]
    descriptions = tables["descriptions"]
    descriptions = descriptions.loc[
        descriptions["active"].eq("1") & descriptions["languageCode"].eq("en")
        & descriptions["conceptId"].isin(set(concepts["id"]))
    ]
    wanted = {snomed_lexical_key(spec["fsn"]) for spec in SNOMED_METADATA_SPECIFICATION.values()}
    label_keys = descriptions["term"].map(snomed_lexical_key)
    matched = descriptions.loc[label_keys.isin(wanted)].copy()
    matched["_metadata_key"] = label_keys.loc[matched.index]
    ids, matched_rows = {}, {}
    for role, spec in SNOMED_METADATA_SPECIFICATION.items():
        rows = matched.loc[matched["_metadata_key"].eq(snomed_lexical_key(spec["fsn"]))]
        candidates = sorted(set(rows["conceptId"]))
        if len(candidates) > 1:
            raise ValueError(f"Ambiguous SNOMED metadata {role}: exact active English FSN matches concepts {candidates}")
        if not candidates:
            if spec["required"]:
                raise ValueError(f"Required SNOMED metadata {role} is missing: {spec['fsn']!r}. Supply a complete edition with active English metadata descriptions.")
            ids[role], matched_rows[role] = None, rows
            continue
        identifier = candidates[0]
        if not isinstance(identifier, str) or not re.fullmatch(r"[0-9]+", identifier):
            raise ValueError(f"SNOMED metadata {role} has a nonnumeric concept identifier")
        ids[role], matched_rows[role] = identifier, rows
    resolved_ids = [identifier for identifier in ids.values() if identifier is not None]
    if len(resolved_ids) != len(set(resolved_ids)):
        raise ValueError("Different SNOMED metadata roles resolve to the same concept")
    fsn_id = ids["SNOMED_FSN"]
    for role, rows in matched_rows.items():
        if not rows.empty and not rows["typeId"].eq(fsn_id).all():
            raise ValueError(f"SNOMED metadata {role} label is not consistently typed by the discovered FSN concept")
    audit = []
    for role, spec in SNOMED_METADATA_SPECIFICATION.items():
        identifier, rows = ids[role], matched_rows[role]
        component, field = spec["component"], spec["field"]
        active_values = tables[component].loc[tables[component]["active"].eq("1"), field]
        use_count = int(active_values.eq(identifier).sum()) if identifier is not None else 0
        concept_rows = concepts.loc[concepts["id"].eq(identifier)]
        source_terms = sorted(set(rows["term"]))
        audit.append({
            "role": role, "sctid": identifier or "", "expected_fsn": spec["fsn"],
            "matched_fsn": source_terms[0] if source_terms else "", "required": spec["required"],
            "matched_terms_json": json.dumps(source_terms, ensure_ascii=False),
            "status": "discovered" if identifier is not None else "optional_metadata_not_present",
            "description_ids_json": json.dumps(sorted(set(rows["id"]))),
            "concept_module_ids_json": json.dumps(sorted(set(concept_rows["moduleId"]))),
            "description_module_ids_json": json.dumps(sorted(set(rows["moduleId"]))),
            "description_effective_times_json": json.dumps(sorted(set(rows["effectiveTime"]))),
            "role_component": component, "role_field": field, "role_use_count": use_count,
            "role_usage_status": "observed_in_expected_field" if use_count else "not_used_in_active_component",
            "description_component_path": components.get("descriptions", {}).get("path", ""),
            "description_component_sha256": components.get("descriptions", {}).get("sha256", ""),
            "concept_component_path": components.get("concepts", {}).get("path", ""),
            "concept_component_sha256": components.get("concepts", {}).get("sha256", ""),
            "role_component_path": components.get(component, {}).get("path", ""),
            "role_component_sha256": components.get(component, {}).get("sha256", ""),
        })
    return {
        "ids": ids, "audit": pd.DataFrame(audit),
        "policy_version": SNOMED_METADATA_DISCOVERY_VERSION,
        "policy": "unique_active_English_full_metadata_label_then_FSN_self_validation_and_field_usage_audit",
        "source_components": components,
    }


def snomed_sha256_stream(stream):
    digest = hashlib.sha256()
    for block in iter(lambda: stream.read(1024 * 1024), b""):
        digest.update(block)
    return digest.hexdigest()


@contextmanager
def snomed_rf2_source(path):
    source = Path(path).expanduser().resolve()
    if not source.exists():
        raise FileNotFoundError(f"SNOMED RF2 source does not exist: {source}")
    if source.is_file():
        if not zipfile.is_zipfile(source):
            raise ValueError("SNOMED source must be an RF2 ZIP or an extracted RF2 directory.")
        with zipfile.ZipFile(source) as archive:
            names = [item.filename for item in archive.infolist() if not item.is_dir()]
            if len(names) != len(set(names)):
                raise ValueError("RF2 ZIP contains duplicate member names.")
            def opener(name):
                parts = PurePosixPath(name).parts
                if PurePosixPath(name).is_absolute() or ".." in parts:
                    raise ValueError("Unsafe RF2 ZIP member path.")
                return archive.open(name, "r")
            yield source, sorted(names), opener
    else:
        files = sorted(p for p in source.rglob("*.txt") if p.is_file())
        names = [p.relative_to(source).as_posix() for p in files]
        def opener(name):
            selected = (source / name).resolve()
            if not selected.is_relative_to(source):
                raise ValueError("RF2 path resolves outside the selected source directory.")
            return selected.open("rb")
        yield source, names, opener


def snomed_rf2_component_inventory(names, config=None):
    """Describe supported Snapshot filenames without reading or accepting content.

    Bilingual editions can omit the filename language suffix. Regional suffixes
    such as en-CA are compatible with languageCode=en; the rows remain the
    authority for language selection. No Full, Delta or TextDefinition file is
    treated as a Description Snapshot.
    """
    language = (config or {}).get("language_code", "en")
    if not isinstance(language, str) or not re.fullmatch(r"[a-z]{2}", language):
        raise ValueError("language_code must be a two-letter ISO language code.")
    optional_language = r"(?:-(?P<language>[a-z]{2}(?:-[A-Za-z0-9]{2,8})*))?"
    suffix = r"_(?P<edition>[^/]+)_(?P<date>\d{8})\.txt"
    patterns = {
        "concepts": r"sct2_Concept_Snapshot" + suffix,
        "descriptions": r"sct2_Description_Snapshot" + optional_language + suffix,
        "relationships": r"sct2_Relationship_Snapshot" + suffix,
        "language_refset": r"der2_cRefset_LanguageSnapshot" + optional_language + suffix,
    }
    rows = []
    for name in sorted(names):
        for kind, pattern in patterns.items():
            match = re.fullmatch(pattern, PurePosixPath(name).name)
            if match:
                filename_language = match.groupdict().get("language") or ""
                rows.append({"kind": kind, "path": name,
                             "filename_language": filename_language,
                             "release_date": match.group("date"),
                             "edition_tag": match.group("edition"),
                             "language_compatible": not filename_language or
                                 filename_language.split("-", 1)[0] == language})
                break
    return rows


def snomed_rf2_select_files(names, config):
    inventory = snomed_rf2_component_inventory(names, config)
    explicit = config.get("component_paths", {})
    if not isinstance(explicit, dict) or set(explicit) - set(SNOMED_RF2_COLUMNS):
        raise ValueError("component_paths must map supported RF2 component kinds to exact paths.")
    selected, dates, edition_tags = {}, set(), set()
    for kind in SNOMED_RF2_COLUMNS:
        available = [row for row in inventory if row["kind"] == kind]
        matches = [row for row in available if row["language_compatible"]]
        if kind in explicit:
            name = str(explicit[kind])
            matches = [row for row in matches if row["path"] == name]
            if not matches:
                raise ValueError(
                    f"Configured {kind} component is absent, incompatible with the selected language, "
                    f"or is not a supported Snapshot file: {name}. "
                    f"Available {kind} Snapshot files: {[row['path'] for row in available]}"
                )
        if len(matches) != 1:
            raise ValueError(
                f"Expected exactly one {kind} Snapshot file, found {len(matches)}. "
                f"Available {kind} Snapshot files: {[row['path'] for row in available]}. "
                "Use one complete release/edition, or set component_paths explicitly. "
                "Language-neutral and compatible regional filenames are supported; "
                "Full/Delta files and separate edition dependencies are not merged automatically."
            )
        match = matches[0]
        selected[kind] = match["path"]
        dates.add(match["release_date"])
        edition_tags.add(match["edition_tag"])
    if len(edition_tags) != 1:
        raise ValueError(f"RF2 component edition tags differ: {sorted(edition_tags)}; supply one complete edition.")
    if len(dates) != 1:
        raise ValueError(f"RF2 components have different release dates: {sorted(dates)}")
    release = next(iter(dates))
    requested = config.get("release_date")
    if requested and str(requested) != release:
        raise ValueError(f"Requested SNOMED release {requested} does not match RF2 filenames ({release}).")
    return selected, release


def snomed_rf2_read_active(opener, name, kind, release, chunk_size):
    required = SNOMED_RF2_COLUMNS[kind]
    pieces, seen, total_rows = [], set(), 0
    with opener(name) as stream:
        source_sha256 = snomed_sha256_stream(stream)
    with opener(name) as stream, pd.read_csv(
        stream, sep="\t", dtype=str, keep_default_na=False,
        chunksize=chunk_size, encoding="utf-8-sig"
    ) as reader:
        for chunk in reader:
            missing = set(required) - set(chunk.columns)
            if missing:
                raise ValueError(f"RF2 {kind} missing columns: {sorted(missing)}")
            chunk = chunk[required]
            total_rows += len(chunk)
            if chunk["id"].eq("").any() or chunk["id"].duplicated().any():
                raise ValueError(f"RF2 {kind} has empty or duplicate Snapshot component IDs.")
            ids = set(chunk["id"])
            if seen.intersection(ids):
                raise ValueError(f"RF2 {kind} has duplicate IDs across Snapshot chunks.")
            seen.update(ids)
            if not chunk["active"].isin(["0", "1"]).all():
                raise ValueError(f"RF2 {kind} has invalid active flags.")
            dates = chunk["effectiveTime"]
            if not dates.str.fullmatch(r"\d{8}").all() or (dates > release).any():
                raise ValueError(f"RF2 {kind} contains unpublished/future effectiveTime values.")
            active = chunk.loc[chunk["active"].eq("1")].copy()
            if not active.empty:
                pieces.append(active)
    result = pd.concat(pieces, ignore_index=True) if pieces else pd.DataFrame(columns=required)
    return result, {"path": name, "sha256": source_sha256,
                    "snapshot_rows": total_rows, "active_rows": len(result)}


SNOMED_DESCRIPTION_ISSUE_COLUMNS = [
    "concept_id", "description_id", "language_member_id", "refset_id", "type_id", "term",
    "description_module_id", "language_module_id", "description_effective_time",
    "language_effective_time", "reason_code",
]


def snomed_resolve_descriptions(concepts, descriptions, terms, refsets, metadata_ids):
    """Resolve display labels while preserving and quarantining local ambiguity.

    A declared refset order chooses among individually unambiguous dialects.
    It never chooses between two preferred descriptions in the same group.
    All active source descriptions and accepted memberships remain untouched.
    """
    SNOMED_FSN = metadata_ids["SNOMED_FSN"]
    SNOMED_SYNONYM = metadata_ids["SNOMED_SYNONYM"]
    SNOMED_PREFERRED = metadata_ids["SNOMED_PREFERRED"]
    preferred = terms.loc[terms["acceptabilityId"].eq(SNOMED_PREFERRED)].copy()
    duplicate_mask = preferred.duplicated(["conceptId", "typeId", "refsetId"], keep=False)
    duplicate_preferred = preferred.loc[duplicate_mask]
    preferred_ambiguous = set(duplicate_preferred["conceptId"])
    unique_preferred = preferred.loc[~duplicate_mask].copy()
    rank = {value: i for i, value in enumerate(refsets)}
    unique_preferred["_rank"] = unique_preferred["refsetId"].map(rank)
    unique_preferred.sort_values(["_rank", "id"], kind="stable", inplace=True)

    fsns = descriptions.loc[descriptions["typeId"].eq(SNOMED_FSN)].copy()
    fsns["_semantic_tag"] = fsns["term"].str.extract(r"\(([^()]*)\)$", expand=False).fillna("")
    unique_active_fsn = fsns.loc[~fsns["conceptId"].duplicated(keep=False)]
    fsn_lookup = unique_active_fsn.set_index("conceptId")["term"].to_dict()
    # There is at most one row per concept/refset here. The only selection is
    # the existing declared dialect order, not ID order or latest effectiveTime.
    selected_fsn = unique_preferred.loc[unique_preferred["typeId"].eq(SNOMED_FSN)].drop_duplicates("conceptId")
    fsn_lookup.update(selected_fsn.set_index("conceptId")["term"].to_dict())
    inconsistent_tags = set(fsns.groupby("conceptId")["_semantic_tag"].nunique().loc[lambda x: x > 1].index)
    missing_tags = set(fsns.loc[fsns["_semantic_tag"].eq(""), "conceptId"])
    unresolved_fsn = set(concepts["id"]) - set(fsn_lookup)
    fsn_ambiguous = inconsistent_tags | missing_tags | unresolved_fsn
    ambiguous = preferred_ambiguous | fsn_ambiguous
    for concept_id in ambiguous:
        fsn_lookup.pop(concept_id, None)
    tag_lookup = {cid: re.search(r"\(([^()]*)\)$", term).group(1) for cid, term in fsn_lookup.items()}

    pt = unique_preferred.loc[unique_preferred["typeId"].eq(SNOMED_SYNONYM)]
    display_pt = pt.loc[~pt["conceptId"].isin(ambiguous)].drop_duplicates("conceptId").set_index("conceptId")
    pt_lookup = display_pt["term"].to_dict()
    pt_refset_lookup = display_pt["refsetId"].to_dict()

    # Detailed candidate JSON is stored only for ambiguous concepts or concepts
    # with multiple active FSNs. The raw tables remain authoritative for all
    # other concepts; empty detail JSON does not mean absent descriptions.
    detail_concepts = ambiguous | set(fsns.loc[fsns["conceptId"].duplicated(keep=False), "conceptId"])
    fsn_candidates, fsn_by_refset, pt_candidates, dialect_pt = {}, {}, {}, {}
    detailed_fsns = fsns.loc[fsns["conceptId"].isin(detail_concepts)].sort_values(["conceptId", "id"])
    for cid, did, term, tag in detailed_fsns[["conceptId", "id", "term", "_semantic_tag"]].itertuples(index=False, name=None):
        fsn_candidates.setdefault(cid, []).append({"description_id": did, "term": term, "semantic_tag": tag})
    ordered_terms = terms.loc[terms["conceptId"].isin(detail_concepts)].sort_values(["conceptId", "refsetId", "id"], kind="stable")
    for cid, did, rid, tid, term, acc in ordered_terms[["conceptId", "id", "refsetId", "typeId", "term", "acceptabilityId"]].itertuples(index=False, name=None):
        if tid == SNOMED_FSN:
            fsn_by_refset.setdefault(cid, {}).setdefault(rid, []).append({"description_id": did, "term": term, "acceptability_id": acc})
        elif tid == SNOMED_SYNONYM and acc == SNOMED_PREFERRED:
            pt_candidates.setdefault(cid, {}).setdefault(rid, []).append({"description_id": did, "term": term})
    for cid, rid, term in pt[["conceptId", "refsetId", "term"]].itertuples(index=False, name=None):
        # Duplicate groups have already been excluded; no same-key overwrite.
        dialect_pt.setdefault(cid, {})[rid] = term
    issue_codes = {cid: sorted(
        (["ambiguous_preferred_descriptions"] if cid in preferred_ambiguous else []) +
        (["ambiguous_fsn"] if cid in fsn_ambiguous else [])) for cid in ambiguous}
    def encode(values):
        return {cid: json.dumps(value, sort_keys=True, ensure_ascii=False, separators=(",", ":")) for cid, value in values.items()}
    concepts = concepts.assign(
        fsn=concepts["id"].map(fsn_lookup).fillna(""),
        semantic_tag=concepts["id"].map(tag_lookup).fillna(""),
        preferred_term=concepts["id"].map(pt_lookup).fillna(""),
        preferred_term_refset_id=concepts["id"].map(pt_refset_lookup).fillna(""),
        preferred_terms_by_refset_json=concepts["id"].map(encode(dialect_pt)).fillna("{}"),
        preferred_term_candidates_by_refset_json=concepts["id"].map(encode(pt_candidates)).fillna("{}"),
        fsn_candidates_json=concepts["id"].map(encode(fsn_candidates)).fillna("[]"),
        fsn_candidates_by_refset_json=concepts["id"].map(encode(fsn_by_refset)).fillna("{}"),
        mapping_eligible=~concepts["id"].isin(ambiguous),
        terminology_status=concepts["id"].isin(ambiguous).map({True: "ambiguous_source_descriptions", False: "ready"}),
        terminology_issue_codes_json=concepts["id"].map(encode(issue_codes)).fillna("[]"),
    )

    def issue_rows(frame, code):
        result = frame.rename(columns={"conceptId": "concept_id", "id": "description_id", "refsetId": "refset_id",
                                       "typeId": "type_id", "moduleId": "description_module_id", "effectiveTime": "description_effective_time"}).copy()
        result["reason_code"] = code
        return result.reindex(columns=SNOMED_DESCRIPTION_ISSUE_COLUMNS, fill_value="")
    issues = [issue_rows(duplicate_preferred, "ambiguous_preferred_descriptions")]
    bad_fsns = fsns.loc[fsns["conceptId"].isin(fsn_ambiguous)]
    memberships = terms[["id", "refsetId", "language_member_id", "language_module_id", "language_effective_time"]]
    bad_fsns = bad_fsns.merge(memberships, on="id", how="left", validate="one_to_many").fillna("")
    issues.append(issue_rows(bad_fsns, "ambiguous_fsn"))
    without_fsn = fsn_ambiguous - set(fsns["conceptId"])
    if without_fsn:
        issues.append(issue_rows(pd.DataFrame({"conceptId": sorted(without_fsn)}), "ambiguous_fsn"))
    issues = pd.concat(issues, ignore_index=True).fillna("").drop_duplicates()
    issues.sort_values(["concept_id", "reason_code", "refset_id", "description_id"], kind="stable", inplace=True)
    return concepts, tag_lookup, issues.reset_index(drop=True)


def read_snomed_snapshot(path, config=None):
    """Import one complete, published RF2 Snapshot without modifying the source.

    config:
      release_date: YYYYMMDD, optional; otherwise unique filename date is used.
      edition: a caller-provided label (not inferred or asserted by this parser).
      language_code: default "en".
      language_refset_ids: optional ordered dialect IDs. None/omitted discovers
        active refset IDs linked to active descriptions in the selected language.
        Automatic order is sorted string ID order, not a dialect preference.
      chunk_size: default 100000.
      component_paths: optional exact relative paths for the four component files.

    Both active concepts and descriptions are required for usable terms. Only
    active Preferred/Acceptable language members yield approved source terms.
    Source-term acceptability does NOT approve a dataset-to-concept mapping.
    Relationships retain group/type/characteristic metadata; no OWL reasoning.
    """
    config = dict(config or {})
    chunk_size = int(config.get("chunk_size", 100000))
    if chunk_size < 1:
        raise ValueError("chunk_size must be positive.")
    requested_refsets = config.get("language_refset_ids")
    if requested_refsets is not None:
        if not isinstance(requested_refsets, (list, tuple)):
            raise ValueError("language_refset_ids must be None (discover) or a list of distinct SCTIDs.")
        if any(isinstance(x, bool) or not isinstance(x, (str, int)) for x in requested_refsets):
            raise ValueError("language_refset_ids must contain SCTID strings or integers.")
        requested_refsets = [str(x) for x in requested_refsets]
        if (not requested_refsets or len(requested_refsets) != len(set(requested_refsets))
                or any(not re.fullmatch(r"[0-9]+", x) for x in requested_refsets)):
            raise ValueError("Specify at least one distinct nonempty numeric language_refset_id.")
    if config.get("module_ids"):
        raise ValueError("Do not prune modules during RF2 import: first load a complete edition and then select a justified concept subset.")
    with snomed_rf2_source(path) as (source, names, opener):
        selected, release = snomed_rf2_select_files(names, config)
        tables, members = {}, {}
        for kind, name in selected.items():
            tables[kind], members[kind] = snomed_rf2_read_active(opener, name, kind, release, chunk_size)
    concepts = tables["concepts"]
    concept_ids = set(concepts["id"])
    if not concept_ids:
        raise ValueError("SNOMED Snapshot contains no active concepts.")
    metadata = snomed_discover_metadata(tables, members)
    metadata_ids = metadata["ids"]
    SNOMED_FSN = metadata_ids["SNOMED_FSN"]
    SNOMED_SYNONYM = metadata_ids["SNOMED_SYNONYM"]
    SNOMED_PREFERRED = metadata_ids["SNOMED_PREFERRED"]
    SNOMED_ACCEPTABLE = metadata_ids["SNOMED_ACCEPTABLE"]
    SNOMED_INFERRED = metadata_ids["SNOMED_INFERRED"]
    SNOMED_IS_A = metadata_ids["SNOMED_IS_A"]
    descriptions = tables["descriptions"]
    descriptions = descriptions.loc[
        descriptions["conceptId"].isin(concept_ids) &
        descriptions["languageCode"].eq(config.get("language_code", "en"))
    ].copy()
    language = tables["language_refset"]
    source_active_refsets = sorted(set(language["refsetId"]))
    if not source_active_refsets or any(not re.fullmatch(r"[0-9]+", x) for x in source_active_refsets):
        raise ValueError("The RF2 language component must contain active nonempty numeric refset IDs.")
    if descriptions.empty:
        raise ValueError(f"The RF2 Snapshot has no active descriptions for languageCode={config.get('language_code', 'en')!r} and active concepts.")
    valid_description_ids = set(descriptions["id"])
    # A combined bilingual component can contain French-only reference sets.
    # Those remain in the source audit but cannot become English defaults.
    language = language.loc[language["referencedComponentId"].isin(valid_description_ids)].copy()
    available_refsets = sorted(set(language["refsetId"]))
    if not available_refsets:
        raise ValueError("The RF2 language component has no active refset members linked to active descriptions in the selected language.")
    refsets = available_refsets if requested_refsets is None else requested_refsets
    if set(refsets) - set(available_refsets):
        raise ValueError(f"Selected language refsets are absent for the selected language: {sorted(set(refsets) - set(available_refsets))}")
    language = language.loc[language["refsetId"].isin(refsets)].copy()
    if not language["acceptabilityId"].isin([SNOMED_PREFERRED, SNOMED_ACCEPTABLE]).all():
        raise ValueError("Selected language refsets contain unsupported acceptability IDs.")
    if language.duplicated(["refsetId", "referencedComponentId"]).any():
        raise ValueError("More than one active language member for the same description and dialect.")
    terms = descriptions.merge(
        language.rename(columns={"id": "language_member_id", "effectiveTime": "language_effective_time", "moduleId": "language_module_id", "active": "language_active"}),
        left_on="id", right_on="referencedComponentId", how="inner", validate="one_to_many"
    )
    terms = terms.loc[terms["typeId"].isin([SNOMED_FSN, SNOMED_SYNONYM])].copy()
    if terms.empty or not terms["typeId"].eq(SNOMED_SYNONYM).any():
        raise ValueError("Selected dialect has no active accepted synonyms in the selected language.")
    concepts, tag_lookup, description_issues = snomed_resolve_descriptions(concepts, descriptions, terms, refsets, metadata_ids)
    terms = terms.rename(columns={"id": "description_id", "conceptId": "concept_id", "refsetId": "refset_id",
                                  "effectiveTime": "description_effective_time", "moduleId": "description_module_id"})
    terms["term_type"] = terms["typeId"].map({SNOMED_FSN: "fsn", SNOMED_SYNONYM: "synonym"})
    terms["acceptability"] = terms["acceptabilityId"].map({SNOMED_PREFERRED: "preferred", SNOMED_ACCEPTABLE: "acceptable"})
    terms["semantic_tag"] = terms["concept_id"].map(tag_lookup).fillna("")
    terms["terminology"] = "SNOMED_CT"
    terms["release_date"] = release
    relationships = tables["relationships"]
    relationship_counts = {
        str(key): int(count)
        for key, count in (
            relationships["characteristicTypeId"].value_counts().sort_index().items()
        )
    }
    inferred_mask = (
        relationships["active"].eq("1")
        & relationships["characteristicTypeId"].eq(SNOMED_INFERRED)
    )
    inferred_count = int(inferred_mask.sum())
    if not (inferred_mask & relationships["typeId"].eq(SNOMED_IS_A)).any():
        raise ValueError(
            "No active inferred is-a relationships were found. "
            f"Observed characteristicTypeId counts: {relationship_counts}. "
            "A complete classified RF2 edition is required."
        )
    print("Active SNOMED relationship counts:", relationship_counts)
    dangling = (~relationships["sourceId"].isin(concept_ids) | ~relationships["destinationId"].isin(concept_ids) |
                ~relationships["typeId"].isin(concept_ids))
    if dangling.any():
        raise ValueError("Active relationships reference concepts absent from the active edition; supply a complete edition with dependencies.")
    for frame, order in [(concepts, ["id"]), (descriptions, ["id"]), (language, ["refsetId", "referencedComponentId"]),
                         (terms, ["concept_id", "refset_id", "description_id"]), (relationships, ["id"])]:
        frame.sort_values(order, kind="stable", inplace=True)
        frame.reset_index(drop=True, inplace=True)
    manifest = {
        "terminology": "SNOMED_CT", "release_date": release,
        "edition_label": str(config.get("edition", "unspecified: see source package and module IDs")),
        "source_name": source.name, "component_files": members,
        "metadata_ids": metadata_ids,
        "metadata_discovery_policy": metadata["policy"],
        "metadata_discovery_policy_version": metadata["policy_version"],
        "metadata_source_components": metadata["source_components"],
        "rf2_edition_tag": snomed_rf2_component_inventory([selected["concepts"]], config)[0]["edition_tag"],
        "language_code": config.get("language_code", "en"), "language_refset_ids": refsets,
        "available_language_refset_ids": available_refsets,
        "source_active_language_refset_ids": source_active_refsets,
        "language_refset_discovery_scope": "active_members_linked_to_active_selected_language_descriptions",
        "selected_language_refset_ids": refsets,
        "language_refset_selection": "all_active_ids_discovered_from_rf2" if requested_refsets is None else "explicit_subset",
        "language_refset_discovery_source": members["language_refset"],
        "language_refset_ids_identify_release": False,
        "accepted_term_counts_by_refset": {str(k): int(v) for k, v in terms.groupby("refset_id").size().items()},
        "preferred_term_display_policy": "first_available_in_sorted_refset_id_order" if requested_refsets is None else "first_available_in_explicit_refset_order",
        "dialect_preferred_terms_preserved": True,
        "description_resolution_policy": "unique_within_refset_then_declared_refset_order_v1",
        "description_resolution_policy_version": "snomed-description-resolution-v1",
        "description_candidate_detail_scope": "ambiguous_concepts_or_multiple_active_fsns; raw_descriptions_and_terms_remain_authoritative",
        "source_description_validation_status": "localized_ambiguities_retained" if len(description_issues) else "passed",
        "description_issue_count": len(description_issues),
        "ambiguous_concept_count": int((~concepts["mapping_eligible"]).sum()),
        "mapping_eligible_concept_count": int(concepts["mapping_eligible"].sum()),
        "description_issue_counts_by_reason": {str(k): int(v) for k, v in description_issues.groupby("reason_code").size().items()},
        "ambiguous_concepts_excluded_from_automatic_mapping": True,
        "description_issue_source_components": {kind: members[kind] for kind in ("concepts", "descriptions", "language_refset")},
        "concept_module_ids": sorted(concepts["moduleId"].unique().tolist()),
        "active_concepts": len(concepts), "active_descriptions": len(descriptions),
        "accepted_terms": len(terms),
        "active_relationships": len(relationships),
        "active_inferred_relationships": inferred_count,
        "active_non_inferred_relationships": len(relationships) - inferred_count,
        "relationship_counts_by_characteristic_type": relationship_counts,
        "relationship_reasoning_policy": "active_inferred_only",
        "non_inferred_relationships_preserved": True,
        "source_terms_are_not_dataset_mapping_approval": True,
        "mrcm_validation_performed": False, "owl_classification_performed": False,
    }
    manifest["source_fingerprint"] = hashlib.sha256(json.dumps(
        {"files": {k: v["sha256"] for k, v in members.items()}, "release_date": release,
         "language_code": config.get("language_code", "en"), "language_refset_ids": refsets}, sort_keys=True).encode()).hexdigest()
    return {"concepts": concepts, "descriptions": descriptions, "language_refset": language,
            "terms": terms, "relationships": relationships, "description_issues": description_issues,
            "metadata_audit": metadata["audit"], "manifest": manifest}


def snomed_mapping_run(snapshot):
    """Start one synchronous mapping run over an unchanged validated snapshot.

    Share RF2 tables without copying them. Only small derived results are cached;
    callers must not mutate source tables during the run. Start a fresh run for
    each execution of section 8 and clear its cache in a finally block.
    """
    if snapshot is None:
        return None
    scoped = dict(snapshot)
    scoped["_mapping_run_cache"] = {}
    scoped["_mapping_run_stats"] = {}
    return scoped


def snomed_mapping_cache_key(snapshot, operation, *parts):
    if "_mapping_run_cache" not in snapshot:
        return None
    encoded = []
    for part in parts:
        if isinstance(part, pd.DataFrame):
            encoded.append((tuple(map(str, part.dtypes)),
                            part.to_json(orient="split", force_ascii=False)))
        else:
            encoded.append(json.dumps(part, sort_keys=True, ensure_ascii=False))
    # Include source/language metadata and detect replacement of a source table.
    source = json.dumps(snapshot["manifest"], sort_keys=True, ensure_ascii=False)
    tables = tuple(id(snapshot.get(name)) for name in
                   ("concepts", "descriptions", "terms", "relationships"))
    return (operation, source, tables, *encoded)


def snomed_mapping_result_copy(value):
    if isinstance(value, pd.DataFrame):
        return value.copy(deep=True)
    return {name: frame.copy(deep=True) for name, frame in value.items()}


def snomed_mapping_cache_get(snapshot, key):
    if key is None:
        return False, None
    cache = snapshot["_mapping_run_cache"]
    counts = snapshot["_mapping_run_stats"].setdefault(key[0], {"hits": 0, "misses": 0})
    if key in cache:
        counts["hits"] += 1
        return True, snomed_mapping_result_copy(cache[key])
    counts["misses"] += 1
    return False, None


def snomed_mapping_cache_put(snapshot, key, result):
    if key is not None:
        snapshot["_mapping_run_cache"][key] = snomed_mapping_result_copy(result)
    return result


def snomed_candidate_query_seeds(labels, aliases=None):
    """Preserve every supplied query and its origin; these are search seeds only."""
    if not {"class_id", "class_label"}.issubset(labels.columns):
        raise ValueError("labels requires class_id and class_label.")
    if labels["class_id"].duplicated().any():
        raise ValueError("class_id must be unique.")
    seeds = labels[["class_id", "class_label"]].rename(columns={"class_label": "query_term"}).copy()
    seeds["query_source"] = "phase2_normalized_label"
    seeds["query_evidence_ids_json"] = "[]"
    seeds["query_rule_id"] = ""
    if aliases is not None and not aliases.empty:
        if not {"class_id", "term"}.issubset(aliases.columns):
            raise ValueError("aliases requires class_id and term.")
        if not set(aliases["class_id"]).issubset(set(labels["class_id"])):
            raise ValueError("Alias refers to an unknown class.")
        extras = aliases.copy().rename(columns={"term": "query_term"})
        for key, default in (("query_source", "supplied_alias"),
                             ("query_evidence_ids_json", "[]"), ("query_rule_id", "")):
            if key not in extras:
                extras[key] = default
        seeds = pd.concat([seeds, extras[list(seeds.columns)]], ignore_index=True)
    if seeds["query_term"].isna().any() or seeds["query_term"].astype(str).str.strip().eq("").any():
        raise ValueError("Candidate query terms must be nonempty strings.")
    seeds = seeds.drop_duplicates().copy()
    seeds["lexical_key"] = seeds["query_term"].map(snomed_lexical_key)
    return seeds.sort_values(["class_id", "lexical_key", "query_source", "query_rule_id"]).reset_index(drop=True)


def snomed_description_policy(policy=None):
    """Validate optional declarative description settings; defaults stay conservative."""
    if policy is None:
        policy = {}
    if not isinstance(policy, dict) or set(policy) - {"candidate_scope", "anchor_types"}:
        raise ValueError("Unsupported SNOMED description policy fields.")
    scope = policy.get("candidate_scope", "accepted_synonyms")
    anchors = policy.get("anchor_types", ["synonym"])
    if not isinstance(scope, str) or scope not in {"accepted_synonyms", "active_source_descriptions"}:
        raise ValueError("Unsupported SNOMED candidate description scope.")
    if (not isinstance(anchors, list) or not anchors
            or any(not isinstance(value, str) or value not in {"synonym", "fsn"} for value in anchors)
            or len(set(anchors)) != len(anchors)):
        raise ValueError("SNOMED anchor_types must be a nonempty unique list of synonym/fsn.")
    return {"candidate_scope": scope, "anchor_types": list(anchors)}


def snomed_description_scope_class_ids(labels, description_policies=None):
    """Select classes by their source policy, never by a medical class name."""
    policies = {} if description_policies is None else description_policies
    if not isinstance(policies, dict) or set(policies) - set(labels["class_id"]):
        raise ValueError("Description policies require known class IDs.")
    return {class_id for class_id in labels["class_id"]
            if snomed_description_policy(policies.get(class_id))["candidate_scope"]
            == "active_source_descriptions"}


def snomed_policy_description_candidates(labels, snapshot, seeds, description_policies=None):
    """Retrieve active descriptions only for policy-enabled classes.

    FSN retrieval separates only the final semantic tag; every clinical
    qualifier and the complete source term remain available for validation.
    Unaccepted descriptions remain unaccepted candidates, never approvals.
    """
    class_ids = snomed_description_scope_class_ids(labels, description_policies)
    scoped_seeds = seeds.loc[seeds["class_id"].isin(class_ids)]
    raw = snapshot.get("descriptions", pd.DataFrame())
    if scoped_seeds.empty or raw.empty:
        return pd.DataFrame()
    manifest = snapshot["manifest"]
    active_ids = set(snapshot["concepts"].loc[
        snapshot["concepts"]["active"].astype(str).eq("1"), "id"].astype(str))
    eligible = (
        raw["active"].astype(str).eq("1")
        & raw["conceptId"].astype(str).isin(active_ids)
        & raw["languageCode"].eq(manifest.get("language_code", "en"))
    )
    # Normalize only the two columns needed for matching. Materialize all RF2
    # fields only for matches; use row positions so duplicate index labels do
    # not duplicate or discard source rows.
    positions = eligible.to_numpy().nonzero()[0]
    lexical = raw.loc[eligible, ["term", "typeId"]].reset_index(drop=True)
    keys = set(scoped_seeds["lexical_key"])
    direct_keys = lexical["term"].map(snomed_lexical_key)
    direct_mask = direct_keys.isin(keys)
    columns = raw.columns.get_indexer(SNOMED_RF2_COLUMNS["descriptions"])
    if (columns < 0).any():
        missing = [name for name in SNOMED_RF2_COLUMNS["descriptions"] if name not in raw]
        raise KeyError(missing)
    direct = raw.iloc[positions[direct_mask.to_numpy()], columns].copy().reset_index(drop=True)
    direct["lexical_key"] = direct_keys.loc[direct_mask].reset_index(drop=True)
    fsn_id = manifest.get("metadata_ids", {}).get("SNOMED_FSN")
    fsn_mask = lexical["typeId"].eq(fsn_id) if fsn_id else pd.Series(False, index=lexical.index)
    fsn_keys = lexical.loc[fsn_mask, "term"].str.replace(
        r"\s*\([^()]*\)$", "", regex=True).map(snomed_lexical_key)
    matched_fsn = fsn_keys.isin(keys)
    fsn_positions = positions[fsn_mask.to_numpy()][matched_fsn.to_numpy()]
    fsns = raw.iloc[fsn_positions, columns].copy().reset_index(drop=True)
    fsns["lexical_key"] = fsn_keys.loc[matched_fsn].reset_index(drop=True)
    raw = pd.concat([direct, fsns], ignore_index=True).drop_duplicates()
    if raw.empty:
        return pd.DataFrame()
    raw = raw.rename(columns={
        "id": "description_id", "conceptId": "concept_id",
        "effectiveTime": "description_effective_time", "moduleId": "description_module_id",
    })
    # Join the unchanged source-accepted terms by the complete description
    # identity. Missing memberships are not promoted to preferred/acceptable.
    description_fields = [field for field in raw.columns if field != "lexical_key"]
    source_terms = snapshot["terms"]
    # This prefilter bounds the copied membership rows. It is not the join:
    # the complete source-description identity is still required below.
    accepted_mask = (
        source_terms["description_id"].isin(raw["description_id"])
        & source_terms["active"].astype(str).eq("1")
        & source_terms["language_active"].astype(str).eq("1")
        & source_terms["acceptability"].isin(["preferred", "acceptable"])
        & source_terms["term_type"].isin(["fsn", "synonym"])
    )
    refsets = manifest.get("selected_language_refset_ids", manifest.get("language_refset_ids", []))
    if refsets:
        accepted_mask &= source_terms["refset_id"].astype(str).isin(set(map(str, refsets)))
    accepted = source_terms.loc[accepted_mask].copy()
    matches = scoped_seeds.merge(raw, on="lexical_key", how="inner")
    matches = matches.merge(accepted, on=description_fields, how="left").fillna("")
    type_names = {value: name for key, name in (("SNOMED_FSN", "fsn"), ("SNOMED_SYNONYM", "synonym"))
                  if (value := manifest.get("metadata_ids", {}).get(key))}
    matches["term_type"] = matches["typeId"].map(type_names).fillna("")
    tags = snapshot["concepts"].set_index("id")["semantic_tag"].to_dict()
    matches["semantic_tag"] = matches["concept_id"].map(tags).fillna("")
    matches["terminology"] = "SNOMED_CT"
    matches["release_date"] = manifest["release_date"]
    return matches


def propose_snomed_candidates(labels, snapshot, aliases=None, description_policies=None):
    """Exact-normalized lexical candidates, never automatically approved.

    labels: DataFrame with class_id, class_label.
    aliases: optional class_id, term plus source/evidence metadata. They widen
    retrieval only; an alias never establishes equivalence or approves a map.
    Ambiguous matches remain separate candidates. Additional active-source
    description searches require explicit per-class settings from the verified
    policy. Retrieval never establishes source acceptance or class equivalence.
    """
    seeds = snomed_candidate_query_seeds(labels, aliases)
    cache_key = snomed_mapping_cache_key(snapshot, "candidates", seeds, description_policies)
    found, cached = snomed_mapping_cache_get(snapshot, cache_key)
    if found:
        return cached
    source_terms = snapshot["terms"]
    eligible = source_terms["term_type"].eq("synonym")
    # The importer already applies these gates; keep them explicit at retrieval
    # as well so a reconstructed caller table cannot introduce inactive terms.
    for field in ("active", "language_active"):
        if field in source_terms:
            eligible &= source_terms[field].astype(str).eq("1")
    if "acceptability" in source_terms:
        eligible &= source_terms["acceptability"].isin(["preferred", "acceptable"])
    if "concepts" in snapshot and "active" in snapshot["concepts"]:
        active_ids = set(snapshot["concepts"].loc[snapshot["concepts"]["active"].astype(str).eq("1"), "id"].astype(str))
        eligible &= source_terms["concept_id"].astype(str).isin(active_ids)
    # The inner join can only use query-matching rows. Normalize one Series
    # first, then copy every source field only for those rows.
    positions = eligible.to_numpy().nonzero()[0]
    lexical_keys = source_terms.loc[eligible, "term"].map(snomed_lexical_key)
    matched = lexical_keys.isin(set(seeds["lexical_key"]))
    accepted = source_terms.iloc[positions[matched.to_numpy()]].copy().reset_index(drop=True)
    accepted["lexical_key"] = lexical_keys.loc[matched].reset_index(drop=True)
    candidates = seeds.merge(accepted, on="lexical_key", how="inner")
    source_candidates = snomed_policy_description_candidates(
        labels, snapshot, seeds, description_policies=description_policies)
    if not source_candidates.empty:
        candidates = pd.concat([
            candidates, source_candidates.reindex(columns=candidates.columns, fill_value="")
        ], ignore_index=True).drop_duplicates()
    candidates["match_method"] = "exact_normalized_lexical"
    candidates["mapping_status"] = "candidate_review_required"
    candidates["mapping_relation"] = "unassessed"
    result = candidates.sort_values(["class_id", "concept_id", "description_id", "refset_id"]).reset_index(drop=True)
    return snomed_mapping_cache_put(snapshot, cache_key, result)


def snomed_candidate_retrieval_audit(labels, snapshot, aliases=None, candidates=None, description_policies=None):
    """Explain searches without treating a retrieval miss as ontology absence."""
    seeds = snomed_candidate_query_seeds(labels, aliases)
    if candidates is None:
        candidates = propose_snomed_candidates(labels, snapshot, aliases, description_policies=description_policies)
    cache_key = snomed_mapping_cache_key(snapshot, "retrieval_audit", seeds, description_policies, candidates)
    found, cached = snomed_mapping_cache_get(snapshot, cache_key)
    if found:
        return cached
    raw = snapshot.get("descriptions", pd.DataFrame())
    if not raw.empty and "term" in raw:
        eligible = pd.Series(True, index=raw.index)
        if "active" in raw:
            eligible &= raw["active"].astype(str).eq("1")
        synonym_id = snapshot.get("manifest", {}).get("metadata_ids", {}).get("SNOMED_SYNONYM")
        if synonym_id and "typeId" in raw:
            eligible &= raw["typeId"].astype(str).eq(str(synonym_id))
        if "concepts" in snapshot and "active" in snapshot["concepts"] and "conceptId" in raw:
            active_ids = set(snapshot["concepts"].loc[snapshot["concepts"]["active"].astype(str).eq("1"), "id"].astype(str))
            eligible &= raw["conceptId"].astype(str).isin(active_ids)
        # Counts and IDs need no copy of the full RF2 table. Preserve duplicate
        # rows because the audit's row count and distinct-ID list differ.
        audit_columns = [name for name in ("id", "term") if name in raw]
        raw = raw.loc[eligible, audit_columns].copy()
        raw["lexical_key"] = raw["term"].map(snomed_lexical_key)
        raw = raw.loc[raw["lexical_key"].isin(set(seeds["lexical_key"])),
                      [name for name in ("id", "lexical_key") if name in raw]]
    raw_groups = {key: frame for key, frame in raw.groupby("lexical_key")} if "lexical_key" in raw else {}
    matches = {(cid, key): frame for (cid, key), frame in candidates.groupby(["class_id", "lexical_key"])} if not candidates.empty else {}
    expanded_class_ids = snomed_description_scope_class_ids(labels, description_policies)
    rows = []
    for seed in seeds.to_dict("records"):
        found = matches.get((seed["class_id"], seed["lexical_key"]), pd.DataFrame())
        descriptions = raw_groups.get(seed["lexical_key"], pd.DataFrame())
        concept_ids = sorted(set(found.get("concept_id", pd.Series(dtype=str)).astype(str)))
        accepted_found = found
        reason = ("matched_active_accepted_synonym" if concept_ids else
                  "active_description_not_in_selected_language_refsets" if len(descriptions) else
                  "no_lexical_match_in_imported_snapshot")
        if seed["class_id"] in expanded_class_ids and not found.empty:
            descriptions = found[["description_id"]].drop_duplicates().rename(columns={"description_id": "id"})
            accepted_found = found.loc[
                found["active"].astype(str).eq("1") & found["language_active"].astype(str).eq("1")
                & found["acceptability"].isin(["preferred", "acceptable"])
                & found["term_type"].isin(["fsn", "synonym"]) & found["refset_id"].ne("")
            ]
            reason = ("matched_active_accepted_description" if not accepted_found.empty else
                      "active_description_not_in_selected_language_refsets")
        rows.append({**seed, "terminology": "SNOMED_CT", "candidate_concept_count": len(concept_ids),
                     "candidate_concept_ids_json": json.dumps(concept_ids, separators=(",", ":")),
                     "accepted_description_ids_json": json.dumps(sorted(set(accepted_found.get("description_id", pd.Series(dtype=str)).astype(str))), separators=(",", ":")),
                     "active_raw_description_count": len(descriptions),
                     "active_raw_description_ids_json": json.dumps(sorted(set(descriptions.get("id", pd.Series(dtype=str)).astype(str))), separators=(",", ":")),
                     "selected_refset_ids_json": json.dumps(snapshot.get("manifest", {}).get("selected_language_refset_ids", snapshot.get("manifest", {}).get("language_refset_ids", [])), separators=(",", ":")),
                     "reason_code": reason, "source_sha256": snapshot.get("manifest", {}).get("source_fingerprint", ""),
                     "ontology_absence_proven": False})
    return snomed_mapping_cache_put(snapshot, cache_key, pd.DataFrame(rows))


def apply_reviewed_snomed_mappings(labels, snapshot, reviews):
    """Validate explicit mapping decisions; export synonyms for approved exact maps.

    reviews columns: class_id, concept_id, mapping_relation, decision,
    reviewer, reviewed_at, rationale, source_reference, release_date.
    Approval may be justified without lexical matching. Source provenance and
    review identity are mandatory; caller remains responsible for their truth.
    """
    required = ["class_id", "concept_id", "mapping_relation", "decision", "reviewer", "reviewed_at", "rationale", "source_reference", "release_date"]
    if not set(required).issubset(reviews.columns):
        raise ValueError(f"Mapping reviews require columns {required}")
    checked = reviews[required].copy().fillna("").astype(str)
    if checked.duplicated(["class_id", "concept_id"]).any():
        raise ValueError("Duplicate mapping review for the same class/concept.")
    if not set(checked["class_id"]).issubset(set(labels["class_id"].astype(str))):
        raise ValueError("Mapping review refers to an unknown dataset class.")
    if not checked["decision"].isin(["approved", "rejected", "review"]).all():
        raise ValueError("Mapping decision must be approved, rejected or review.")
    if not checked["mapping_relation"].isin(["exact", "broader", "narrower", "related", "unmapped"]).all():
        raise ValueError("Invalid mapping relation.")
    approved = checked.loc[checked["decision"].eq("approved")]
    for field in ["concept_id", "reviewer", "reviewed_at", "rationale", "source_reference"]:
        if approved[field].str.strip().eq("").any():
            raise ValueError(f"Approved mappings require a nonempty {field}.")
    if not approved["release_date"].eq(snapshot["manifest"]["release_date"]).all():
        raise ValueError("Approved mappings must cite this exact SNOMED release.")
    if not set(approved["concept_id"]).issubset(set(snapshot["concepts"]["id"])):
        raise ValueError("Approved mapping refers to an absent or inactive SNOMED concept.")
    if approved["mapping_relation"].eq("unmapped").any():
        raise ValueError("An unmapped relationship cannot approve a concept mapping.")
    if "mapping_eligible" in snapshot["concepts"]:
        eligible_ids = set(snapshot["concepts"].loc[snapshot["concepts"]["mapping_eligible"].eq(True), "id"])
        if not set(approved["concept_id"]).issubset(eligible_ids):
            raise ValueError("Approved mapping refers to a concept with unresolved source-description ambiguity.")
    exact = approved.loc[approved["mapping_relation"].eq("exact")]
    if exact["class_id"].duplicated().any():
        raise ValueError("Multiple exact concept mappings for one class require explicit compositional modelling, not synonym union.")
    synonyms = exact.merge(snapshot["terms"].loc[snapshot["terms"]["term_type"].eq("synonym")], on="concept_id", suffixes=("_mapping", "_term"))
    synonyms["approval_basis"] = "reviewed_exact_mapping_and_active_language_acceptability"
    return {"reviewed_mappings": checked, "approved_synonyms": synonyms.sort_values(["class_id", "term", "refset_id"]).reset_index(drop=True)}


def snomed_ancestor_subset(snapshot, seed_concept_ids):
    """Select seed concepts, their ancestors and their imported defining attributes.

    is_a_closure contains ancestor pairs only for the requested seeds, computed
    over the complete imported inferred graph before pruning. Non-is-a edges are
    retained only when their source is a seed; their destination and relationship
    type concepts are also retained, including their source-accepted terms.

    These relationships describe SNOMED concept definitions. They neither add
    patient/frame observations nor validate postcoordinated expressions. Original
    relationshipGroup, characteristicTypeId and modifierId are never flattened.
    All returned objects are DataFrames, including relationship_scope metadata.
    """
    metadata_ids = snapshot["manifest"]["metadata_ids"]
    SNOMED_INFERRED = metadata_ids["SNOMED_INFERRED"]
    SNOMED_IS_A = metadata_ids["SNOMED_IS_A"]
    seeds = sorted(set(map(str, seed_concept_ids)))
    cache_key = snomed_mapping_cache_key(snapshot, "ancestor_subset", seeds)
    found, cached = snomed_mapping_cache_get(snapshot, cache_key)
    if found:
        return cached
    known = set(snapshot["concepts"]["id"])
    if not set(seeds).issubset(known):
        raise ValueError("Subset seed is absent or inactive in the imported edition.")
    imported = snapshot["relationships"]
    active_inferred = imported.loc[
        imported["active"].eq("1") & imported["characteristicTypeId"].eq(SNOMED_INFERRED)
    ]
    edges = active_inferred.loc[active_inferred["typeId"].eq(SNOMED_IS_A)]
    parents = edges.groupby("sourceId")["destinationId"].agg(list).to_dict()
    rows, retained = [], set(seeds)
    for seed in seeds:
        seen, pending = set(), list(parents.get(seed, []))
        while pending:
            ancestor = pending.pop()
            if ancestor == seed:
                raise ValueError("Cycle detected in inferred is-a hierarchy.")
            if ancestor not in seen:
                seen.add(ancestor)
                pending.extend(parents.get(ancestor, []))
        retained.update(seen)
        rows.extend({"concept_id": seed, "ancestor_id": ancestor} for ancestor in sorted(seen))
    seed_attributes = active_inferred.loc[
        active_inferred["sourceId"].isin(seeds) & ~active_inferred["typeId"].eq(SNOMED_IS_A)
    ].copy()
    retained.update(seed_attributes["destinationId"])
    retained.update(seed_attributes["typeId"])
    if not retained.issubset(known):
        raise ValueError("Seed attribute references concepts absent from the active edition.")
    retained_is_a = edges.loc[
        edges["sourceId"].isin(retained) & edges["destinationId"].isin(retained)
    ]
    relationships = pd.concat([retained_is_a, seed_attributes], ignore_index=True)
    relationships = relationships.drop_duplicates("id").sort_values("id", kind="stable").reset_index(drop=True)
    seed_attributes = seed_attributes.sort_values("id", kind="stable").reset_index(drop=True)
    scope = pd.DataFrame([
        {"table": "is_a_closure", "relationship_scope": "transitive_is_a_for_requested_seeds_only",
         "semantic_role": "terminology_hierarchy", "patient_observation": False,
         "mrcm_validation_performed": False},
        {"table": "seed_attribute_relationships", "relationship_scope": "outgoing_non_is_a_inferred_edges_for_requested_seeds_only",
         "semantic_role": "terminology_concept_definition", "patient_observation": False,
         "mrcm_validation_performed": False},
        {"table": "relationships", "relationship_scope": "seed_attribute_edges_and_is_a_edges_between_retained_nodes",
         "semantic_role": "terminology_graph_subset", "patient_observation": False,
         "mrcm_validation_performed": False},
    ])
    scope["seed_concept_ids_json"] = json.dumps(seeds, separators=(",", ":"))
    scope["release_date"] = snapshot["manifest"]["release_date"]
    scope["source_fingerprint"] = snapshot["manifest"]["source_fingerprint"]
    result = {"concepts": snapshot["concepts"].loc[snapshot["concepts"]["id"].isin(retained)].copy(),
            "terms": snapshot["terms"].loc[snapshot["terms"]["concept_id"].isin(retained)].copy(),
            "is_a_closure": pd.DataFrame(rows, columns=["concept_id", "ancestor_id"]),
            "relationships": relationships,
            "seed_attribute_relationships": seed_attributes,
            "relationship_scope": scope}
    return snomed_mapping_cache_put(snapshot, cache_key, result)


## 7. Resolve source inputs and import terminology snapshots

For each source: check Drive; if absent, download to a unique `.partial` file on Drive, validate the container and calculate SHA-256 after completion; only then publish the final file. HTML login pages, interrupted transfers and invalid containers do not become accepted inputs. Persist the reference, then copy to the VM and compare SHA-256 against it. Existing Drive sources skip external downloads. A corrupt local cache is recopied and verified. A transfer error or hash mismatch triggers a fresh copy, up to `vm_copy_attempts=3` total attempts. Every attempt uses the same fixed reference; exhausting attempts stops import. A changed source in a later run receives a separate version record.

Read `ontology_manifest.json` on every run. Reuse stored SNOMED release/component identifiers only for matching file bytes and language; derive language refsets from the actual RF2 content and persist the results. The default public download is MST; CEST/SNOMED can download automatically when authorized source URLs/access are configured. CEST remains a reviewed transcription plus source document, not an arbitrary PDF-to-ontology conversion. The input packages are cached once per version. No public SNOMED demonstration API is used for bulk extraction.


In [7]:
"""Hash source bytes on mounted Drive, then verify a separate local snapshot.

No network download and no built-in expected source hash. The observed source
hash identifies this run's input, not publisher authenticity or mapping approval.
"""


def terminology_stream_digest(path):
    digest, size = hashlib.sha256(), 0
    with Path(path).open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(chunk)
            size += len(chunk)
    return size, digest.hexdigest()


def terminology_source_inventory(path):
    """File SHA or canonical full directory manifest; reject escaping symlinks."""
    path = Path(path).expanduser()
    if path.is_symlink():
        raise ValueError("Terminology source must not be a symbolic link")
    path = path.resolve()
    if path.is_file():
        size, digest = terminology_stream_digest(path)
        return {"kind": "file", "identity_algorithm": "sha256", "sha256": digest,
                "size_bytes": size, "files": [{"relative_path": ".", "size_bytes": size, "sha256": digest}]}
    if not path.is_dir():
        raise FileNotFoundError(f"Terminology source does not exist: {path}")
    files = []
    for child in sorted(path.rglob("*")):
        if child.is_symlink():
            raise ValueError(f"Symbolic link inside terminology directory: {child}")
        if child.is_dir():
            continue
        if not child.is_file():
            raise ValueError(f"Unsupported terminology source entry: {child}")
        size, digest = terminology_stream_digest(child)
        files.append({"relative_path": child.relative_to(path).as_posix(), "size_bytes": size, "sha256": digest})
    if not files:
        raise ValueError(f"Empty terminology source directory: {path}")
    payload = json.dumps(files, sort_keys=True, separators=(",", ":"), ensure_ascii=False).encode()
    return {"kind": "directory", "identity_algorithm": "sha256_tree_v1",
            "sha256": hashlib.sha256(b"mmvqa-directory-sha256-v1\0" + payload).hexdigest(),
            "size_bytes": sum(row["size_bytes"] for row in files), "files": files}


def terminology_verify_staged(path, reference):
    actual = terminology_source_inventory(path)
    if actual != reference:
        raise ValueError(f"Terminology VM copy differs from its recorded source SHA-256: {path}")
    return actual


def terminology_stage_source(source_path, local_root, registry_root, source_key, require_drive=False, copy_attempts=3):
    """Record immutable source reference on Drive; copy/reuse only verified bytes.

    The initial source inventory is fixed before the copy. Transfer/hash failures
    retry against that same reference up to copy_attempts, never against changed
    source bytes. A new source version gets its own hash-addressed registry/cache
    location. Corrupt cached copies are deterministically recopied. Pending bytes
    are verified before replacing a cache entry and cleaned after each failure.
    """
    if isinstance(copy_attempts, bool) or not isinstance(copy_attempts, int) or copy_attempts < 1:
        raise ValueError("copy_attempts must be a positive integer")
    if not re.fullmatch(r"[A-Za-z0-9_-]+", source_key):
        raise ValueError("Invalid terminology source key")
    original = Path(source_path).expanduser()
    if original.is_symlink():
        raise ValueError("Terminology source must not be a symbolic link")
    original = original.resolve()
    registry_root, local_root = Path(registry_root).resolve(), Path(local_root).resolve()
    if require_drive:
        drive_root = Path("/content/drive").resolve()
        if not original.is_relative_to(drive_root) or not registry_root.is_relative_to(drive_root):
            raise ValueError("In Colab, ontology sources and their reference registry must be in mounted Google Drive")
        if local_root.is_relative_to(drive_root):
            raise ValueError("The verified working copy must be on the VM, outside Google Drive")
    if local_root == original or local_root.is_relative_to(original):
        raise ValueError("The local cache cannot be inside the source package")
    if original.is_dir() and (registry_root == original or registry_root.is_relative_to(original)):
        raise ValueError("The reference registry cannot be inside the source package")
    reference = terminology_source_inventory(original)
    # The registry is the persisted baseline for this exact input, before parsing.
    registry_path = registry_root / source_key / (reference["sha256"] + ".json")
    record = {"schema_version": "mmvqa-source-reference-v1", "source_key": source_key,
              "integrity_scope": "exact_input_bytes_not_publisher_authenticity", "reference": reference}
    registry_path.parent.mkdir(parents=True, exist_ok=True)
    if registry_path.exists():
        if json.loads(registry_path.read_text(encoding="utf-8")) != record:
            raise ValueError(f"Existing source registry record is inconsistent: {registry_path}")
    else:
        # Exclusive creation avoids silently replacing a different reference.
        try:
            with registry_path.open("x", encoding="utf-8") as handle:
                json.dump(record, handle, ensure_ascii=False, sort_keys=True, indent=2)
        except FileExistsError:
            if json.loads(registry_path.read_text(encoding="utf-8")) != record:
                raise ValueError(f"Conflicting source registry record: {registry_path}")
    if json.loads(registry_path.read_text(encoding="utf-8")) != record:
        raise ValueError("Source reference did not persist correctly")
    # Include basename so identical bytes uploaded with another name remain valid.
    cache_parent = local_root / source_key / reference["sha256"]
    destination = cache_parent / original.name
    cache_parent.mkdir(parents=True, exist_ok=True)
    cache_valid = False
    copy_attempts_used = 0
    if destination.exists():
        try:
            terminology_verify_staged(destination, reference)
            cache_valid = True
        except (ValueError, OSError):
            pass
    if not cache_valid:
        for attempt in range(1, copy_attempts + 1):
            copy_attempts_used = attempt
            try:
                with tempfile.TemporaryDirectory(prefix=".staging-", dir=cache_parent) as temporary:
                    pending = Path(temporary) / original.name
                    if reference["kind"] == "file":
                        shutil.copyfile(original, pending)
                    else:
                        shutil.copytree(original, pending, symlinks=True)
                    terminology_verify_staged(pending, reference)
                    if destination.is_dir() and not destination.is_symlink():
                        shutil.rmtree(destination)
                    elif destination.exists() or destination.is_symlink():
                        destination.unlink()
                    pending.replace(destination)
                terminology_verify_staged(destination, reference)
                break
            except (ValueError, OSError):
                if attempt == copy_attempts:
                    raise RuntimeError(
                        f"Terminology VM copy failed verification or transfer after {copy_attempts} attempts; "
                        "the recorded source SHA-256 remains unchanged."
                    ) from None
    return {"source_key": source_key, "source_path": str(original), "local_path": str(destination),
            "reference_manifest_path": str(registry_path), "reference": reference,
            "source_sha256": reference["sha256"], "vm_sha256": reference["sha256"],
            "identity_algorithm": reference["identity_algorithm"], "verification_status": "verified",
            "copy_attempts_used": copy_attempts_used, "copy_retries": max(0, copy_attempts_used - 1),
            "cache_status": "reused_verified" if cache_valid else "copied_and_verified"}

"""Acquire missing terminology inputs once, directly into mounted storage.

Existing files/directories never cause network or secret lookups. Downloaded
containers are validated before publication; full terminology parsers remain
responsible for clinical/schema validation. Authentication is transient.
"""


def _acquisition_safe_url(url, origin_only=False):
    """Remove credential-bearing URL fields before any metadata persistence."""
    if not url:
        return None
    try:
        parsed = urlsplit(str(url))
        hostname = parsed.hostname or ""
        if ":" in hostname:
            hostname = "[" + hostname + "]"
        netloc = hostname + ((":" + str(parsed.port)) if parsed.port else "")
        return urlunsplit((parsed.scheme, netloc, "" if origin_only else parsed.path, "", ""))
    except (ValueError, TypeError):
        return "<invalid-url>"


def sanitize_acquisition_spec(spec):
    """Export configuration without URL credentials, query tokens or headers.

    Secret *names* are references, not secret values. Only the URL origin is
    retained because path segments can also contain authentication tokens.
    """
    spec = dict(spec or {})
    safe = {}
    allowed = {"kind", "url_secret_name", "header_secret_names", "timeout_seconds", "max_bytes"}
    for key in allowed:
        if key in spec:
            safe[key] = spec[key]
    if "url" in spec:
        safe["url_origin"] = _acquisition_safe_url(spec["url"], origin_only=True)
    if "headers" in spec:
        safe["headers"] = {str(name): "<redacted>" for name in dict(spec["headers"] or {})}
    return safe


def _acquisition_external_ids(url):
    """Retain only official MLDS release identifiers, never opaque URL tokens."""
    parsed = urlsplit(str(url))
    if parsed.scheme != "https" or parsed.hostname != "mlds.ihtsdotools.org" or parsed.port not in {None, 443}:
        return {}
    match = re.fullmatch(r"/api/releasePackages/(\d+)/releaseVersions/(\d+)/releaseFiles/(\d+)/download/?", parsed.path)
    return dict(zip(("package_id", "release_version_id", "release_file_id"), match.groups())) if match else {}


class _TerminologyHTTPSRedirectHandler(HTTPRedirectHandler):
    def redirect_request(self, req, fp, code, msg, headers, newurl):
        target = urlsplit(newurl)
        if target.scheme.lower() != "https":
            raise RuntimeError("Terminology download refused a non-HTTPS redirect")
        redirected = super().redirect_request(req, fp, code, msg, headers, newurl)
        if redirected is None:
            return None
        source = urlsplit(req.full_url)
        source_origin = (source.scheme.lower(), source.hostname, source.port or 443)
        target_origin = (target.scheme.lower(), target.hostname, target.port or 443)
        if source_origin != target_origin:
            # Never forward caller authentication to another origin. The new
            # URL itself may be a provider-issued signed HTTPS download URL.
            for name, _ in list(redirected.header_items()):
                redirected.remove_header(name)
            redirected.add_header("User-Agent", "MMVQA-Terminology-Importer/1.0")
            redirected.add_header("Accept-Encoding", "identity")
        return redirected


def _acquisition_digest(path):
    digest, size = hashlib.sha256(), 0
    with Path(path).open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            size += len(block)
            digest.update(block)
    return size, digest.hexdigest()


def _acquisition_validate_container(path, kind):
    with Path(path).open("rb") as handle:
        prefix = handle.read(4096)
    if not prefix:
        raise ValueError("Terminology download is empty")
    lower = prefix.lstrip(b"\xef\xbb\xbf \t\r\n").lower()
    if lower.startswith((b"<!doctype html", b"<html", b"<head", b"<body")):
        raise ValueError("Terminology download is an HTML page, possibly a sign-in page")
    if kind == "xls":
        if not prefix.startswith(bytes.fromhex("D0CF11E0A1B11AE1")):
            raise ValueError("Terminology download is not an XLS workbook")
    elif kind == "zip":
        if not zipfile.is_zipfile(path):
            raise ValueError("Terminology download is not a complete ZIP container")
        with zipfile.ZipFile(path) as archive:
            files = [entry.filename for entry in archive.infolist() if not entry.is_dir()]
            if not files or len(files) != len(set(files)):
                raise ValueError("Terminology ZIP is empty or has duplicate member names")
    elif kind == "json":
        try:
            with Path(path).open("r", encoding="utf-8-sig") as handle:
                payload = json.load(handle)
        except (UnicodeError, json.JSONDecodeError):
            raise ValueError("Terminology download is not valid JSON") from None
        if not isinstance(payload, dict):
            raise ValueError("Terminology JSON package must have an object root")
    elif kind == "document":
        # CEST original sources may be PDF or UTF-8 source text. No HTML login
        # page, binary executable or generic unknown binary is accepted.
        if not prefix.startswith(b"%PDF-"):
            try:
                prefix.decode("utf-8-sig")
            except UnicodeError:
                raise ValueError("Terminology document must be PDF or UTF-8 text") from None
            if b"\x00" in prefix or lower.startswith(b"<?xml"):
                raise ValueError("Terminology document is not an accepted PDF/text source")
    else:
        raise ValueError("Unsupported terminology download kind")


def acquire_terminology_source(destination, spec, *, secret_resolver=None, opener=None):
    """Return an existing input or acquire its missing bytes once.

    spec: kind=xls|zip|json|document; url or url_secret_name; optional headers,
    header_secret_names={header:secret_name}, timeout_seconds, max_bytes.
    secret_resolver(name) returns an ephemeral secret value. ``opener`` is an
    optional Request/timeout-compatible callable for controlled integration tests.

    Missing configuration returns source_configuration_required, never a fake
    successful import. Network/container failures raise sanitized errors. A
    same-directory lock serializes this notebook's writers. Publication uses a
    same-filesystem rename only after rechecking that the destination is absent.
    The SHA-256 is computed from completed storage bytes, after download.
    """
    destination = Path(destination).expanduser()
    if destination.is_symlink():
        raise ValueError("Terminology acquisition destination must not be a symbolic link")
    destination = destination.resolve()
    spec = dict(spec or {})
    kind = spec.get("kind", "document")
    if destination.exists():
        if not destination.is_file() and not (destination.is_dir() and kind == "zip"):
            raise ValueError("Existing terminology input has an unsupported filesystem type")
        return {"path": str(destination), "status": "available", "acquisition": "existing_storage",
                "downloaded": False, "source_origin": None,
                "size_bytes": destination.stat().st_size if destination.is_file() else None,
                "sha256": None, "external_ids": {}, "integrity_status": "pending_source_staging_verification"}
    if kind not in {"xls", "zip", "json", "document"}:
        raise ValueError("Unsupported terminology download kind")
    if spec.get("url") and spec.get("url_secret_name"):
        raise ValueError("Configure either url or url_secret_name, not both")
    missing = []

    def secret_value(name):
        if not secret_resolver:
            missing.append(str(name))
            return None
        try:
            value = secret_resolver(str(name))
        except Exception:
            value = None  # A secret provider's exception may contain credentials.
        if not value:
            missing.append(str(name))
        return value

    url = spec.get("url")
    if spec.get("url_secret_name"):
        url = secret_value(spec["url_secret_name"])
    headers = {str(k): str(v) for k, v in dict(spec.get("headers") or {}).items()}
    for header, name in dict(spec.get("header_secret_names") or {}).items():
        value = secret_value(name)
        if value:
            headers[str(header)] = str(value)
    if missing or not url:
        return {"path": None, "status": "source_configuration_required", "downloaded": False,
                "acquisition": "not_attempted", "source_origin": None,
                "reason": "A direct download URL and any referenced authentication secrets are required",
                "missing_secret_names": sorted(set(missing)), "size_bytes": None, "sha256": None, "external_ids": {}}
    try:
        parsed = urlsplit(str(url))
        if parsed.scheme.lower() != "https" or not parsed.hostname or parsed.username or parsed.password:
            raise ValueError()
        parsed.port
    except (ValueError, TypeError):
        raise ValueError("Configure a valid HTTPS download URL without embedded user credentials") from None
    timeout = int(spec.get("timeout_seconds", 60))
    limit = spec.get("max_bytes")
    limit = int(limit) if limit is not None else None
    if timeout < 1 or (limit is not None and limit < 1):
        raise ValueError("Download timeout and optional size limit must be positive")
    headers.setdefault("User-Agent", "MMVQA-Terminology-Importer/1.0")
    headers.setdefault("Accept-Encoding", "identity")
    try:
        request = Request(str(url), headers=headers, method="GET")
    except Exception:
        raise ValueError("Terminology download request configuration is invalid") from None
    open_request = opener or build_opener(_TerminologyHTTPSRedirectHandler()).open
    destination.parent.mkdir(parents=True, exist_ok=True)
    lock_path = destination.with_name("." + destination.name + ".acquire.lock")
    try:
        lock = lock_path.open("x")
    except FileExistsError:
        raise RuntimeError("Terminology acquisition is already active or was interrupted; inspect the acquisition lock before retrying") from None
    pending = None
    try:
        with lock:
            if destination.exists():
                raise FileExistsError("Terminology destination appeared before download; rerun to reuse it")
            with tempfile.NamedTemporaryFile(prefix="." + destination.name + ".", suffix=".partial",
                                             dir=destination.parent, delete=False) as stream:
                pending = Path(stream.name)
                try:
                    with open_request(request, timeout=timeout) as response:
                        status = getattr(response, "status", None)
                        if status is None and hasattr(response, "getcode"):
                            status = response.getcode()
                        if status != 200:
                            raise RuntimeError("http_status_" + str(int(status)))
                        content_type = str(response.headers.get("Content-Type", "")).lower()
                        if "text/html" in content_type or "application/xhtml" in content_type:
                            raise RuntimeError("html_response")
                        content_length = response.headers.get("Content-Length")
                        expected_size = int(content_length) if content_length is not None else None
                        if expected_size is not None and (expected_size < 1 or (limit and expected_size > limit)):
                            raise RuntimeError("invalid_content_length")
                        received = 0
                        while True:
                            block = response.read(1024 * 1024)
                            if not block:
                                break
                            received += len(block)
                            if limit and received > limit:
                                raise RuntimeError("download_size_limit")
                            stream.write(block)
                        if expected_size is not None and received != expected_size:
                            raise RuntimeError("truncated_or_inconsistent_content_length")
                except Exception as exc:
                    # Never propagate HTTPError URLs, response bodies or provider
                    # exception strings: they can include signed URLs/tokens.
                    reason = str(exc) if isinstance(exc, RuntimeError) and (str(exc) in {
                        "http_status", "html_response", "invalid_content_length", "download_size_limit",
                        "truncated_or_inconsistent_content_length"} or re.fullmatch(r"http_status_\d{3}", str(exc))) else type(exc).__name__
                    raise RuntimeError("Terminology download failed (" + reason + "); check source URL, access and network") from None
                stream.flush()
                os.fsync(stream.fileno())
            _acquisition_validate_container(pending, kind)
            size, digest = _acquisition_digest(pending)
            if destination.exists() or destination.is_symlink():
                raise FileExistsError("Terminology destination appeared during download; existing source was preserved")
            pending.rename(destination)
            pending = None
            return {"path": str(destination), "status": "available", "acquisition": "downloaded_to_storage",
                    "downloaded": True, "source_origin": _acquisition_safe_url(url, origin_only=True),
                    "size_bytes": size, "sha256": digest, "external_ids": _acquisition_external_ids(url),
                    "integrity_status": "download_complete_container_validated"}
    finally:
        if pending is not None:
            pending.unlink(missing_ok=True)
        lock_path.unlink(missing_ok=True)

"""Persistent lookup of ontology source identities and parser identifiers."""


def load_ontology_manifest(path):
    path = Path(path)
    if not path.exists():
        return {"schema_version": "mmvqa-ontology-storage-v1", "sources": {}}
    data = json.loads(path.read_text(encoding="utf-8"))
    if data.get("schema_version") != "mmvqa-ontology-storage-v1" or not isinstance(data.get("sources"), dict):
        raise ValueError("Unsupported ontology storage manifest")
    return data


def save_ontology_manifest(path, manifest):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    text = json.dumps(manifest, sort_keys=True, ensure_ascii=False, indent=2, allow_nan=False)
    fd, pending_name = tempfile.mkstemp(prefix=".ontology-manifest-", suffix=".partial", dir=path.parent)
    pending = Path(pending_name)
    try:
        with os.fdopen(fd, "w", encoding="utf-8") as stream:
            stream.write(text)
        if json.loads(pending.read_text(encoding="utf-8")) != manifest:
            raise ValueError("Ontology storage manifest write verification failed")
        pending.replace(path)
    finally:
        pending.unlink(missing_ok=True)


def record_ontology_source(manifest, key, acquisition, staged=None, read_ids=None):
    previous = manifest["sources"].get(key, {})
    versions = dict(previous.get("versions", {}))
    entry = {"acquisition": acquisition, "status": acquisition["status"], "versions": versions}
    if staged is not None:
        fingerprint = staged["source_sha256"]
        version = dict(versions.get(fingerprint, {}))
        if "acquisition" not in version or acquisition.get("source_origin"):
            version["acquisition"] = acquisition
        version.update({"source_sha256": fingerprint, "identity_algorithm": staged["identity_algorithm"],
                        "size_bytes": staged["reference"]["size_bytes"], "drive_path": staged["source_path"],
                        "reference_manifest_path": staged["reference_manifest_path"],
                        "verification_status": staged["verification_status"]})
        if read_ids is not None:
            version["read_ids"] = read_ids
        versions[fingerprint] = version
        entry.update({"active_sha256": fingerprint, "status": "imported" if read_ids is not None else "stored_and_verified",
                      "read_ids": read_ids if read_ids is not None else {}, "drive_path": staged["source_path"]})
    manifest["sources"][key] = entry
    return entry


def ontology_read_ids(manifest, key, artifact_sha256):
    """Only reuse parser IDs when the exact artifact identity still matches."""
    version = manifest.get("sources", {}).get(key, {}).get("versions", {}).get(artifact_sha256, {})
    return dict(version.get("read_ids", {}))


def snomed_read_ids(snapshot):
    source = snapshot["manifest"]
    return {
        "release_date": source["release_date"], "rf2_edition_tag": source["rf2_edition_tag"],
        "edition_label": source["edition_label"], "language_code": source["language_code"],
        "available_language_refset_ids": source["available_language_refset_ids"],
        "source_active_language_refset_ids": source.get("source_active_language_refset_ids", source["available_language_refset_ids"]),
        "selected_language_refset_ids": source["selected_language_refset_ids"],
        "concept_module_ids": source["concept_module_ids"],
        "component_paths": {key: row["path"] for key, row in source["component_files"].items()},
        "component_sha256": {key: row["sha256"] for key, row in source["component_files"].items()},
        "source_fingerprint": source["source_fingerprint"],
        "description_resolution_policy_version": source.get("description_resolution_policy_version"),
        "source_description_validation_status": source.get("source_description_validation_status", "not_evaluated"),
        "description_issue_count": source.get("description_issue_count"),
        "ambiguous_concept_count": source.get("ambiguous_concept_count"),
        "metadata_ids": dict(source.get("metadata_ids", {})),
        "metadata_discovery_policy_version": source.get("metadata_discovery_policy_version"),
        "metadata_source_components": source.get("metadata_source_components", {}),
    }

"""Terminology-only mapping, source gating and deterministic export helpers."""


def terminology_json(value):
    return json.dumps(value, ensure_ascii=False, sort_keys=True, separators=(",", ":"), allow_nan=False)


def terminology_key(value):
    # A search key only: the Phase 2 label itself is retained without modification.
    return " ".join(re.sub(r"[_\-]+", " ", mst_lookup_key(value)).split())


SNOMED_RF2_FILENAME_PATTERNS = ("SnomedCT*.zip", "SNOMED_CT*.zip", "SNOMED-CT*.zip")


def terminology_file_choice(explicit, root, patterns):
    """Select one immediate child, matching filename case independently of the OS.

    Discovery does not validate RF2 contents. Acquisition, SHA-256 staging and
    the RF2 importer still perform their respective checks after selection.
    """
    if explicit:
        path = Path(explicit).expanduser()
        if not path.exists():
            raise FileNotFoundError(f"Configured terminology source does not exist: {path}")
        return path
    root = Path(root)
    folded_patterns = tuple(pattern.casefold() for pattern in patterns)
    matches = sorted(
        (path for path in root.iterdir() if path.is_file()
         and any(fnmatch.fnmatchcase(path.name.casefold(), pattern) for pattern in folded_patterns)),
        key=lambda path: (path.name.casefold(), path.name),
    ) if root.exists() else []
    if len(matches) > 1:
        raise ValueError(f"Ambiguous terminology sources under {root}; set an explicit path: {[p.name for p in matches]}")
    return matches[0] if matches else None


def terminology_supplement(package):
    """Import explicit nomenclature facts, separately labelled from original CEST."""
    fingerprint = hashlib.sha256(terminology_json(package).encode()).hexdigest()
    rows, aliases = [], []
    for term in package["terms"]:
        rows.append({"terminology": package["terminology"], "version": package["version"],
                     "term_id": term["term_id"], "preferred_term": term["preferred_term"],
                     "term_key": mst_lookup_key(term["preferred_term"]), "term_type": term["term_type"],
                     "section": term["section"], "organ": "Small bowel", "source_url": package["source_url"],
                     "source_locator": term["source_locator"], "source_sha256": fingerprint})
    for synonym in package["synonyms"]:
        aliases.append({"term_id": synonym["term_id"], "synonym": synonym["synonym"],
                        "synonym_key": mst_lookup_key(synonym["synonym"]), "language": "en",
                        "status": "source_attested_equivalent_name", "source_locator": synonym["source_locator"],
                        "source_url": package["source_url"], "source_sha256": fingerprint})
    return {"terms": pd.DataFrame(rows, columns=MST_TERM_COLUMNS),
            "synonyms": pd.DataFrame(aliases, columns=MST_SYNONYM_COLUMNS),
            "attributes": pd.DataFrame(columns=MST_ATTRIBUTE_COLUMNS),
            "manifest": {"status": "imported_selected_nomenclature", "terminology": package["terminology"],
                         "version": package["version"], "source_url": package["source_url"],
                         "package_sha256": fingerprint, "scope": package["scope"],
                         "not_original_cest": True, "not_synonyms": package["not_synonyms"]}}


TERMINOLOGY_CROSSWALK_COLUMNS = [
    "mapping_id", "dataset_class_id", "normalized_label", "terminology", "target_id", "target_term",
    "target_type", "relation", "mapping_status", "match_method", "source_sha256", "source_url",
    "source_locator", "required_attribute_constraints_json", "rationale", "reviewer", "reviewed_at", "source_reference",
    "target_fsn", "matched_descriptions_json", "matched_query_terms_json",
]


def terminology_candidate(class_row, term, relation="unassessed", method="exact_lexical_candidate", constraints=None, rationale=""):
    values = {"dataset_class_id": class_row["dataset_class_id"], "normalized_label": class_row["normalized_label"],
              "terminology": term["terminology"], "target_id": term["term_id"], "target_term": term["preferred_term"],
              "target_type": term["term_type"], "relation": relation, "mapping_status": "candidate_review_required",
              "match_method": method, "source_sha256": term["source_sha256"], "source_url": term["source_url"],
              "source_locator": term["source_locator"],
              "target_fsn": term.get("fsn", ""), "matched_descriptions_json": "[]", "matched_query_terms_json": "[]",
              "required_attribute_constraints_json": terminology_json(constraints or []),
              "rationale": rationale, "reviewer": "", "reviewed_at": "", "source_reference": term["source_url"] + " " + term["source_locator"]}
    values["mapping_id"] = "map:" + hashlib.sha256(terminology_json({k: values[k] for k in (
        "dataset_class_id", "terminology", "target_id", "source_sha256", "required_attribute_constraints_json"
    )}).encode()).hexdigest()[:24]
    return values


def terminology_snomed_query_aliases(classes, evidence_bundle=None):
    """Build source-traceable retrieval seeds; their use never approves a map."""
    columns = ["class_id", "term", "query_source", "query_evidence_ids_json", "query_rule_id"]
    if evidence_bundle is None:
        return pd.DataFrame(columns=columns)
    profiles = {}
    for profile in evidence_bundle.get("profiles", []):
        key = terminology_key(profile.get("class_label", ""))
        if not key or key in profiles:
            raise ValueError("Retrieval profiles require unique nonempty class labels")
        profiles[key] = profile
    rows = []
    for class_row in classes.to_dict("records"):
        profile = profiles.get(terminology_key(class_row["normalized_label"]), {})
        policy = profile.get("snomed_concept_policy", {})
        sources = [("rag_class_lexical_label", profile.get("allowed_lexical_labels", []), profile.get("passage_ids", []), ""),
                   ("rag_snomed_search_alias", policy.get("search_aliases", []), profile.get("passage_ids", []), "")]
        for rule in policy.get("rules", []):
            for field in ("match_labels", "accepted_fsn_labels"):
                sources.append(("rag_snomed_rule_" + field, rule.get(field, []), rule.get("passage_ids", []), rule.get("rule_id", "")))
        for origin, labels, evidence, rule_id in sources:
            if not isinstance(labels, list) or any(not isinstance(label, str) or not label.strip() for label in labels):
                raise ValueError("RAG retrieval aliases must be lists of nonempty strings")
            for label in labels:
                rows.append({"class_id": class_row["dataset_class_id"], "term": label,
                             "query_source": origin, "query_evidence_ids_json": terminology_json(sorted(set(evidence))),
                             "query_rule_id": rule_id})
    return pd.DataFrame(rows, columns=columns).drop_duplicates().reset_index(drop=True)


def terminology_snomed_description_policies(classes, evidence_bundle=None):
    """Bind verified profile settings to the unchanged Phase 2 class identities."""
    profiles = {}
    for profile in (evidence_bundle or {}).get("profiles", []):
        key = terminology_key(profile.get("class_label", ""))
        if not key or key in profiles:
            raise ValueError("Description profiles require unique nonempty class labels")
        profiles[key] = profile
    result = {}
    for row in classes.to_dict("records"):
        profile = profiles.get(terminology_key(row["normalized_label"]), {})
        policy = profile.get("snomed_concept_policy", {})
        result[row["dataset_class_id"]] = snomed_description_policy(
            policy.get("description_policy"))
    return result


def terminology_build_crosswalk(classes, bundles, mst_proposals, snomed=None, reviews=None, evidence_bundle=None):
    """Keep source terminology acceptance separate from dataset crosswalk approval."""
    review_rows = [] if reviews is None else reviews.get("reviews", [])
    if reviews is not None and reviews.get("schema_version") != "mmvqa-terminology-mapping-review-v1":
        raise ValueError("Unsupported mapping-review schema")
    class_rows = classes.to_dict("records")
    classes_by_id = {r["dataset_class_id"]: r for r in class_rows}
    if len(classes_by_id) != len(class_rows) or not class_rows or any(not r.get("dataset_class_id") for r in class_rows):
        raise ValueError("Dataset class IDs must be nonempty and unique")
    classes_by_label = {terminology_key(r["normalized_label"]): r for r in class_rows}
    if len(classes_by_label) != len(class_rows):
        raise ValueError("Search-key collision between distinct Phase 2 classes; explicit disambiguation required")
    term_frames = [bundle["terms"] for bundle in bundles.values()]
    terms = pd.concat(term_frames, ignore_index=True)
    if terms.duplicated(["terminology", "term_id"]).any():
        raise ValueError("Duplicate term IDs within a terminology source")
    registry = {(row["terminology"], row["term_id"]): row for row in terms.to_dict("records")}
    candidates = []
    # Lexical matches are suggestions, including homonyms; they are not approval.
    for row in terms.to_dict("records"):
        # Do not bind ERCP/EUS diagnoses to capsule classes merely by matching text.
        if row["terminology"] == "MST" and row["term_type"] == "diagnosis" and row["section"] != "Enteroscopy":
            continue
        class_row = classes_by_label.get(terminology_key(row["preferred_term"]))
        if class_row:
            candidates.append(terminology_candidate(class_row, row))
    proposal_audit = []
    mst_bundle = bundles.get("MST")
    mst_source_hash = mst_bundle["manifest"]["sha256"] if mst_bundle is not None else None
    for proposal in mst_proposals["rows"]:
        class_row = classes_by_label.get(terminology_key(proposal["class_label"]))
        if class_row is None:
            continue
        audit = {"dataset_class_id": class_row["dataset_class_id"],
                 "normalized_label": class_row["normalized_label"], "target_id": proposal["target_id"],
                 "proposal_source_sha256": proposal["source_sha256"],
                 "imported_source_sha256": mst_source_hash,
                 "status": "applied_candidate", "reason": "Source-located proposal matches this imported snapshot; mapping review is still required."}
        proposal_audit.append(audit)
        if mst_bundle is None:
            audit.update(status="source_required", reason="MST source has not been imported.")
            continue
        if proposal["source_sha256"] != mst_source_hash:
            audit.update(status="source_version_mismatch", reason="Historical worksheet locators are not rebound to different source bytes; review against the new source is required.")
            continue
        if not proposal["target_id"]:
            audit.update(status="unmapped_source_proposal", reason=proposal["justification"])
            continue
        term = registry.get(("MST", proposal["target_id"]))
        if term is None or term["source_sha256"] != proposal["source_sha256"]:
            raise ValueError("An embedded MST proposal target is inconsistent with its matching source snapshot")
        constraints = proposal["required_attribute_constraints"]
        attrs = bundles["MST"]["attributes"]
        for constraint in constraints:
            found = attrs.loc[attrs["term_id"].eq(term["term_id"]) & attrs["attribute"].eq(constraint["attribute"]) & attrs["value"].eq(constraint["value"])]
            if found.empty:
                raise ValueError("An MST proposal uses a constraint absent from the source workbook")
        candidates.append(terminology_candidate(class_row, term, proposal["relation"], "source_located_proposal", constraints, proposal["justification"]))

    snomed_candidates = pd.DataFrame()
    snomed_retrieval_audit = pd.DataFrame()
    snomed_subset = None
    snomed_blocked_ids, snomed_blocked_classes = set(), set()
    if snomed is not None:
        if "mapping_eligible" in snomed["concepts"]:
            snomed_blocked_ids = set(snomed["concepts"].loc[~snomed["concepts"]["mapping_eligible"].eq(True), "id"].astype(str))
        labels = classes[["dataset_class_id", "normalized_label"]].rename(columns={"dataset_class_id": "class_id", "normalized_label": "class_label"})
        labels["class_label"] = labels["class_label"].map(terminology_key)
        query_aliases = terminology_snomed_query_aliases(classes, evidence_bundle)
        description_policies = terminology_snomed_description_policies(classes, evidence_bundle)
        snomed_candidates = propose_snomed_candidates(
            labels, snomed, aliases=query_aliases, description_policies=description_policies)
        snomed_retrieval_audit = snomed_candidate_retrieval_audit(
            labels, snomed, aliases=query_aliases, candidates=snomed_candidates,
            description_policies=description_policies)
        if not snomed_candidates.empty:
            snomed_blocked_classes = set(snomed_candidates.loc[
                snomed_candidates["concept_id"].isin(snomed_blocked_ids), "class_id"])
        seed_ids = set(snomed_candidates.get("concept_id", []))
        seed_ids.update(str(r.get("target_id", "")) for r in review_rows if r.get("terminology") == "SNOMED_CT" and r.get("target_id"))
        snomed_subset = snomed_ancestor_subset(snomed, seed_ids)
        snomed_terms = []
        for concept in snomed_subset["concepts"].to_dict("records"):
            row = {"terminology": "SNOMED_CT", "version": snomed["manifest"]["release_date"],
                   "term_id": concept["id"], "preferred_term": concept.get("preferred_term") or concept.get("fsn") or concept["id"],
                   "fsn": concept.get("fsn", ""),
                   "term_key": "", "term_type": concept.get("semantic_tag", ""), "section": "selected_class_candidates_and_ancestors",
                   "organ": "", "source_url": "http://snomed.info/id/" + concept["id"],
                   "source_locator": "RF2 Concept/Description/Language Snapshot; " + snomed["manifest"]["release_date"],
                   "source_sha256": snomed["manifest"]["source_fingerprint"]}
            row["term_key"] = mst_lookup_key(row["preferred_term"])
            registry[("SNOMED_CT", row["term_id"])] = row
            snomed_terms.append(row)
        terms = pd.concat([terms, pd.DataFrame(snomed_terms, columns=MST_TERM_COLUMNS)], ignore_index=True)
        for (class_id, concept_id), matches in snomed_candidates.groupby(["class_id", "concept_id"], sort=True):
            method = "active_accepted_snomed_description_match"
            if not (
                matches["active"].astype(str).eq("1") & matches["language_active"].astype(str).eq("1")
                & matches["acceptability"].isin(["preferred", "acceptable"])
                & matches["term_type"].isin(["fsn", "synonym"]) & matches["refset_id"].ne("")
            ).any():
                method = "active_snomed_description_match"  # Retrieval is not language acceptance.
            candidate = terminology_candidate(classes_by_id[class_id], registry[("SNOMED_CT", concept_id)],
                                              method=method)
            provenance = [field for field in ("description_id", "refset_id", "term", "acceptability", "query_term",
                          "query_source", "query_evidence_ids_json", "query_rule_id") if field in matches]
            provenance += [field for field in ("term_type", "active", "language_active", "language_member_id") if field in matches]
            candidate["matched_descriptions_json"] = terminology_json(matches[provenance].drop_duplicates().to_dict("records"))
            candidate["matched_query_terms_json"] = terminology_json(sorted(set(matches["query_term"])))
            candidates.append(candidate)

    # Later explicit source-located proposals replace duplicate lexical suggestions.
    dedup = {(r["dataset_class_id"], r["terminology"], r["target_id"], r["required_attribute_constraints_json"]): r for r in candidates}
    reviewed_keys = set()
    for review in review_rows:
        by_id = classes_by_id.get(review.get("dataset_class_id"))
        by_label = classes_by_label.get(terminology_key(review.get("class_label", "")))
        if review.get("dataset_class_id") and by_id is None:
            raise ValueError("Mapping review has an unknown dataset_class_id")
        if review.get("class_label") and by_label is None:
            raise ValueError("Mapping review has an unknown class_label")
        if by_id is not None and by_label is not None and by_id["dataset_class_id"] != by_label["dataset_class_id"]:
            raise ValueError("Mapping review dataset_class_id and class_label disagree")
        class_row = by_id or by_label
        if class_row is None:
            raise ValueError("Mapping review refers to an unknown Phase 2 class")
        terminology = review.get("terminology", "")
        target_id = str(review.get("target_id", ""))
        term = registry.get((terminology, target_id))
        if term is None:
            raise ValueError(f"Mapping review target is absent from the imported source: {terminology}/{target_id}")
        if review.get("source_sha256") != term["source_sha256"]:
            raise ValueError("Mapping review must cite the exact imported source fingerprint")
        decision, relation = review.get("decision"), review.get("relation")
        if decision not in {"approved", "rejected", "review"} or relation not in {"exact", "broader", "narrower", "related", "compositional"}:
            raise ValueError("Invalid review decision or mapping relation")
        constraints = review.get("required_attribute_constraints", [])
        if terminology == "SNOMED_CT" and (relation == "compositional" or constraints):
            raise ValueError("SNOMED postcoordination is outside this import stage; use a reviewed existing concept or keep the mapping for review")
        if decision == "approved":
            if terminology == "SNOMED_CT" and (target_id in snomed_blocked_ids or class_row["dataset_class_id"] in snomed_blocked_classes):
                raise ValueError("SNOMED source-description ambiguity blocks approval for this class; an explicit review cannot bypass unresolved source conflicts")
            for field in ("reviewer", "reviewed_at", "rationale", "source_reference"):
                if not str(review.get(field, "")).strip():
                    raise ValueError(f"Approved mapping requires {field}")
            if relation == "compositional" and not constraints:
                raise ValueError("A compositional mapping must preserve explicit attribute constraints")
            if terminology == "MST" and relation in {"exact", "compositional"}:
                # Preserve qualified dataset classes even if a different workbook
                # invalidates all historical row-located proposals. An exact map
                # to an explicitly qualified new term remains possible.
                qualifier = {"blood fresh": "Fresh", "blood hematin": "Hematin"}.get(
                    terminology_key(class_row["normalized_label"]))
                if qualifier and terminology_key(term["preferred_term"]) == "blood":
                    type_values = {r.get("value") for r in constraints if r.get("attribute") == "Type"}
                    if relation != "compositional" or type_values != {qualifier}:
                        raise ValueError("Qualified Phase 2 class requires its matching Blood Type constraint; an unqualified or conflicting map is not permitted")
                for proposal in mst_proposals["rows"]:
                    if (proposal["source_sha256"] != term["source_sha256"] or
                        terminology_key(proposal["class_label"]) != terminology_key(class_row["normalized_label"]) or proposal["target_id"] != target_id):
                        continue
                    required = {(r["attribute"], r["value"]) for r in proposal["required_attribute_constraints"]}
                    supplied = {(r.get("attribute"), r.get("value")) for r in constraints}
                    if required and (relation != "compositional" or not required.issubset(supplied)):
                        raise ValueError("Qualified Phase 2 class requires its source-attested compositional constraints; it cannot become an unqualified exact synonym")
            if constraints:
                attrs = bundles[terminology]["attributes"]
                for constraint in constraints:
                    if attrs.loc[attrs["term_id"].eq(target_id) & attrs["attribute"].eq(constraint["attribute"]) & attrs["value"].eq(constraint["value"])].empty:
                        raise ValueError("Reviewed mapping constraint not found in imported source")
        key = (class_row["dataset_class_id"], terminology, target_id, terminology_json(constraints))
        if key in reviewed_keys:
            raise ValueError("Duplicate review decision")
        reviewed_keys.add(key)
        row = terminology_candidate(class_row, term, relation, "explicit_review", constraints, review.get("rationale", ""))
        row.update(mapping_status={"approved": "approved", "rejected": "rejected", "review": "candidate_review_required"}[decision],
                   reviewer=review.get("reviewer", ""), reviewed_at=review.get("reviewed_at", ""),
                   source_reference=review.get("source_reference", ""))
        if key in dedup:
            for field in ("matched_descriptions_json", "matched_query_terms_json"):
                row[field] = dedup[key].get(field, "[]")
        dedup[key] = row

    rows = list(dedup.values())
    for class_row in class_rows:
        for terminology in ["MST", "CEST", "SNOMED_CT"]:
            if any(r["dataset_class_id"] == class_row["dataset_class_id"] and r["terminology"] == terminology for r in rows):
                continue
            available = terminology in bundles if terminology != "SNOMED_CT" else snomed is not None
            empty = {k: "" for k in TERMINOLOGY_CROSSWALK_COLUMNS}
            empty.update(dataset_class_id=class_row["dataset_class_id"], normalized_label=class_row["normalized_label"],
                         terminology=terminology, relation="unmapped", mapping_status="unmapped" if available else "source_required",
                         required_attribute_constraints_json="[]", matched_descriptions_json="[]", matched_query_terms_json="[]",
                         rationale="No justified mapping selected; retain original Phase 2 label.")
            empty["mapping_id"] = "map:" + hashlib.sha256(terminology_json([class_row["dataset_class_id"], terminology, empty["mapping_status"]]).encode()).hexdigest()[:24]
            rows.append(empty)
    crosswalk = pd.DataFrame(rows, columns=TERMINOLOGY_CROSSWALK_COLUMNS).sort_values(["normalized_label", "terminology", "target_id", "required_attribute_constraints_json"]).reset_index(drop=True)
    exact = crosswalk.loc[crosswalk["mapping_status"].eq("approved") & crosswalk["relation"].eq("exact")]
    if exact.duplicated(["dataset_class_id", "terminology"]).any():
        raise ValueError("Multiple exact maps in one terminology require explicit disambiguation")
    if exact["required_attribute_constraints_json"].ne("[]").any():
        raise ValueError("A constrained mapping must be compositional, not an unqualified exact synonym map")
    aliases = []
    for mapping in exact.to_dict("records"):
        if mapping["terminology"] == "SNOMED_CT":
            source_terms = snomed["terms"].loc[snomed["terms"]["concept_id"].eq(mapping["target_id"]) & snomed["terms"]["term_type"].eq("synonym")]
            for row in source_terms.to_dict("records"):
                aliases.append({"dataset_class_id": mapping["dataset_class_id"], "mapping_id": mapping["mapping_id"],
                                "terminology": "SNOMED_CT", "target_id": mapping["target_id"], "term": row["term"],
                                "source_description_id": row["description_id"], "language_refset_id": row["refset_id"], "language": row.get("language_code", "en"),
                                "source_sha256": mapping["source_sha256"], "approval_basis": "reviewed_exact_map_and_source_accepted_description"})
        else:
            aliases.append({"dataset_class_id": mapping["dataset_class_id"], "mapping_id": mapping["mapping_id"],
                            "terminology": mapping["terminology"], "target_id": mapping["target_id"], "term": mapping["target_term"],
                            "source_description_id": "", "language_refset_id": "", "language": "en", "source_sha256": mapping["source_sha256"],
                            "approval_basis": "reviewed_exact_map_to_source_term"})
            source_aliases = bundles[mapping["terminology"]]["synonyms"]
            for row in source_aliases.loc[source_aliases["term_id"].eq(mapping["target_id"])].to_dict("records"):
                aliases.append({"dataset_class_id": mapping["dataset_class_id"], "mapping_id": mapping["mapping_id"],
                                "terminology": mapping["terminology"], "target_id": mapping["target_id"], "term": row["synonym"],
                                "source_description_id": "", "language_refset_id": "", "language": row.get("language", "en"), "source_sha256": mapping["source_sha256"],
                                "approval_basis": "reviewed_exact_map_and_source_attested_synonym"})
    alias_columns = ["dataset_class_id", "mapping_id", "terminology", "target_id", "term", "source_description_id", "language_refset_id", "language", "source_sha256", "approval_basis"]
    source_synonyms = pd.concat([bundle["synonyms"].assign(terminology=name) for name, bundle in bundles.items()], ignore_index=True)
    return {"crosswalk": crosswalk, "terms": terms, "source_synonyms": source_synonyms,
            "approved_class_synonyms": pd.DataFrame(aliases, columns=alias_columns),
            "snomed_candidates": snomed_candidates, "snomed_subset": snomed_subset,
            "snomed_retrieval_audit": snomed_retrieval_audit,
            "mst_proposal_audit": pd.DataFrame(proposal_audit, columns=[
                "dataset_class_id", "normalized_label", "target_id", "proposal_source_sha256",
                "imported_source_sha256", "status", "reason"])}


def terminology_write_table(root, name, table):
    """Store flat Parquet and deterministic JSON records, retaining empty schemas."""
    root.mkdir(parents=True, exist_ok=True)
    if not len(table.columns):
        table = pd.DataFrame(columns=["status"])
    table.to_parquet(root / f"{name}.parquet", index=False)
    # Pandas conversion normalizes NaN/NA to JSON null instead of invalid NaN.
    records = json.loads(table.to_json(orient="records", force_ascii=False))
    (root / f"{name}.json").write_text(terminology_json(records) + "\n", encoding="utf-8")


def terminology_review_template(crosswalk):
    rows = []
    for row in crosswalk.loc[crosswalk["target_id"].ne("")].to_dict("records"):
        rows.append({"dataset_class_id": row["dataset_class_id"], "class_label": row["normalized_label"],
                     "terminology": row["terminology"], "target_id": row["target_id"], "target_term": row["target_term"],
                     "relation": row["relation"] if row["relation"] != "unassessed" else "related",
                     "decision": "review", "reviewer": "", "reviewed_at": "", "rationale": row["rationale"],
                     "source_reference": row["source_url"] + " " + row["source_locator"],
                     "source_sha256": row["source_sha256"],
                     "required_attribute_constraints": json.loads(row["required_attribute_constraints_json"])})
    return {"schema_version": "mmvqa-terminology-mapping-review-v1", "reviews": rows}

if IN_COLAB and CONFIG["mount_google_drive"]:
    drive.mount("/content/drive")
TERM_INPUT_ROOT = Path(TERMINOLOGY_CONFIG["input_root"] or (STORAGE_ROOT / "inputs" / "terminology")).expanduser().resolve()
if IN_COLAB and not TERM_INPUT_ROOT.is_relative_to(Path("/content/drive").resolve()):
    raise ValueError("Ontology input_root must be inside the Google Drive mount point")
TERM_INPUT_ROOT.mkdir(parents=True, exist_ok=True)
TERM_REGISTRY_ROOT = TERM_INPUT_ROOT / "_source_registry"
TERM_CACHE_ROOT = WORK / "terminology_cache"
TERM_OUTPUT = RUN_DIR / "terminology"
TERM_OUTPUT.mkdir(parents=True, exist_ok=True)
ONTOLOGY_MANIFEST_PATH = TERM_INPUT_ROOT / "ontology_manifest.json"
ONTOLOGY_STORAGE_MANIFEST = load_ontology_manifest(ONTOLOGY_MANIFEST_PATH)
SOURCE_STAGING, SOURCE_ACQUISITION = {}, {}


def resolve_ontology_secret(name):
    # Resolve only for an actual download; values are never logged or persisted.
    if IN_COLAB:
        try:
            value = userdata.get(name)
        except Exception:
            raise ValueError("Required ontology download secret is unavailable in Colab Secrets") from None
    else:
        value = os.environ.get(name)
    if not value:
        raise ValueError("Required ontology download secret is not configured")
    return value


def acquire_ontology_input(key, explicit, patterns, default_name):
    if explicit:
        destination = Path(explicit).expanduser().resolve()
    else:
        existing = terminology_file_choice(None, TERM_INPUT_ROOT, patterns)
        destination = existing or (TERM_INPUT_ROOT / default_name)
    if IN_COLAB and not destination.resolve().is_relative_to(Path("/content/drive").resolve()):
        raise ValueError("Ontology downloads must target the Google Drive mount point")
    result = acquire_terminology_source(destination, TERMINOLOGY_CONFIG["downloads"][key], secret_resolver=resolve_ontology_secret)
    SOURCE_ACQUISITION[key] = result
    record_ontology_source(ONTOLOGY_STORAGE_MANIFEST, key, result)
    save_ontology_manifest(ONTOLOGY_MANIFEST_PATH, ONTOLOGY_STORAGE_MANIFEST)
    return Path(result["path"]) if result["path"] is not None else None


def stage_terminology_input(source_path, key):
    record = terminology_stage_source(source_path, TERM_CACHE_ROOT, TERM_REGISTRY_ROOT, key,
                                      require_drive=IN_COLAB, copy_attempts=TERMINOLOGY_CONFIG["vm_copy_attempts"])
    acquisition = SOURCE_ACQUISITION.get(key, {"path": str(source_path), "status": "already_present"})
    if acquisition.get("sha256") and acquisition["sha256"] != record["source_sha256"]:
        raise ValueError("Ontology source changed in Drive between download and VM staging; the download reference was not accepted")
    SOURCE_STAGING[key] = record
    record_ontology_source(ONTOLOGY_STORAGE_MANIFEST, key, acquisition, record)
    save_ontology_manifest(ONTOLOGY_MANIFEST_PATH, ONTOLOGY_STORAGE_MANIFEST)
    return Path(record["local_path"])


def remember_ontology_ids(key, identifiers):
    record_ontology_source(ONTOLOGY_STORAGE_MANIFEST, key, SOURCE_ACQUISITION[key], SOURCE_STAGING[key], identifiers)
    save_ontology_manifest(ONTOLOGY_MANIFEST_PATH, ONTOLOGY_STORAGE_MANIFEST)


MST_DRIVE_PATH = acquire_ontology_input("MST", TERMINOLOGY_CONFIG["mst_source_path"], ["mst30_tables.xls"], "mst30_tables.xls")
if MST_DRIVE_PATH is None:
    raise FileNotFoundError("MST is missing from Drive and no download source is configured")
MST_PATH = stage_terminology_input(MST_DRIVE_PATH, "MST")
SOURCE_BUNDLES = {"MST": read_mst_workbook(MST_PATH, SOURCE_STAGING["MST"]["source_sha256"])}
SOURCE_STATUS = {"MST": "imported", "CEST": "source_required", "SNOMED_CT": "source_required"}
SOURCE_MANIFESTS = {"MST": SOURCE_BUNDLES["MST"]["manifest"]}
SOURCE_MANIFESTS["MST"]["source_artifact"] = SOURCE_STAGING["MST"]
remember_ontology_ids("MST", {"version": "3.0", "source_sha256": SOURCE_STAGING["MST"]["source_sha256"],
                            "term_ids": SOURCE_BUNDLES["MST"]["terms"]["term_id"].tolist(),
                            "identifier_scope": "local_workbook_locators_not_official_concept_codes"})

CEST_DRIVE_PATH = acquire_ontology_input("CEST_PACKAGE", TERMINOLOGY_CONFIG["cest_package_path"], ["cest_package.json"], "cest_package.json")
SOURCE_MANIFESTS["CEST"] = {"status": "source_required", "reason": "Original CEST source-backed transcription and document are required", "source_url": "https://doi.org/10.1055/s-2005-870329"}
if CEST_DRIVE_PATH is not None:
    CEST_PATH = stage_terminology_input(CEST_DRIVE_PATH, "CEST_PACKAGE")
    CEST_SOURCE_FILE = json.loads(CEST_PATH.read_text(encoding="utf-8-sig")).get("metadata", {}).get("source_file")
    if not CEST_SOURCE_FILE:
        raise ValueError("CEST metadata.source_file must identify the original document")
    CEST_DOCUMENT_REFERENCE = Path(CEST_SOURCE_FILE).expanduser()
    if not CEST_DOCUMENT_REFERENCE.is_absolute():
        CEST_DOCUMENT_REFERENCE = CEST_DRIVE_PATH.parent / CEST_DOCUMENT_REFERENCE
    # A downloaded package cannot choose arbitrary output paths for its document.
    # Existing referenced files are reused; absent documents download to input_root.
    CEST_DOCUMENT_DRIVE_PATH = acquire_ontology_input(
        "CEST_DOCUMENT", CEST_DOCUMENT_REFERENCE if CEST_DOCUMENT_REFERENCE.is_file() else None,
        ["cest_source.pdf"], "cest_source.pdf",
    )
    if CEST_DOCUMENT_DRIVE_PATH is not None:
        CEST_DOCUMENT_PATH = stage_terminology_input(CEST_DOCUMENT_DRIVE_PATH, "CEST_DOCUMENT")
        SOURCE_BUNDLES["CEST"] = read_cest_package(CEST_PATH, source_document_path=CEST_DOCUMENT_PATH)
        SOURCE_STATUS["CEST"] = "imported"
        SOURCE_MANIFESTS["CEST"] = SOURCE_BUNDLES["CEST"]["manifest"]
        SOURCE_MANIFESTS["CEST"]["source_artifacts"] = {key: SOURCE_STAGING[key] for key in ("CEST_PACKAGE", "CEST_DOCUMENT")}
        remember_ontology_ids("CEST_PACKAGE", {"version": SOURCE_MANIFESTS["CEST"]["version"],
            "source_fingerprint": SOURCE_MANIFESTS["CEST"]["source_fingerprint"],
            "document_source_sha256": SOURCE_STAGING["CEST_DOCUMENT"]["source_sha256"],
            "term_ids": SOURCE_BUNDLES["CEST"]["terms"]["term_id"].tolist()})
else:
    CEST_DOCUMENT_DRIVE_PATH = acquire_ontology_input("CEST_DOCUMENT", None, ["cest_source.pdf"], "cest_source.pdf")
    if CEST_DOCUMENT_DRIVE_PATH is not None:
        stage_terminology_input(CEST_DOCUMENT_DRIVE_PATH, "CEST_DOCUMENT")

SNOMED_DRIVE_PATH = acquire_ontology_input(
    "SNOMED_CT", TERMINOLOGY_CONFIG["snomed_rf2_path"], SNOMED_RF2_FILENAME_PATTERNS, "SnomedCT_RF2.zip")
SNOMED_SNAPSHOT = None
SNOMED_METADATA_IDS = {}
SNOMED_FSN = SNOMED_SYNONYM = SNOMED_PREFERRED = SNOMED_ACCEPTABLE = None
SNOMED_US_ENGLISH = SNOMED_GB_ENGLISH = SNOMED_INFERRED = SNOMED_IS_A = None
if SNOMED_DRIVE_PATH is not None:
    SNOMED_PATH = stage_terminology_input(SNOMED_DRIVE_PATH, "SNOMED_CT")
    with snomed_rf2_source(SNOMED_PATH) as (_, SNOMED_MEMBER_NAMES, _):
        SNOMED_COMPONENT_INVENTORY = snomed_rf2_component_inventory(
            SNOMED_MEMBER_NAMES, {"language_code": TERMINOLOGY_CONFIG["snomed_language_code"]})
    SNOMED_INVENTORY_PATH = TERM_OUTPUT / "snomed_rf2_component_inventory.json"
    SNOMED_INVENTORY_PATH.write_text(terminology_json({
        "source_sha256": SOURCE_STAGING["SNOMED_CT"]["source_sha256"],
        "language_code": TERMINOLOGY_CONFIG["snomed_language_code"],
        "snapshot_components": SNOMED_COMPONENT_INVENTORY,
        "archive_member_names": SNOMED_MEMBER_NAMES,
        "component_content_validated": False,
    }), encoding="utf-8")
    print("SNOMED Snapshot filename inventory (content validation follows):")
    display(pd.DataFrame(SNOMED_COMPONENT_INVENTORY))
    print("SNOMED component inventory saved:", SNOMED_INVENTORY_PATH)
    CACHED_SNOMED_READ_IDS = ontology_read_ids(ONTOLOGY_STORAGE_MANIFEST, "SNOMED_CT", SOURCE_STAGING["SNOMED_CT"]["source_sha256"])
    if (CACHED_SNOMED_READ_IDS.get("language_code") != TERMINOLOGY_CONFIG["snomed_language_code"]
        or (TERMINOLOGY_CONFIG["snomed_component_paths"] and TERMINOLOGY_CONFIG["snomed_component_paths"] != CACHED_SNOMED_READ_IDS.get("component_paths"))
        or (TERMINOLOGY_CONFIG["snomed_release_date"] and TERMINOLOGY_CONFIG["snomed_release_date"] != CACHED_SNOMED_READ_IDS.get("release_date"))):
        CACHED_SNOMED_READ_IDS = {}
    SNOMED_SNAPSHOT = read_snomed_snapshot(SNOMED_PATH, {
        "release_date": TERMINOLOGY_CONFIG["snomed_release_date"] or CACHED_SNOMED_READ_IDS.get("release_date"),
        "edition": TERMINOLOGY_CONFIG["snomed_edition"],
        "language_refset_ids": TERMINOLOGY_CONFIG["snomed_language_refset_ids"],
        "language_code": TERMINOLOGY_CONFIG["snomed_language_code"],
        "component_paths": TERMINOLOGY_CONFIG["snomed_component_paths"] or CACHED_SNOMED_READ_IDS.get("component_paths", {}),
        "chunk_size": 100000,
    })
    SOURCE_STATUS["SNOMED_CT"] = "imported"
    SOURCE_MANIFESTS["SNOMED_CT"] = SNOMED_SNAPSHOT["manifest"]
    SOURCE_MANIFESTS["SNOMED_CT"]["source_artifact"] = SOURCE_STAGING["SNOMED_CT"]
    SNOMED_METADATA_IDS = dict(SNOMED_SNAPSHOT["manifest"]["metadata_ids"])
    SNOMED_FSN = SNOMED_METADATA_IDS["SNOMED_FSN"]
    SNOMED_SYNONYM = SNOMED_METADATA_IDS["SNOMED_SYNONYM"]
    SNOMED_PREFERRED = SNOMED_METADATA_IDS["SNOMED_PREFERRED"]
    SNOMED_ACCEPTABLE = SNOMED_METADATA_IDS["SNOMED_ACCEPTABLE"]
    SNOMED_US_ENGLISH = SNOMED_METADATA_IDS["SNOMED_US_ENGLISH"]
    SNOMED_GB_ENGLISH = SNOMED_METADATA_IDS["SNOMED_GB_ENGLISH"]
    SNOMED_INFERRED = SNOMED_METADATA_IDS["SNOMED_INFERRED"]
    SNOMED_IS_A = SNOMED_METADATA_IDS["SNOMED_IS_A"]
    remember_ontology_ids("SNOMED_CT", snomed_read_ids(SNOMED_SNAPSHOT))
    terminology_write_table(TERM_OUTPUT, "snomed_metadata_audit", SNOMED_SNAPSHOT["metadata_audit"])
    SNOMED_METADATA_MANIFEST = {
        "source_sha256": SOURCE_STAGING["SNOMED_CT"]["source_sha256"],
        **{key: SNOMED_SNAPSHOT["manifest"][key] for key in (
            "source_fingerprint", "release_date", "metadata_ids", "metadata_discovery_policy",
            "metadata_discovery_policy_version", "metadata_source_components",
        )},
    }
    (TERM_OUTPUT / "snomed_metadata_manifest.json").write_text(
        terminology_json(SNOMED_METADATA_MANIFEST), encoding="utf-8")
    SNOMED_METADATA_DRIVE_ROOT = (TERM_INPUT_ROOT / "_source_audits" / "SNOMED_CT"
        / SNOMED_METADATA_MANIFEST["source_sha256"] / SNOMED_METADATA_MANIFEST["source_fingerprint"]
        / SNOMED_METADATA_MANIFEST["metadata_discovery_policy_version"])
    SNOMED_METADATA_DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
    for metadata_name in ("snomed_metadata_audit.json", "snomed_metadata_audit.parquet", "snomed_metadata_manifest.json"):
        metadata_local = TERM_OUTPUT / metadata_name
        metadata_copy = SNOMED_METADATA_DRIVE_ROOT / metadata_name
        shutil.copy2(metadata_local, metadata_copy)
        if terminology_stream_digest(metadata_local) != terminology_stream_digest(metadata_copy):
            raise IOError("SNOMED metadata audit SHA-256 differs after copying to Drive")
    print("SNOMED metadata IDs discovered from this RF2 Snapshot:")
    display(SNOMED_SNAPSHOT["metadata_audit"][["role", "sctid", "status", "role_use_count"]])
    # Persist diagnostic evidence before downstream mapping can fail.
    terminology_write_table(TERM_OUTPUT, "snomed_description_issues", SNOMED_SNAPSHOT["description_issues"])
    SNOMED_DESCRIPTION_AUDIT = {
        "source_sha256": SOURCE_STAGING["SNOMED_CT"]["source_sha256"],
        "source_fingerprint": SNOMED_SNAPSHOT["manifest"]["source_fingerprint"],
        **{key: SNOMED_SNAPSHOT["manifest"][key] for key in (
            "release_date", "selected_language_refset_ids", "component_files",
            "description_resolution_policy_version", "source_description_validation_status",
            "description_issue_count", "ambiguous_concept_count")},
    }
    (TERM_OUTPUT / "snomed_description_audit.json").write_text(terminology_json(SNOMED_DESCRIPTION_AUDIT), encoding="utf-8")
    SNOMED_AUDIT_DRIVE_ROOT = (TERM_INPUT_ROOT / "_source_audits" / "SNOMED_CT"
        / SNOMED_DESCRIPTION_AUDIT["source_sha256"] / SNOMED_DESCRIPTION_AUDIT["source_fingerprint"]
        / SNOMED_DESCRIPTION_AUDIT["description_resolution_policy_version"])
    SNOMED_AUDIT_DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
    for SNOMED_AUDIT_NAME in ("snomed_description_issues.json", "snomed_description_issues.parquet", "snomed_description_audit.json"):
        SNOMED_AUDIT_LOCAL = TERM_OUTPUT / SNOMED_AUDIT_NAME
        SNOMED_AUDIT_COPY = SNOMED_AUDIT_DRIVE_ROOT / SNOMED_AUDIT_NAME
        shutil.copy2(SNOMED_AUDIT_LOCAL, SNOMED_AUDIT_COPY)
        if terminology_stream_digest(SNOMED_AUDIT_LOCAL) != terminology_stream_digest(SNOMED_AUDIT_COPY):
            raise ValueError("SNOMED description audit copy failed SHA-256 verification")
    print("SNOMED description validation:", SNOMED_DESCRIPTION_AUDIT["source_description_validation_status"])
    print("SNOMED description issue rows:", SNOMED_DESCRIPTION_AUDIT["description_issue_count"],
          "; concepts withheld from mapping:", SNOMED_DESCRIPTION_AUDIT["ambiguous_concept_count"])
    print("SNOMED description audit saved in Drive:", SNOMED_AUDIT_DRIVE_ROOT)
    if not SNOMED_SNAPSHOT["description_issues"].empty:
        display(SNOMED_SNAPSHOT["description_issues"].head(20))
else:
    SOURCE_MANIFESTS["SNOMED_CT"] = {
        "status": "source_required",
        "reason": "No matching RF2 source was found; set snomed_rf2_path for another filename or configure an authorized download URL",
        "input_root": str(TERM_INPUT_ROOT),
        "filename_patterns": list(SNOMED_RF2_FILENAME_PATTERNS),
        "visible_zip_files": sorted(path.name for path in TERM_INPUT_ROOT.iterdir()
                                    if path.is_file() and path.suffix.casefold() == ".zip"),
    }

ONTOLOGY_READ_IDS = {key: entry.get("read_ids", {}) for key, entry in ONTOLOGY_STORAGE_MANIFEST["sources"].items()}
SUPPLEMENT_NAME = GI_SUPPLEMENT_PACKAGE["terminology"]
SOURCE_BUNDLES[SUPPLEMENT_NAME] = terminology_supplement(GI_SUPPLEMENT_PACKAGE)
SOURCE_STATUS[SUPPLEMENT_NAME] = "imported"
SOURCE_MANIFESTS[SUPPLEMENT_NAME] = SOURCE_BUNDLES[SUPPLEMENT_NAME]["manifest"]
display(pd.DataFrame([{"terminology": name, "status": state} for name, state in SOURCE_STATUS.items()]))
display(pd.DataFrame([{"source": key, "acquisition": record["status"], "path": record.get("path")}
                      for key, record in SOURCE_ACQUISITION.items()]))
display(pd.DataFrame([{key: record[key] for key in ("source_key", "source_sha256", "vm_sha256", "verification_status", "cache_status")} for record in SOURCE_STAGING.values()]))
if SNOMED_SNAPSHOT is not None:
    print("SNOMED language refsets discovered/selected:", ONTOLOGY_READ_IDS["SNOMED_CT"]["selected_language_refset_ids"])
print("Ontology storage manifest:", ONTOLOGY_MANIFEST_PATH)
MST_LABEL_ALIGNMENT = SOURCE_BUNDLES["MST"]["label_alignment"]
print("MST source-label alignment methods:")
display(MST_LABEL_ALIGNMENT.groupby("link_status").size().rename("rows").reset_index())
print("MST accepted non-exact source-label links:")
display(MST_LABEL_ALIGNMENT.loc[
    MST_LABEL_ALIGNMENT["term_id"].ne("") & MST_LABEL_ALIGNMENT["match_method"].ne("exact_source_label"),
    ["source_label", "target_term", "match_method", "source_locator", "target_source_locator"],
])
print("MST labels with words absent from the other source sheets:")
display(SOURCE_BUNDLES["MST"]["labels_missing_words"][[
    "source_label", "missing_tokens_json", "source_locator",
]])
print("MST other unresolved labels (ambiguous or incompatible source context):")
display(SOURCE_BUNDLES["MST"]["labels_unresolved"][[
    "source_label", "reason", "candidate_ids_json", "source_locator",
]])
print("MST attribute-layout issues retained:", len(SOURCE_BUNDLES["MST"]["attribute_issues"]))
display(SOURCE_BUNDLES["MST"]["attribute_issues"])


Mounted at /content/drive
SNOMED Snapshot filename inventory (content validation follows):


,kind,path,filename_language,release_date,edition_tag,language_compatible
0,language_refset,SnomedCT_Canadian_EditionRelease_PRODUCTION_20...,,20260831,CanadianEdition,True
1,concepts,SnomedCT_Canadian_EditionRelease_PRODUCTION_20...,,20260831,CanadianEdition,True
2,descriptions,SnomedCT_Canadian_EditionRelease_PRODUCTION_20...,,20260831,CanadianEdition,True
3,relationships,SnomedCT_Canadian_EditionRelease_PRODUCTION_20...,,20260831,CanadianEdition,True


SNOMED component inventory saved: /content/mmvqa_phase3_sector1/outputs/phase3/sector1/runs/20261113T172734_754892Z/terminology/snomed_rf2_component_inventory.json
Active SNOMED relationship counts: {'900000000000011006': 1745734, '900000000000227009': 42949}
SNOMED metadata IDs discovered from this RF2 Snapshot:


,role,sctid,status,role_use_count
0,SNOMED_FSN,900000000000003001,discovered,806701
1,SNOMED_SYNONYM,900000000000013009,discovered,1185384
2,SNOMED_PREFERRED,900000000000548007,discovered,3901228
3,SNOMED_ACCEPTABLE,900000000000549004,discovered,915669
4,SNOMED_US_ENGLISH,900000000000509007,discovered,1489051
5,SNOMED_GB_ENGLISH,900000000000508004,discovered,1368893
6,SNOMED_INFERRED,900000000000011006,discovered,1745734
7,SNOMED_IS_A,116680003,discovered,730918


SNOMED description validation: localized_ambiguities_retained
SNOMED description issue rows: 93 ; concepts withheld from mapping: 85
SNOMED description audit saved in Drive: /content/drive/MyDrive/MMVQA_Clinical/inputs/terminology/_source_audits/SNOMED_CT/3c897d349ed1b63cd17358625f5242f5aa89113198efd05c4539c80523bb0e81/123a8b54fb3bb27e46e2a16d8b23af3538b4e3290f9823f01e3494565bc3e4d9/snomed-description-resolution-v1


,concept_id,description_id,language_member_id,refset_id,type_id,term,description_module_id,language_module_id,description_effective_time,language_effective_time,reason_code
0,146921000146105,,,,,,,,,,ambiguous_fsn
1,166291000087109,1080271000087117,a979b9cc-3a5d-11f0-a788-38f3ab70978c,900000000000509007,900000000000013009,Pinaverium bromide,20621000087109,20621000087109,20250531,20250531,ambiguous_preferred_descriptions
2,166291000087109,6578004010,b27d8488-8712-4055-baec-3aa198913fed,900000000000509007,900000000000013009,Pinaverium bromide only product,900000000000207008,900000000000207008,20260701,20260701,ambiguous_preferred_descriptions
3,201991000087106,,,,,,,,,,ambiguous_fsn
4,202001000087109,,,,,,,,,,ambiguous_fsn
5,202011000087106,,,,,,,,,,ambiguous_fsn
6,202021000087103,,,,,,,,,,ambiguous_fsn
7,202031000087101,,,,,,,,,,ambiguous_fsn
8,202041000087107,,,,,,,,,,ambiguous_fsn
9,202051000087105,,,,,,,,,,ambiguous_fsn


... (9 outputs left out)

MST other unresolved labels (ambiguous or incompatible source context):


,source_label,reason,candidate_ids_json,source_locator
2,Extrinsic compression,known_words_without_compatible_term,[],Luminal_findings_organ!R8C2


MST attribute-layout issues retained: 9


,source_locator,reason,term_id,attribute_raw,value_raw
0,Luminal_findings!R47,ambiguous_attribute_layout,MST3:Luminal_findings:R47:C2,,Patchy
1,Luminal_findings!R48,ambiguous_attribute_layout,MST3:Luminal_findings:R48:C2,,Generalised
2,Luminal_findings!R49,ambiguous_attribute_layout,MST3:Luminal_findings:R49:C2,,v
3,Luminal_findings!R50,ambiguous_attribute_layout,MST3:Luminal_findings:R50:C2,,v
4,Luminal_findings!R51,ambiguous_attribute_layout,MST3:Luminal_findings:R51:C2,,v
5,Luminal_findings!R52,ambiguous_attribute_layout,MST3:Luminal_findings:R52:C2,,v
6,Luminal_findings!R53,ambiguous_attribute_layout,MST3:Luminal_findings:R53:C2,,v
7,Luminal_findings!R61,ambiguous_attribute_layout,MST3:Luminal_findings:R61:C2,,Patchy
8,Luminal_findings!R62,ambiguous_attribute_layout,MST3:Luminal_findings:R62:C2,,Generalised


In [8]:
if TERMINOLOGY_CONFIG.get("mapping_review_path"):
    raise ValueError(
        "A loca file overwrites the semantic policy."
    )

RAG_GITHUB_CONFIG["ref"] = "main"

RAG_EVIDENCE_PACKAGE, RAG_SOURCE_PROVENANCE = load_github_rag_evidence(
    repository=RAG_GITHUB_CONFIG["repository"],
    ref=RAG_GITHUB_CONFIG["ref"],
    manifest_path=RAG_GITHUB_CONFIG["manifest_path"],
    cache_root=WORK / "rag_github_cache",
    fetch=STORE._download,
    attempts=RAG_GITHUB_CONFIG["download_attempts"],
)

ulcer_profile = next(
    p for p in RAG_EVIDENCE_PACKAGE["profiles"]
    if p["class_label"] == "ulcer"
)

ulcer_rules = ulcer_profile["snomed_concept_policy"]["rules"]

assert {
    (tuple(r["semantic_tags"]), r["relation"])
    for r in ulcer_rules
} == {
    (("disorder",), "exact"),
    (("morphologic abnormality",), "related"),
}, "Loaded Ulcer policy doesn't match."

print("Semantic policy:", RAG_EVIDENCE_PACKAGE["evidence_version"])
print("SHA-256 verified:", RAG_SOURCE_PROVENANCE["vm_sha256"])

GitHub read: /bogdanparvu18/msc-graduate-project/main/outputs/phase3/sector1/terminology/rag_evidence_manifest.json
GitHub read: /bogdanparvu18/msc-graduate-project/main/outputs/phase3/sector1/terminology/phase3_01_rag_evidence.json
GitHub read: /bogdanparvu18/msc-graduate-project/main/outputs/phase3/sector1/terminology/rag_evidence_manifest.json
Semantic policy: 2026-11-13.5
SHA-256 verified: 707ae6de6eb294564132b82a4189b99ee273501b4f54e8936f7159580de15b85


## 8. Build source-linked class mappings and approved synonym tables

The inventory always retains all observed Phase 2 classes. Exact lexical matches and applicable source-located MST proposals remain review candidates. Embedded proposals retain the fingerprint of the workbook used to prepare them: a different Drive source is imported normally, but old row-based proposals are skipped and audited until reviewed against the new source. Their fingerprints are never silently rebound. The proposal for each blood class keeps its Type qualifier; related anatomical expressions are not treated as exact synonyms.

The dated RAG evidence package fetched and SHA-256 verified in section 1 contains source URLs, precise passage locations, short paraphrases and AI-assisted mapping decisions. SNOMED retrieval now uses original class labels and source-supported search aliases from its concept policies. The query audit distinguishes no lexical match from an active description absent from the selected language refsets. Search aliases propose candidates; they do not approve them.

The deterministic reviewer binds each policy to an actual active concept, accepted source descriptions, the full FSN label and its semantic type. A preferred label such as `Structure of ampulla of Vater` can represent the same approved anatomical concept as the accepted source synonym `Ampulla of Vater`; the preferred term need not repeat the class wording. Qualified or competing concepts remain distinguishable. Explicit policies can approve `related` terminology links without turning an object, substance or disorder into an exact synonym for an image finding. No concept identifier is invented and no SNOMED postcoordinated expression or MRCM validation is claimed.

No manual medical approval file is required. `approved` means accepted for this project's scoped terminology use by an **AI-assisted RAG review**, with `human_reviewed=False`; it does not mean clinician approval or validation of individual images. Missing sources, native-only classes and unresolved associations remain explicit. Existing raw Phase 2 labels are retained. After an exact concept mapping is approved, all its active, source-accepted synonym descriptions are eligible without a second hand-written lexical allowlist. Every description/refset provenance row remains available. `distinct_class_vocabulary` groups class/term/language entries while retaining all provenance and concept contexts; ambiguous surface forms require their concept/class context. Related and compositional mappings cannot become interchangeable aliases. The report shows both provenance-row counts and distinct vocabulary counts.

This contributes to **step 2 — defining permitted descriptions**, using the mappings from step 1. The vocabulary says which expressions are semantically suitable; verified Phase 2 observations still determine what can be asserted about a frame. It does not create severity, size, shape or disease diagnoses. All existing task/evidence rules and expected coverage outcomes remain unchanged.

The literature retrieval was performed when preparing the evidence package, not during every notebook run. Section 1 fetches the JSON and its manifest from `main`; no RAG commit is fixed in the configuration. The VM file SHA-256 must match `artifact.sha256` in that manifest before the evidence is used. A cached copy is reused only after this check against the freshly fetched manifest. A failed transfer or mismatch is retried up to three times. The manifest is checked again to detect an update during download. Each run exports the exact accepted file and its provenance, so changes on main remain traceable. The same input bytes produce the same review; changing an ontology version cannot silently reuse old MST row decisions. `mapping_review_path` is an optional explicit override, never auto-discovered or required.


In [9]:
"""Deterministic project mapping decisions from a frozen, cited RAG evidence bundle.

This stage neither retrieves documents nor changes the generation contract. The
evidence bundle records an AI-assisted literature assessment; project acceptance
is not a claim of human clinical review, image truth, or MRCM validation.
"""



RAG_REVIEWER = "AI-assisted RAG review"
RAG_LEXICAL_TERMINOLOGIES = {"CEST", "SNOMED_CT", "SBCE_VASCULAR_DELPHI_2019"}
RAG_RELATIONS = {"exact", "broader", "narrower", "related", "compositional"}
SNOMED_CONCEPT_POLICY_VERSION = "snomed-concept-policy-v1"
RAG_AUDIT_COLUMNS = [
    "audit_id", "mapping_id", "dataset_class_id", "normalized_label", "terminology",
    "target_id", "target_term", "target_type", "source_sha256", "verdict", "reason_code",
    "reason", "relation", "required_attribute_constraints_json", "review_method",
    "human_reviewed", "reviewer", "reviewed_at", "evidence_ids_json",
    "source_reference", "bundle_sha256", "target_fsn", "concept_policy_rule_id",
    "matched_source_descriptions_json",
]


def rag_json(value):
    return json.dumps(value, ensure_ascii=False, sort_keys=True, separators=(",", ":"), allow_nan=False)


def rag_text(value):
    if value is None or (not isinstance(value, (dict, list, tuple)) and pd.isna(value)):
        return ""
    return str(value).strip()


def rag_lexical_key(value):
    """Case/Unicode/space normalization only; never drop medical qualifiers."""
    return " ".join(unicodedata.normalize("NFKC", rag_text(value)).casefold().split())


def rag_records(value):
    return value.to_dict("records") if isinstance(value, pd.DataFrame) else list(value or [])


def rag_select_records(table, column, values):
    """Filter RF2 tables before materializing Python records."""
    if isinstance(table, pd.DataFrame):
        if column not in table:
            return []
        return table.loc[table[column].astype(str).isin(values)].to_dict("records")
    return [row for row in rag_records(table) if rag_text(row.get(column)) in values]


def rag_constraints(value):
    if isinstance(value, str):
        value = json.loads(value)
    if value is None:
        value = []
    if not isinstance(value, list) or any(
        not isinstance(item, dict) or not rag_text(item.get("attribute")) or not rag_text(item.get("value"))
        for item in value
    ):
        raise ValueError("RAG mapping constraints must be attribute/value records")
    return sorted(value, key=rag_json)


def rag_validate_bundle(bundle):
    if bundle.get("schema_version") != "mmvqa-rag-evidence-v1":
        raise ValueError("Unsupported RAG evidence schema")
    date.fromisoformat(bundle.get("review_date", ""))
    sources, passages, profiles, decisions = {}, {}, {}, {}
    for source in bundle.get("sources", []):
        key = rag_text(source.get("source_id"))
        url = urlparse(rag_text(source.get("url")))
        if not key or key in sources or url.scheme not in {"https", "http"} or not url.netloc:
            raise ValueError("RAG sources require unique IDs and actual HTTP(S) URLs")
        if not rag_text(source.get("title")) or type(source.get("independent_of_dataset")) is not bool:
            raise ValueError("RAG sources require a title and explicit dataset-independence flag")
        sources[key] = source
    for passage in bundle.get("passages", []):
        key = rag_text(passage.get("passage_id"))
        if not key or key in passages or passage.get("source_id") not in sources:
            raise ValueError("RAG passages require unique IDs and registered source IDs")
        if not rag_text(passage.get("location")) or not rag_text(passage.get("paraphrase")):
            raise ValueError("RAG passages require a source location and a substantive paraphrase")
        passages[key] = passage
    for profile in bundle.get("profiles", []):
        key = rag_lexical_key(profile.get("class_label"))
        if not key or key in profiles or profile.get("outcome") not in {"mapped", "native_only"}:
            raise ValueError("RAG class profiles require unique labels and a supported outcome")
        if not rag_text(profile.get("rationale")):
            raise ValueError("RAG class profiles require a rationale")
        profiles[key] = profile
    for decision in bundle.get("target_decisions", []):
        key = (rag_lexical_key(decision.get("class_label")), rag_text(decision.get("terminology")),
               rag_text(decision.get("target_id")))
        if not all(key) or key in decisions or key[0] not in profiles:
            raise ValueError("RAG target decisions require unique class/source/target keys and a profile")
        if decision.get("decision") not in {"approved", "rejected"} or decision.get("relation") not in RAG_RELATIONS:
            raise ValueError("RAG target decision or relation is unsupported")
        if not rag_text(decision.get("rationale")):
            raise ValueError("RAG target decisions require a rationale")
        rag_constraints(decision.get("required_attribute_constraints", []))
        decisions[key] = decision
    for owner in list(profiles.values()) + list(decisions.values()):
        ids = owner.get("passage_ids", [])
        if not isinstance(ids, list) or any(pid not in passages for pid in ids):
            raise ValueError("RAG decision references an unregistered passage")
    for profile in profiles.values():
        policy = profile.get("snomed_concept_policy")
        if policy is None:
            continue
        if not isinstance(policy, dict) or policy.get("policy_version") != SNOMED_CONCEPT_POLICY_VERSION:
            raise ValueError("Unsupported SNOMED concept policy")
        snomed_description_policy(policy.get("description_policy"))
        for field in ("search_aliases", "rules"):
            if not isinstance(policy.get(field), list):
                raise ValueError("SNOMED concept policy requires search_aliases and rules lists")
        if any(not isinstance(value, str) or not value.strip() for value in policy["search_aliases"]):
            raise ValueError("SNOMED search aliases must be nonempty strings")
        seen_rules = set()
        for rule in policy["rules"]:
            if not isinstance(rule, dict):
                raise ValueError("SNOMED concept rules must be objects")
            rule_id = rag_text(rule.get("rule_id"))
            if not rule_id or rule_id in seen_rules:
                raise ValueError("SNOMED concept rule IDs must be nonempty and unique per profile")
            seen_rules.add(rule_id)
            if rule.get("relation") not in RAG_RELATIONS - {"compositional"} or not rag_text(rule.get("rationale")):
                raise ValueError("SNOMED concept rules require a supported relation and rationale")
            for field in ("semantic_tags", "accepted_fsn_labels", "match_labels"):
                values = rule.get(field)
                if not isinstance(values, list) or not values or any(not isinstance(value, str) or not value.strip() for value in values):
                    raise ValueError("SNOMED concept rules require nonempty semantic_tags, accepted_fsn_labels and match_labels")
            ids = rule.get("passage_ids")
            if not isinstance(ids, list) or not ids or any(pid not in passages for pid in ids):
                raise ValueError("SNOMED concept rule references an unregistered passage or has no evidence")
    digest = hashlib.sha256(rag_json(bundle).encode("utf-8")).hexdigest()
    return sources, passages, profiles, decisions, digest


def rag_evidence(profile, decision, sources, passages):
    ids = sorted(set(profile.get("passage_ids", []) + (decision or {}).get("passage_ids", [])))
    used_sources = {passages[pid]["source_id"] for pid in ids}
    independent_urls = {sources[sid]["url"] for sid in used_sources if sources[sid]["independent_of_dataset"]}
    dataset_urls = {sources[sid]["url"] for sid in used_sources
                    if not sources[sid]["independent_of_dataset"]
                    and sources[sid].get("source_role") == "dataset_definition"}
    # The same document cannot supply both dataset definition and independent corroboration.
    adequate = bool(dataset_urls and independent_urls - dataset_urls)
    refs = []
    for pid in ids:
        passage = passages[pid]
        source = sources[passage["source_id"]]
        refs.append(f"[{pid}] {source['title']}; {source['url']}; {passage['location']}")
    return ids, " | ".join(refs), adequate




def rag_snomed_check(row, snapshot, allowed):
    if snapshot is None:
        return "snomed_snapshot_required", "The parsed active SNOMED snapshot is unavailable."
    if rag_text(snapshot.get("manifest", {}).get("source_fingerprint")) != row["source_sha256"]:
        return "snomed_source_mismatch", "The candidate does not belong to the supplied SNOMED snapshot."
    concepts = rag_select_records(snapshot.get("concepts"), "id", {row["target_id"]})
    if len(concepts) != 1 or rag_text(concepts[0].get("active")) != "1":
        return "snomed_concept_not_active", "Exactly one active concept must support the candidate."
    concept = concepts[0]
    if concept.get("mapping_eligible", True) != True:
        return "snomed_source_description_ambiguity", "The imported concept has unresolved source-description conflicts; no canonical term or mapping may be accepted."
    fsn = rag_text(concept.get("fsn"))
    tag = re.search(r"\s*\(([^()]*)\)$", fsn)
    if (tag is None or rag_lexical_key(tag.group(1)) != rag_lexical_key(row["target_type"])
            or rag_lexical_key(concept.get("semantic_tag")) != rag_lexical_key(row["target_type"])):
        return "snomed_fsn_tag_mismatch", "The active FSN must have the permitted semantic tag."
    if rag_lexical_key(fsn[:tag.start()]) not in allowed:
        return "snomed_fsn_not_exact", "The FSN contains an unapproved concept label or additional medical qualifiers."
    accepted = [item for item in rag_select_records(snapshot.get("terms"), "concept_id", {row["target_id"]})
                if item.get("term_type") == "synonym"
                and rag_lexical_key(item.get("term")) == rag_lexical_key(row["target_term"])
                and rag_text(item.get("active")) == "1"
                and rag_text(item.get("language_active")) == "1"
                and item.get("acceptability") in {"preferred", "acceptable"}
                and rag_text(item.get("description_id")) and rag_text(item.get("refset_id"))]
    if not accepted:
        return "snomed_description_not_accepted", "An active source-accepted synonym description must support the target label."
    return None


def rag_snomed_concept_rule(row, snapshot, profile):
    """Resolve a scoped rule by concept identity, FSN and source descriptions.

    Preferred spelling need not repeat a dataset alias. Qualifiers in the FSN
    are never stripped: every approved FSN label is an explicit evidence rule.
    The returned source rows prove that the class anchor belongs to this exact
    imported concept. This does not construct postcoordinated expressions.
    """
    policy = profile.get("snomed_concept_policy")
    empty = {"rule": None, "target_fsn": "", "matched_descriptions": []}
    def failure(code, reason):
        return dict(empty, reason_code=code, reason=reason)
    if policy is None:
        return failure("snomed_policy_missing", "No concept-level SNOMED policy is defined for this class.")
    if snapshot is None:
        return failure("snomed_snapshot_required", "The parsed active SNOMED snapshot is unavailable.")
    if rag_text(snapshot.get("manifest", {}).get("source_fingerprint")) != row["source_sha256"]:
        return failure("snomed_source_mismatch", "The candidate does not belong to the supplied SNOMED snapshot.")
    concepts = rag_select_records(snapshot.get("concepts"), "id", {row["target_id"]})
    if len(concepts) != 1 or rag_text(concepts[0].get("active")) != "1":
        return failure("snomed_concept_not_active", "Exactly one active concept must support the candidate.")
    concept = concepts[0]
    empty["target_fsn"] = rag_text(concept.get("fsn"))
    if concept.get("mapping_eligible", True) != True:
        return failure("snomed_source_description_ambiguity", "The imported concept has unresolved source-description conflicts.")
    fsn = empty["target_fsn"]
    tag = re.search(r"\s*\(([^()]*)\)$", fsn)
    if (tag is None or rag_lexical_key(tag.group(1)) != rag_lexical_key(row["target_type"])
            or rag_lexical_key(concept.get("semantic_tag")) != rag_lexical_key(row["target_type"])):
        return failure("snomed_fsn_tag_mismatch", "The source FSN, concept semantic tag and candidate type must agree.")
    typed = [rule for rule in policy["rules"]
             if rag_lexical_key(row["target_type"]) in {rag_lexical_key(value) for value in rule["semantic_tags"]}]
    if not typed:
        return failure("snomed_no_rule_for_type", "No evidence-backed rule permits this concept's semantic type for the class.")
    labelled = [rule for rule in typed
                if rag_lexical_key(fsn[:tag.start()]) in {rag_lexical_key(value) for value in rule["accepted_fsn_labels"]}]
    if not labelled:
        return failure("snomed_fsn_qualifier_not_permitted", "The full source concept label is outside the scoped policy; no anatomical or clinical qualifier is removed.")
    description_policy = snomed_description_policy(policy.get("description_policy"))
    anchor_types = set(description_policy["anchor_types"])
    # An opted-in FSN must be the complete canonical FSN of this source concept.
    # Acceptance and identity gates apply equally to synonyms and FSNs.
    accepted = [value for value in rag_select_records(snapshot.get("terms"), "concept_id", {row["target_id"]})
                if value.get("term_type") in anchor_types
                and (value.get("term_type") != "fsn"
                     or rag_lexical_key(value.get("term")) == rag_lexical_key(fsn))
                and rag_text(value.get("active")) == "1"
                and rag_text(value.get("language_active")) == "1"
                and value.get("acceptability") in {"preferred", "acceptable"}
                and rag_text(value.get("description_id")) and rag_text(value.get("refset_id"))]
    if not any(rag_lexical_key(value.get("term")) == rag_lexical_key(row["target_term"]) for value in accepted):
        return failure("snomed_description_not_accepted", "The displayed preferred term requires an active accepted description on this same concept.")
    matches = []
    for rule in labelled:
        anchor_labels = {rag_lexical_key(value) for value in rule["match_labels"] + profile.get("allowed_lexical_labels", [])}
        anchors = [value for value in accepted if rag_lexical_key(value.get("term")) in anchor_labels]
        if "fsn" in anchor_types:
            anchors += [value for value in accepted if value.get("term_type") == "fsn"
                        and rag_lexical_key(fsn[:tag.start()]) in anchor_labels and value not in anchors]
        if anchors:
            matches.append((rule, anchors))
    if not matches:
        return failure("snomed_source_anchor_missing",
                       "No active accepted description of a permitted type supports a policy-approved class anchor.")
    if len(matches) != 1:
        return failure("snomed_policy_rule_ambiguity", "Multiple scoped rules match this concept; no rule is selected by ordering.")
    rule, anchors = matches[0]
    proof_fields = ("concept_id", "description_id", "refset_id", "term", "acceptability", "language_code", "term_type")
    proof_rows = []
    for value in anchors:
        item = {key: rag_text(value.get(key)) for key in proof_fields}
        item["language_code"] = (
            rag_text(value.get("language_code"))
            or rag_text(value.get("languageCode"))
        )
        proof_rows.append(rag_json(item))
    proof = sorted(set(proof_rows))
    return {"rule": rule, "target_fsn": fsn, "matched_descriptions": [json.loads(value) for value in proof],
            "reason_code": "source_concept_policy_match", "reason": rule["rationale"]}


def build_rag_mapping_reviews(candidate_result, evidence_bundle, snomed_snapshot=None):
    """Evaluate all candidates; return unchanged-crosswalk-compatible reviews and audit.

    ``approved`` means accepted by this project's frozen automated evidence policy.
    Unresolved candidates produce audit rows, not fictitious approvals or a required
    human review file. Existing crosswalk validation remains a separate final gate.
    """
    sources, passages, profiles, decisions, digest = rag_validate_bundle(evidence_bundle)
    rows = []
    for raw in rag_records(candidate_result.get("crosswalk")):
        row = {key: rag_text(raw.get(key)) for key in (
            "mapping_id", "dataset_class_id", "normalized_label", "terminology", "target_id",
            "target_term", "target_type", "source_sha256", "mapping_status", "match_method", "relation")}
        row["constraints"] = rag_constraints(raw.get("required_attribute_constraints_json", "[]"))
        rows.append(row)
    rows.sort(key=lambda row: rag_json(row))
    # Restrict large RF2 tables once to the concepts under review; no candidate
    # materializes every description in a national edition as Python objects.
    if snomed_snapshot is not None:
        candidate_ids = {row["target_id"] for row in rows if row["terminology"] == "SNOMED_CT" and row["target_id"]}
        snomed_snapshot = dict(snomed_snapshot)
        for table_name, id_column in (("concepts", "id"), ("terms", "concept_id")):
            table = snomed_snapshot.get(table_name)
            if isinstance(table, pd.DataFrame) and id_column in table:
                snomed_snapshot[table_name] = table.loc[table[id_column].astype(str).isin(candidate_ids)]
    snomed_blocked_ids = set()
    if snomed_snapshot is not None:
        concept_table = snomed_snapshot.get("concepts")
        if isinstance(concept_table, pd.DataFrame) and "mapping_eligible" in concept_table:
            snomed_blocked_ids = set(concept_table.loc[~concept_table["mapping_eligible"].eq(True), "id"].astype(str))
        elif not isinstance(concept_table, pd.DataFrame):
            snomed_blocked_ids = {rag_text(item.get("id")) for item in rag_records(concept_table)
                                 if item.get("mapping_eligible", True) != True}
    snomed_blocked_classes = {row["dataset_class_id"] for row in rows
                             if row["terminology"] == "SNOMED_CT" and row["target_id"] in snomed_blocked_ids}
    registry = {}
    for term in rag_records(candidate_result.get("terms")):
        key = (rag_text(term.get("terminology")), rag_text(term.get("term_id")))
        if key in registry:
            raise ValueError("Imported terminology registry has duplicate source/target IDs")
        registry[key] = term
    # Count before lexical filtering: a convenient label must not hide another
    # plausible concept with the same allowed semantic type.
    eligible = Counter()
    concept_matches, exact_concept_ids = {}, {}
    for row in rows:
        profile = profiles.get(rag_lexical_key(row["normalized_label"]), {})
        allowed_types = {rag_lexical_key(value) for value in profile.get("allowed_target_types", {}).get(row["terminology"], [])}
        if row["target_id"] and rag_lexical_key(row["target_type"]) in allowed_types:
            eligible[(row["dataset_class_id"], row["terminology"])] += 1
        if row["terminology"] == "SNOMED_CT" and row["target_id"] and profile.get("snomed_concept_policy") is not None:
            match = rag_snomed_concept_rule(row, snomed_snapshot, profile)
            concept_matches[rag_json(row)] = match
            if match["rule"] is not None and match["rule"]["relation"] == "exact":
                exact_concept_ids.setdefault(row["dataset_class_id"], set()).add(row["target_id"])

    audits, reviews = [], []
    for index, row in enumerate(rows):
        label = rag_lexical_key(row["normalized_label"])
        profile = profiles.get(label, {})
        decision = decisions.get((label, row["terminology"], row["target_id"]))
        concept_match = concept_matches.get(rag_json(row))
        selected_rule = concept_match["rule"] if concept_match is not None else None
        evidence_decision = decision
        if selected_rule is not None:
            evidence_decision = {"passage_ids": sorted(set((decision or {}).get("passage_ids", []) + selected_rule["passage_ids"]))}
        evidence_ids, reference, adequate = rag_evidence(profile, evidence_decision, sources, passages)
        relation, constraints = row["relation"], row["constraints"]
        verdict, code, reason = "unresolved", "no_curated_profile", "No frozen evidence profile exists for this class."
        term = registry.get((row["terminology"], row["target_id"]))
        allowed = {rag_lexical_key(value) for value in profile.get("allowed_lexical_labels", [])}
        types = {rag_lexical_key(value) for value in profile.get("allowed_target_types", {}).get(row["terminology"], [])}
        if not row["target_id"]:
            verdict = "source_required" if row["mapping_status"] == "source_required" else "unmapped"
            code, reason = verdict, "Source unavailable." if verdict == "source_required" else "No external mapping candidate; retain the Phase 2 label."
            if verdict == "unmapped" and row["terminology"] == "SNOMED_CT" and profile.get("snomed_concept_policy") is not None:
                code = "no_candidate_in_search_scope"
                reason = profile["snomed_concept_policy"].get("unmapped_reason") or "No accepted description matched the recorded query scope; this does not establish that SNOMED lacks the concept."
        elif not profile:
            pass
        elif profile.get("outcome") == "native_only":
            verdict, code, reason = "unmapped", "native_only_profile", profile["rationale"]
        elif not row["source_sha256"]:
            code, reason = "source_fingerprint_missing", "The candidate has no imported-source fingerprint."
        elif term is None:
            code, reason = "imported_target_missing", "The target is absent from the imported terminology registry."
        elif row["terminology"] == "SNOMED_CT" and row["dataset_class_id"] in snomed_blocked_classes:
            code, reason = "snomed_source_description_ambiguity", "A SNOMED candidate for this class has unresolved source-description conflicts. Preserve all candidates without approving an alternative by omission."
        elif (rag_text(term.get("source_sha256")) != row["source_sha256"]
              or rag_text(term.get("preferred_term")) != row["target_term"]
              or rag_text(term.get("term_type")) != row["target_type"]):
            code, reason = "imported_target_mismatch", "The candidate no longer matches its imported source record."
        elif not adequate:
            code, reason = "independent_evidence_missing", "Both a dataset definition and distinct independent medical evidence are required."
        elif decision is not None and (
            rag_text(decision.get("source_sha256")) != row["source_sha256"]
            or rag_text(decision.get("target_term")) != row["target_term"]
            or rag_text(decision.get("target_type")) != row["target_type"]
            or rag_constraints(decision.get("required_attribute_constraints", [])) != constraints
        ):
            code, reason = "pinned_target_mismatch", "The frozen decision does not match the imported target, source fingerprint, or attribute constraints."
        elif decision is not None and decision["decision"] == "rejected":
            verdict, code, reason = "rejected", "explicit_evidence_rejection", decision["rationale"]
            relation = decision["relation"]
        elif row["terminology"] == "MST":
            if decision is None:
                code, reason = "mst_target_not_pinned", "This MST target has no source-pinned evidence decision."
            elif decision["relation"] == "exact" and constraints:
                code, reason = "exact_mapping_has_constraints", "A qualified expression cannot become an interchangeable term."
            elif decision["relation"] == "compositional" and not constraints:
                code, reason = "composition_missing_constraints", "A compositional decision requires source-attested attribute constraints."
            else:
                verdict, code, reason = "accepted", "source_pinned_evidence_decision", decision["rationale"]
                relation = decision["relation"]
        elif row["terminology"] not in RAG_LEXICAL_TERMINOLOGIES:
            code, reason = "unsupported_terminology", "No automatic evidence policy is defined for this terminology."
        elif row["terminology"] == "SNOMED_CT" and profile.get("snomed_concept_policy") is not None:
            if constraints:
                code, reason = "snomed_postcoordination_not_executed", "This stage evaluates existing concepts; constrained SNOMED expressions require a separate postcoordination validator."
            elif row["match_method"] != "active_accepted_snomed_description_match":
                code, reason = "unsupported_match_method", "The candidate lacks the required active accepted source-description match."
                if row["match_method"] == "active_snomed_description_match":
                    code, reason = "snomed_description_not_accepted", "An active description was found, but no matched description is accepted in the selected language refsets."
            elif selected_rule is None:
                code, reason = concept_match["reason_code"], concept_match["reason"]
            elif decision is not None and decision["relation"] != selected_rule["relation"]:
                code, reason = "snomed_pinned_relation_conflict", "The explicit source-pinned relation conflicts with the concept-level evidence rule."
            elif selected_rule["relation"] == "exact" and len(exact_concept_ids.get(row["dataset_class_id"], set())) != 1:
                code, reason = "ambiguous_snomed_concept", "Multiple concepts satisfy exact semantic rules for this class; preferred spelling cannot resolve them."
            else:
                verdict, code, reason = "accepted", "source_concept_policy_match", selected_rule["rationale"]
                relation = selected_rule["relation"]
        elif rag_lexical_key(row["target_type"]) not in types:
            code, reason = "target_type_not_permitted", "The target's semantic type is outside the class evidence profile."
        elif eligible[(row["dataset_class_id"], row["terminology"])] != 1:
            code, reason = "ambiguous_target", "Multiple candidates remain after semantic-type filtering; lexical similarity cannot resolve them."
        elif constraints:
            code, reason = "lexical_mapping_has_constraints", "Lexical approval does not support constrained expressions."
        elif decision is not None and decision["relation"] != "exact":
            code, reason = "non_mst_relation_requires_policy", "This terminology has no automatic policy for non-exact mapping decisions."
        elif rag_lexical_key(row["target_term"]) not in allowed:
            code, reason = "target_label_not_curated", "The source label is not an evidence-supported exact lexical equivalent."
        elif row["match_method"] != ("active_accepted_snomed_description_match" if row["terminology"] == "SNOMED_CT" else "exact_lexical_candidate"):
            code, reason = "unsupported_match_method", "The candidate lacks the required exact source-description match."
        else:
            failure = rag_snomed_check(row, snomed_snapshot, allowed) if row["terminology"] == "SNOMED_CT" else None
            if failure:
                code, reason = failure
            else:
                verdict, code, reason = "accepted", "unique_evidence_supported_lexical_match", profile["rationale"]
                relation = "exact"

        audit = {key: row[key] for key in ("mapping_id", "dataset_class_id", "normalized_label", "terminology", "target_id", "target_term", "target_type", "source_sha256")}
        audit.update(verdict=verdict, reason_code=code, reason=reason, relation=relation,
                     required_attribute_constraints_json=rag_json(constraints), review_method="AI_assisted_RAG",
                     human_reviewed=False, reviewer=RAG_REVIEWER, reviewed_at=evidence_bundle["review_date"],
                     evidence_ids_json=rag_json(evidence_ids),
                     source_reference=reference, bundle_sha256=digest,
                     target_fsn=concept_match["target_fsn"] if concept_match is not None else "",
                     concept_policy_rule_id=selected_rule["rule_id"] if selected_rule is not None else "",
                     matched_source_descriptions_json=rag_json(concept_match["matched_descriptions"] if concept_match is not None else []))
        audit["audit_id"] = "rag:" + hashlib.sha256(rag_json([row, digest, index]).encode()).hexdigest()[:24]
        audits.append(audit)
        if verdict in {"accepted", "rejected"}:
            reviews.append({
                "dataset_class_id": row["dataset_class_id"], "class_label": row["normalized_label"],
                "terminology": row["terminology"], "target_id": row["target_id"], "target_term": row["target_term"],
                "decision": "approved" if verdict == "accepted" else "rejected", "relation": relation,
                "required_attribute_constraints": constraints, "source_sha256": row["source_sha256"],
                "rationale": reason, "reviewer": RAG_REVIEWER, "reviewed_at": evidence_bundle["review_date"],
                "source_reference": reference, "review_method": "AI_assisted_RAG", "human_reviewed": False,
                "evidence_ids": evidence_ids, "bundle_sha256": digest,
            })
    counts = Counter(item["verdict"] for item in audits)
    return {
        "reviews": {"schema_version": "mmvqa-terminology-mapping-review-v1", "reviews": reviews},
        "audit": pd.DataFrame(audits, columns=RAG_AUDIT_COLUMNS),
        "summary": {
            "schema_version": "mmvqa-rag-review-summary-v1", "review_date": evidence_bundle["review_date"],
            "bundle_sha256": digest, "candidate_count": len(rows), "review_count": len(reviews),
            "verdict_counts": {key: counts[key] for key in ("accepted", "rejected", "unresolved", "source_required", "unmapped")},
            "review_method": "AI_assisted_RAG", "human_reviewed": False,
            "acceptance_scope": "Project terminology mapping only; not human clinical validation or image-level truth.",
        },
    }


def build_source_synonym_audit(mapping_result, evidence_bundle, snomed_snapshot=None):
    """Accept source-attested descriptions on an approved exact concept mapping.

    Literature establishes the class-to-concept association. The imported
    terminology establishes which descriptions name that concept. Requiring
    every source synonym to occur verbatim in the literature allowlist would
    discard legitimate terminology. Related/compositional maps cannot enter
    this interchangeable vocabulary; all provenance rows remain auditable.
    """
    _, _, _, _, digest = rag_validate_bundle(evidence_bundle)
    base = mapping_result.get("approved_class_synonyms")
    base_columns = list(base.columns) if isinstance(base, pd.DataFrame) else [
        "dataset_class_id", "mapping_id", "terminology", "target_id", "term",
        "source_description_id", "language_refset_id", "language", "source_sha256", "approval_basis"]
    extra = ["rag_lexical_supported", "rag_decision", "reason_code", "reason", "term_key",
             "requires_concept_context", "concept_contexts_json", "rag_bundle_sha256"]
    columns = list(dict.fromkeys(base_columns + extra))
    rows = rag_records(base)
    if not rows:
        return pd.DataFrame(columns=columns)
    crosswalk = {}
    for mapping in rag_records(mapping_result.get("crosswalk")):
        crosswalk.setdefault(rag_text(mapping.get("mapping_id")), []).append(mapping)
    registry = {(rag_text(term.get("terminology")), rag_text(term.get("term_id"))): term
                for term in rag_records(mapping_result.get("terms"))}
    source_synonyms = rag_records(mapping_result.get("source_synonyms"))
    concept_by_id, snomed_descriptions = {}, {}
    source_homonyms = {}
    relevant_keys = {rag_lexical_key(row.get("term")) for row in rows}
    if snomed_snapshot is not None:
        relevant_ids = {rag_text(row.get("target_id")) for row in rows if row.get("terminology") == "SNOMED_CT"}
        for concept in rag_select_records(snomed_snapshot.get("concepts"), "id", relevant_ids):
            concept_by_id.setdefault(rag_text(concept.get("id")), []).append(concept)
        description_table = snomed_snapshot.get("terms")
        if isinstance(description_table, pd.DataFrame) and "term" in description_table:
            keys = (description_table["term"].astype(str).str.normalize("NFKC").str.casefold()
                    .str.replace(r"\s+", " ", regex=True).str.strip())
            description_rows = description_table.loc[keys.isin(relevant_keys)].to_dict("records")
        else:
            description_rows = [value for value in rag_records(description_table) if rag_lexical_key(value.get("term")) in relevant_keys]
        for description in description_rows:
            if (description.get("term_type") != "synonym" or rag_text(description.get("active")) != "1"
                    or rag_text(description.get("language_active")) != "1"
                    or description.get("acceptability") not in {"preferred", "acceptable"}):
                continue
            key = (rag_text(description.get("concept_id")), rag_text(description.get("description_id")),
                   rag_text(description.get("refset_id")))
            snomed_descriptions.setdefault(key, []).append(description)
            term_key = rag_lexical_key(description.get("term"))
            if term_key in relevant_keys:
                source_homonyms.setdefault(term_key, set()).add(rag_text(description.get("concept_id")))
    decisions = []
    for source_row in rows:
        row = dict(source_row)
        row.update(rag_lexical_supported=False, rag_decision="withheld_source_synonym",
                   reason_code="exact_mapping_required", reason="An approved unconstrained exact mapping is required.",
                   term_key=rag_lexical_key(row.get("term")), requires_concept_context=False,
                   concept_contexts_json="[]", rag_bundle_sha256=digest)
        matching = crosswalk.get(rag_text(row.get("mapping_id")), [])
        mapping = matching[0] if len(matching) == 1 else None
        valid_mapping = (mapping is not None and mapping.get("mapping_status") == "approved"
                         and mapping.get("relation") == "exact"
                         and not rag_constraints(mapping.get("required_attribute_constraints_json", "[]")))
        if not valid_mapping:
            decisions.append(row)
            continue
        identity_fields = ("dataset_class_id", "terminology", "target_id", "source_sha256")
        if (any(rag_text(row.get(key)) != rag_text(mapping.get(key)) for key in identity_fields)
                or not rag_text(row.get("source_sha256")) or not row["term_key"]):
            row.update(reason_code="mapping_source_identity_mismatch", reason="Synonym class, concept and source must match the exact approved mapping.")
            decisions.append(row)
            continue
        terminology, concept_id = rag_text(row.get("terminology")), rag_text(row.get("target_id"))
        accepted = False
        if terminology == "SNOMED_CT":
            concept_rows = concept_by_id.get(concept_id, [])
            concept_valid = (len(concept_rows) == 1 and rag_text(concept_rows[0].get("active")) == "1"
                             and concept_rows[0].get("mapping_eligible", True) == True)
            if snomed_snapshot is None or rag_text(snomed_snapshot.get("manifest", {}).get("source_fingerprint")) != row["source_sha256"]:
                row.update(reason_code="snomed_source_mismatch", reason="Synonym provenance does not match the imported snapshot.")
            elif not concept_valid:
                row.update(reason_code="snomed_concept_not_eligible", reason="The mapped concept must be active and free of unresolved source-description conflicts.")
            else:
                key = (concept_id, rag_text(row.get("source_description_id")), rag_text(row.get("language_refset_id")))
                descriptions = snomed_descriptions.get(key, [])
                accepted = bool(key[1] and key[2] and any(
                    rag_text(value.get("term")) == rag_text(row.get("term"))
                    and (not rag_text(value.get("language_code", value.get("languageCode")))
                         or rag_text(value.get("language_code", value.get("languageCode"))) == rag_text(row.get("language")))
                    for value in descriptions))
                if not accepted:
                    row.update(reason_code="snomed_description_not_accepted", reason="The exact description/refset/term must be active and preferred or acceptable on the same concept.")
        else:
            term = registry.get((terminology, concept_id))
            if term is None or rag_text(term.get("source_sha256")) != row["source_sha256"]:
                row.update(reason_code="imported_target_mismatch", reason="The mapped source term is absent or belongs to a different source fingerprint.")
            else:
                canonical = rag_text(term.get("preferred_term")) == rag_text(row.get("term"))
                attested = any(rag_text(value.get("terminology")) == terminology
                               and rag_text(value.get("term_id")) == concept_id
                               and rag_text(value.get("synonym")) == rag_text(row.get("term"))
                               and rag_text(value.get("source_sha256")) == row["source_sha256"]
                               and rag_text(value.get("language", "en")) == rag_text(row.get("language", "en"))
                               and bool(rag_text(value.get("source_locator")))
                               for value in source_synonyms)
                accepted = canonical or attested
                if not accepted:
                    row.update(reason_code="source_synonym_not_attested", reason="The alternate name must be attested for the same imported source term and fingerprint.")
        if accepted:
            row.update(rag_lexical_supported=True, rag_decision="accepted_source_synonym",
                       reason_code="source_accepted_description_on_exact_map",
                       reason="The imported terminology attests this name for the exact approved class concept; no literature spelling allowlist is required.")
        decisions.append(row)
    term_contexts = {}
    for row in decisions:
        if row["rag_lexical_supported"]:
            context = {key: rag_text(row.get(key)) for key in ("dataset_class_id", "terminology", "target_id", "mapping_id")}
            term_contexts.setdefault(row["term_key"], set()).add(rag_json(context))
    for row in decisions:
        contexts = sorted(term_contexts.get(row["term_key"], set()))
        context_rows = [json.loads(value) for value in contexts]
        # Terminologies can agree on one class without creating ambiguity. A
        # cross-class homonym or a SNOMED homonym requires its concept context.
        ambiguous = (len({value["dataset_class_id"] for value in context_rows}) > 1
                     or len(source_homonyms.get(row["term_key"], set())) > 1)
        row.update(requires_concept_context=ambiguous, concept_contexts_json=rag_json(context_rows))
    return pd.DataFrame(decisions, columns=columns)


def build_distinct_class_vocabulary(synonym_audit):
    """One spelling per class/language, retaining all source rows as provenance."""
    columns = ["dataset_class_id", "term_key", "language", "display_term", "provenance_count",
               "concept_context_count", "requires_concept_context", "provenance_json", "concept_contexts_json"]
    grouped = {}
    for row in rag_records(synonym_audit):
        if row.get("rag_decision") != "accepted_source_synonym":
            continue
        key = (rag_text(row.get("dataset_class_id")), rag_lexical_key(row.get("term")), rag_text(row.get("language")))
        grouped.setdefault(key, []).append(row)
    result = []
    for key, group in sorted(grouped.items()):
        provenance_fields = ("mapping_id", "terminology", "target_id", "term", "source_description_id",
                             "language_refset_id", "source_sha256", "approval_basis", "rag_bundle_sha256")
        provenance = [{field: rag_text(row.get(field)) for field in provenance_fields} for row in group]
        provenance.sort(key=rag_json)
        contexts = sorted({value for row in group for value in
                           [rag_json(item) for item in json.loads(row.get("concept_contexts_json", "[]"))]})
        result.append(dict(zip(columns, [*key, sorted({rag_text(row.get("term")) for row in group})[0],
                      len(group), len(contexts), any(bool(row.get("requires_concept_context")) for row in group),
                      rag_json(provenance), rag_json([json.loads(value) for value in contexts])])))
    return pd.DataFrame(result, columns=columns)

# Refresh the complete policy before applying generic description settings.
RAG_GITHUB_CONFIG["ref"] = "main"
RAG_EVIDENCE_PACKAGE, RAG_SOURCE_PROVENANCE = load_github_rag_evidence(
    repository=RAG_GITHUB_CONFIG["repository"],
    ref=RAG_GITHUB_CONFIG["ref"],
    manifest_path=RAG_GITHUB_CONFIG["manifest_path"],
    cache_root=WORK / "rag_github_cache",
    fetch=STORE._download,
    attempts=RAG_GITHUB_CONFIG["download_attempts"],
)

# Use the complete, manifest-verified policy from GitHub.
_, _, _, _, rag_policy_sha256 = rag_validate_bundle(RAG_EVIDENCE_PACKAGE)
if rag_policy_sha256 != RAG_SOURCE_PROVENANCE["bundle_content_sha256"]:
    raise ValueError("Politica RAG din memorie diferă de JSON-ul verificat.")

RAG_RUNTIME_EVIDENCE_PACKAGE = RAG_EVIDENCE_PACKAGE

# Cache is local to this execution, never reused across notebook reruns.
SNOMED_MAPPING_SNAPSHOT = snomed_mapping_run(SNOMED_SNAPSHOT)
try:
    CANDIDATE_MAPPING_RESULT = terminology_build_crosswalk(CLASSES, SOURCE_BUNDLES, MST_MAPPING_PROPOSALS, SNOMED_MAPPING_SNAPSHOT, evidence_bundle=RAG_RUNTIME_EVIDENCE_PACKAGE)
    RAG_RESULT = build_rag_mapping_reviews(CANDIDATE_MAPPING_RESULT, RAG_RUNTIME_EVIDENCE_PACKAGE, SNOMED_SNAPSHOT)
    RAG_MAPPING_AUDIT = RAG_RESULT["audit"]
    MAPPING_REVIEWS = RAG_RESULT["reviews"]
    REVIEW_PATH = None
    if TERMINOLOGY_CONFIG["mapping_review_path"]:
        REVIEW_DRIVE_PATH = terminology_file_choice(TERMINOLOGY_CONFIG["mapping_review_path"], TERM_INPUT_ROOT, [])
        REVIEW_PATH = stage_terminology_input(REVIEW_DRIVE_PATH, "MAPPING_REVIEWS")
        MAPPING_REVIEWS = json.loads(REVIEW_PATH.read_text(encoding="utf-8-sig"))
    MAPPING_RESULT = terminology_build_crosswalk(CLASSES, SOURCE_BUNDLES, MST_MAPPING_PROPOSALS, SNOMED_MAPPING_SNAPSHOT, MAPPING_REVIEWS, evidence_bundle=RAG_RUNTIME_EVIDENCE_PACKAGE)
finally:
    SNOMED_REUSE_STATS = {}
    if SNOMED_MAPPING_SNAPSHOT is not None:
        SNOMED_REUSE_STATS = {name: dict(counts) for name, counts in
                              SNOMED_MAPPING_SNAPSHOT["_mapping_run_stats"].items()}
        SNOMED_MAPPING_SNAPSHOT["_mapping_run_cache"].clear()
    del SNOMED_MAPPING_SNAPSHOT

CROSSWALK = MAPPING_RESULT["crosswalk"]
TERM_CATALOG = MAPPING_RESULT["terms"]
APPROVED_CLASS_SYNONYMS = MAPPING_RESULT["approved_class_synonyms"]
RAG_SYNONYM_AUDIT = build_source_synonym_audit(MAPPING_RESULT, RAG_RUNTIME_EVIDENCE_PACKAGE, SNOMED_SNAPSHOT)
APPROVED_CLASS_SYNONYMS = RAG_SYNONYM_AUDIT.loc[RAG_SYNONYM_AUDIT["rag_decision"].eq("accepted_source_synonym")].copy()
APPROVED_CLASS_SYNONYMS["review_method"] = "explicit_override_with_source_concept_validation" if REVIEW_PATH else "AI_assisted_RAG_concept_and_source_synonyms"
APPROVED_CLASS_SYNONYMS["human_reviewed"] = False
APPROVED_CLASS_SYNONYMS["rag_bundle_sha256"] = RAG_RESULT["summary"]["bundle_sha256"]
MAPPING_RESULT["approved_class_synonyms"] = APPROVED_CLASS_SYNONYMS
DISTINCT_CLASS_VOCABULARY = build_distinct_class_vocabulary(RAG_SYNONYM_AUDIT)
SNOMED_RETRIEVAL_AUDIT = MAPPING_RESULT["snomed_retrieval_audit"]
if REVIEW_PATH is None:
    rag_metadata = RAG_MAPPING_AUDIT[["mapping_id", "verdict", "reason_code", "evidence_ids_json", "bundle_sha256", "review_method", "human_reviewed"]].rename(columns={
        "verdict": "automatic_review_status", "reason_code": "automatic_review_reason", "bundle_sha256": "rag_bundle_sha256"
    })
    CROSSWALK = CROSSWALK.merge(rag_metadata, on="mapping_id", how="left", validate="one_to_one")
    MAPPING_RESULT["crosswalk"] = CROSSWALK
ATTRIBUTE_CATALOG = pd.concat([bundle["attributes"].assign(terminology=name) for name, bundle in SOURCE_BUNDLES.items()], ignore_index=True)
SOURCE_SYNONYMS = pd.concat([bundle["synonyms"].assign(terminology=name) for name, bundle in SOURCE_BUNDLES.items()], ignore_index=True)
if SNOMED_SNAPSHOT is not None:
    snomed_alias_rows = []
    for row in MAPPING_RESULT["snomed_subset"]["terms"].to_dict("records"):
        if row["term_type"] == "synonym":
            snomed_alias_rows.append({
                "terminology": "SNOMED_CT", "term_id": row["concept_id"], "synonym": row["term"],
                "synonym_key": mst_lookup_key(row["term"]), "language": TERMINOLOGY_CONFIG["snomed_language_code"],
                "status": "source_accepted_description_not_class_mapping_approval",
                "source_locator": "RF2 description " + row["description_id"] + "; language refset " + row["refset_id"],
                "source_url": "http://snomed.info/id/" + row["concept_id"],
                "source_sha256": SNOMED_SNAPSHOT["manifest"]["source_fingerprint"],
                "source_description_id": row["description_id"], "language_refset_id": row["refset_id"],
            })
    if snomed_alias_rows:
        SOURCE_SYNONYMS = pd.concat([SOURCE_SYNONYMS, pd.DataFrame(snomed_alias_rows)], ignore_index=True)
if "automatic_review_reason" not in CROSSWALK:
    CROSSWALK["automatic_review_reason"] = "explicit_review_override"
display(CROSSWALK[["normalized_label", "terminology", "target_id", "target_term", "target_fsn", "target_type", "relation", "mapping_status", "automatic_review_reason", "required_attribute_constraints_json"]])
print("Approved synonym provenance rows:", len(APPROVED_CLASS_SYNONYMS))
print("Distinct class/term/language entries:", len(DISTINCT_CLASS_VOCABULARY))
print("Distinct surface forms across classes:", APPROVED_CLASS_SYNONYMS["term"].map(rag_lexical_key).nunique())
print("Related/compositional mappings remain contextual; only exact approved mappings supply interchangeable class terms.")
display(RAG_MAPPING_AUDIT.loc[~RAG_MAPPING_AUDIT["verdict"].eq("accepted"), ["normalized_label", "terminology", "target_id", "target_term", "verdict", "reason_code", "reason"]])
if not SNOMED_RETRIEVAL_AUDIT.empty:
    display(SNOMED_RETRIEVAL_AUDIT)

if SNOMED_REUSE_STATS:
    print("SNOMED reuse within this mapping run:", SNOMED_REUSE_STATS)


GitHub read: /bogdanparvu18/msc-graduate-project/main/outputs/phase3/sector1/terminology/rag_evidence_manifest.json
GitHub read: /bogdanparvu18/msc-graduate-project/main/outputs/phase3/sector1/terminology/rag_evidence_manifest.json


,normalized_label,terminology,target_id,target_term,target_fsn,target_type,relation,mapping_status,automatic_review_reason,required_attribute_constraints_json
0,ampulla of vater,CEST,,,,,unmapped,source_required,source_required,[]
1,ampulla of vater,MST,MST3:Anatomy:R35:C2,Major papilla,,anatomical_site,related,approved,source_pinned_evidence_decision,[]
2,ampulla of vater,SNOMED_CT,67109009,Structure of ampulla of Vater,Structure of ampulla of Vater (body structure),body structure,exact,approved,source_concept_policy_match,[]
3,angiectasia,CEST,,,,,unmapped,source_required,source_required,[]
4,angiectasia,MST,MST3:Diagnoses:R77:C2,Angiectasia,,diagnosis,related,rejected,explicit_evidence_rejection,[]
5,angiectasia,MST,MST3:Luminal_findings:R67:C2,Angiectasia,,finding,exact,approved,source_pinned_evidence_decision,[]
6,angiectasia,SBCE_VASCULAR_DELPHI_2019,SBCE2019:angiectasia,Angiectasia,,finding,exact,approved,unique_evidence_supported_lexical_match,[]
7,angiectasia,SNOMED_CT,174637007,Angiectasia,Angiectasia (morphologic abnormality),morphologic abnormality,exact,approved,source_concept_policy_match,[]
8,blood fresh,CEST,,,,,unmapped,source_required,source_required,[]
9,blood fresh,MST,MST3:Luminal_findings:R34:C2,Blood,,finding,compositional,approved,source_pinned_evidence_decision,"[{""attribute"":""Type"",""source_locator"":""Luminal..."


Approved synonym provenance rows: 84
Distinct class/term/language entries: 29
Distinct surface forms across classes: 29
Related/compositional mappings remain contextual; only exact approved mappings supply interchangeable class terms.


,normalized_label,terminology,target_id,target_term,verdict,reason_code,reason
0,foreign body,CEST,,,source_required,source_required,Source unavailable.
4,erythema,CEST,,,source_required,source_required,Source unavailable.
9,pylorus,CEST,,,source_required,source_required,Source unavailable.
13,ileocecal valve,CEST,,,source_required,source_required,Source unavailable.
16,angiectasia,MST,MST3:Diagnoses:R77:C2,Angiectasia,rejected,explicit_evidence_rejection,Exclude duplicate diagnosis-role target from t...
17,angiectasia,CEST,,,source_required,source_required,Source unavailable.
19,ampulla of vater,CEST,,,source_required,source_required,Source unavailable.
22,normal clean mucosa,CEST,,,source_required,source_required,Source unavailable.
23,normal clean mucosa,MST,,,unmapped,unmapped,No external mapping candidate; retain the Phas...
24,normal clean mucosa,SNOMED_CT,,,unmapped,no_candidate_in_search_scope,native_composite_observation_no_reviewed_exact...


,class_id,query_term,query_source,query_evidence_ids_json,query_rule_id,lexical_key,terminology,candidate_concept_count,candidate_concept_ids_json,accepted_description_ids_json,active_raw_description_count,active_raw_description_ids_json,selected_refset_ids_json,reason_code,source_sha256,ontology_absence_proven
0,kvasir_capsule:077f739a329fd4b8ec8df87f,Disorder due to presence of foreign body,rag_snomed_rule_match_labels,"[""passage:ESGE_SBCE_2022"",""passage:KVASIR_2021...",foreign_body_disorder_vocabulary,disorder due to presence of foreign body,SNOMED_CT,1,"[""125670008""]","[""194158013""]",1,"[""194158013""]","[""19491000087109"",""20581000087109"",""2298510000...",matched_active_accepted_synonym,123a8b54fb3bb27e46e2a16d8b23af3538b4e3290f9823...,False
1,kvasir_capsule:077f739a329fd4b8ec8df87f,Disorder due to presence of foreign body,rag_snomed_rule_match_labels,"[""passage:ESGE_SBCE_2022"",""passage:KVASIR_2021...",foreign_body_object_vocabulary,disorder due to presence of foreign body,SNOMED_CT,1,"[""125670008""]","[""194158013""]",1,"[""194158013""]","[""19491000087109"",""20581000087109"",""2298510000...",matched_active_accepted_synonym,123a8b54fb3bb27e46e2a16d8b23af3538b4e3290f9823...,False
2,kvasir_capsule:077f739a329fd4b8ec8df87f,Disorder due to presence of foreign body,rag_snomed_search_alias,"[""passage:ESGE_SBCE_2022"",""passage:KVASIR_2021...",,disorder due to presence of foreign body,SNOMED_CT,1,"[""125670008""]","[""194158013""]",1,"[""194158013""]","[""19491000087109"",""20581000087109"",""2298510000...",matched_active_accepted_synonym,123a8b54fb3bb27e46e2a16d8b23af3538b4e3290f9823...,False
3,kvasir_capsule:077f739a329fd4b8ec8df87f,foreign body,phase2_normalized_label,[],,foreign body,SNOMED_CT,2,"[""125670008"",""19227008""]","[""32410013"",""473556018""]",2,"[""32410013"",""473556018""]","[""19491000087109"",""20581000087109"",""2298510000...",matched_active_accepted_synonym,123a8b54fb3bb27e46e2a16d8b23af3538b4e3290f9823...,False
4,kvasir_capsule:077f739a329fd4b8ec8df87f,foreign body,rag_class_lexical_label,"[""passage:ESGE_SBCE_2022"",""passage:KVASIR_2021...",,foreign body,SNOMED_CT,2,"[""125670008"",""19227008""]","[""32410013"",""473556018""]",2,"[""32410013"",""473556018""]","[""19491000087109"",""20581000087109"",""2298510000...",matched_active_accepted_synonym,123a8b54fb3bb27e46e2a16d8b23af3538b4e3290f9823...,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
123,kvasir_capsule:f5398876eb2cebb8704ca29b,Ulcer,rag_snomed_rule_accepted_fsn_labels,"[""passage:KVASIR_2021"",""passage:SNOMED_AU_ULCE...",ulcer_morphology_context,ulcer,SNOMED_CT,2,"[""429040005"",""56208002""]","[""2693981015"",""93469010""]",2,"[""2693981015"",""93469010""]","[""19491000087109"",""20581000087109"",""2298510000...",matched_active_accepted_synonym,123a8b54fb3bb27e46e2a16d8b23af3538b4e3290f9823...,False
124,kvasir_capsule:f5398876eb2cebb8704ca29b,Ulcer,rag_snomed_rule_accepted_fsn_labels,"[""passage:KVASIR_2021"",""passage:SNOMED_AU_ULCE...",ulcer_observed_clinical_finding,ulcer,SNOMED_CT,2,"[""429040005"",""56208002""]","[""2693981015"",""93469010""]",2,"[""2693981015"",""93469010""]","[""19491000087109"",""20581000087109"",""2298510000...",matched_active_accepted_synonym,123a8b54fb3bb27e46e2a16d8b23af3538b4e3290f9823...,False
125,kvasir_capsule:f5398876eb2cebb8704ca29b,Ulcer,rag_snomed_rule_match_labels,"[""passage:KVASIR_2021"",""passage:SNOMED_AU_ULCE...",ulcer_morphology_context,ulcer,SNOMED_CT,2,"[""429040005"",""56208002""]","[""2693981015"",""93469010""]",2,"[""2693981015"",""93469010""]","[""19491000087109"",""20581000087109"",""2298510000...",matched_active_accepted_synonym,123a8b54fb3bb27e46e2a16d8b23af3538b4e3290f9823...,False
126,kvasir_capsule:f5398876eb2cebb8704ca29b,Ulcer,rag_snomed_rule_match_labels,"[""passage:KVASIR_2021"",""passage:SNOMED_AU_ULCE...",ulcer_observed_clinical_finding,ulcer,SNOMED_CT,2,"[""429040005"",""56208002""]","[""2693981015"",""93469010""]",2,"[""2693981015"",""93

SNOMED reuse within this mapping run: {'candidates': {'hits': 1, 'misses': 1}, 'retrieval_audit': {'hits': 1, 'misses': 1}, 'ancestor_subset': {'hits': 1, 'misses': 1}}


## 9. Enrich evidence meanings, variants, task policies and S8 hints

`EVIDENCE_STATE_MEANINGS`, `VARIANT_SPECIFICATION`, `TASK_EVIDENCE_POLICY` and `S8_HINT_SPECIFICATION` now carry source-linked clinical concept slots and mapping identifiers. `TASK_SPECIFICATION` also receives these bindings. The original definitions are preserved under `base_definition` or alongside `terminology_contract`.

Technical facts remain technical: a box is geometric evidence, a crop is a representation, an unlabelled neighbour supplies a possible temporal reference. No clinical concept is assigned to that neighbour. A temporal finding slot instead requires a separate positively annotated frame. Attribute definitions specify what could be asked after observation-level evidence is available; they never populate patient/image attribute values.

`TASK_EVIDENCE_POLICY` becomes `task → evidence_state → object`, with the unchanged A/N/R value under `expected_policy_code`. Evidence meanings and S8 hints become structured objects. Future generation code must use these fields rather than the old tuple/string layout. The coverage matrix retains its original expected verdicts and adds terminology-binding references.

Policies also receive `semantic_compositions` linked to the actual imported source relations: MST term-plus-attribute constraints (including the two blood types), SNOMED `is-a` classification, and grouped SNOMED defining attributes when a release is supplied. Temporal/spatial operations remain the existing evidence operators. These are conditional declarations, not executed QA programs. The capability report checks the actual Phase 2 schema and annotation inventory; physical size, morphology and severity remain unavailable without independently verified observation-level values. Ontology-defined attributes never become image observations.


In [10]:
"""Attach terminology domains to the existing evidence contract without changing policy.

Bindings describe permitted schema roles, not observed patient/image facts. Missing
terminology sources and unreviewed mappings remain explicit unresolved states.
"""


_CONTRACT_NAMES = (
    "TASK_SPECIFICATION", "EVIDENCE_STATES", "EVIDENCE_STATE_MEANINGS",
    "VARIANT_SPECIFICATION", "TASK_EVIDENCE_POLICY", "S8_HINT_SPECIFICATION",
)
_TERMINOLOGIES = ("MST", "CEST", "SNOMED_CT")


def _cb_clean(value):
    if isinstance(value, float) and math.isnan(value):
        return None
    if isinstance(value, dict):
        return {str(k): _cb_clean(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [_cb_clean(v) for v in value]
    return value


def _cb_json(value):
    return json.dumps(_cb_clean(value), ensure_ascii=False, sort_keys=True,
                      separators=(",", ":"), allow_nan=False)


def _cb_hash(value):
    return hashlib.sha256(_cb_json(value).encode("utf-8")).hexdigest()


def _cb_records(value):
    return _cb_clean(value.to_dict("records") if hasattr(value, "to_dict") else list(value))


def _cb_constraints(value):
    if value is None or value == "":
        return []
    if isinstance(value, str):
        value = json.loads(value)
    if not isinstance(value, (dict, list)):
        raise ValueError("Mapping attribute constraints must be a JSON object or list")
    return value


def enrich_terminology_contract(original_contracts, classes, crosswalk, attributes, source_status):
    """Return enriched_contracts, binding_rows, and audit; never mutate inputs.

    original_contracts contains the six uppercase names listed above. Inputs may
    be record lists or pandas DataFrames. Only explicit reviewed `approved`
    mappings are labelled approved. Even approved exact mappings do not establish
    that a concept or attribute is present in an image.
    """
    missing = set(_CONTRACT_NAMES) - set(original_contracts)
    if missing:
        raise ValueError("Missing original contract definitions: " + ", ".join(sorted(missing)))
    original = copy.deepcopy({k: original_contracts[k] for k in _CONTRACT_NAMES})
    base_hash = _cb_hash(original)
    states = list(original["EVIDENCE_STATES"])
    if len(states) != len(set(states)):
        raise ValueError("Evidence-state names must be unique")
    class_rows, map_rows, attribute_rows = map(_cb_records, (classes, crosswalk, attributes))
    class_by_id = {r["dataset_class_id"]: r for r in class_rows}
    if len(class_by_id) != len(class_rows) or not class_rows:
        raise ValueError("Class identifiers must be nonempty and unique")
    for r in class_rows:
        if not r.get("dataset_class_id") or not r.get("normalized_label"):
            raise ValueError("Each class needs dataset_class_id and normalized_label")
    maps_by_class = {cid: [] for cid in class_by_id}
    for row in map_rows:
        cid = row.get("dataset_class_id")
        if cid not in class_by_id:
            raise ValueError("Crosswalk references an unknown dataset class: " + str(cid))
        if row.get("normalized_label") != class_by_id[cid]["normalized_label"]:
            raise ValueError("Crosswalk label differs from its dataset class identity")
        if row.get("terminology") not in set(_TERMINOLOGIES) | set(source_status):
            raise ValueError("Unsupported terminology name: " + str(row.get("terminology")))
        if row.get("mapping_status") not in {"candidate_review_required", "approved", "unmapped", "source_required", "rejected"}:
            raise ValueError("Unknown mapping status: " + str(row.get("mapping_status")))
        if row.get("mapping_status") == "approved" and (
            not row.get("target_id") or not row.get("mapping_id") or not row.get("source_sha256")
        ):
            raise ValueError("An approved mapping must retain target, mapping and source identities")
        _cb_constraints(row.get("required_attribute_constraints_json"))
        maps_by_class[cid].append(row)
    for cid in maps_by_class:
        present = {r["terminology"] for r in maps_by_class[cid]}
        for terminology in _TERMINOLOGIES:
            if terminology not in present:
                maps_by_class[cid].append({
                    "dataset_class_id": cid,
                    "normalized_label": class_by_id[cid]["normalized_label"],
                    "terminology": terminology,
                    "target_id": None, "target_term": None, "relation": None,
                    "mapping_status": "source_required" if source_status.get(terminology) != "imported" else "unmapped",
                    "source_sha256": None, "mapping_id": None,
                    "required_attribute_constraints_json": "[]",
                })
        maps_by_class[cid].sort(key=_cb_json)
    attrs_by_term = {}
    for row in attribute_rows:
        if row.get("term_id"):
            attrs_by_term.setdefault((row.get("terminology"), row["term_id"]), []).append(row)
    for term_id in attrs_by_term:
        attrs_by_term[term_id].sort(key=_cb_json)

    # Each slot has an explicit observation source and keeps technical relations
    # outside clinical ontologies. These are prerequisites, never supplied facts.
    slots = {
        "observed_finding": {
            "evidence_fields": ["annotation_id", "ground_truth_ref", "source_label", "image_relpath"],
            "scope": "verified_annotation_attached_to_displayed_source",
        },
        "indicated_region_finding": {
            "evidence_fields": ["annotation_id", "ground_truth_ref", "source_label", "bbox_xyxy", "image_relpath"],
            "scope": "same_class_annotation_with_valid_geometry_in_displayed_coordinates",
        },
        "observed_finding_set": {
            "evidence_fields": ["annotation_ids", "ground_truth_refs", "verified_labels", "image_relpath"],
            "scope": "each_set_member_has_its_own_positive_annotation; no_exhaustive_absence_claim",
        },
        "normal_reference_finding": {
            "evidence_fields": ["comparison_reference_image_relpath", "comparison_reference_verified_labels", "annotation_id"],
            "scope": "only_labels_explicitly_authorized_by_verified_normal_reference_contract; no_inferred_normality",
        },
        "same_finding_positive_anchors": {
            "evidence_fields": ["annotation_ids", "source_label", "video_key", "frame_index"],
            "scope": "at_least_two_distinct_positive_same_finding_frames_in_one_video; no_continuity_inference",
        },
        "positive_target_finding": {
            "evidence_fields": ["annotation_id", "ground_truth_ref", "source_label", "video_key", "frame_index", "reference_frame_index"],
            "scope": "positive_target_on_distinct_same_video_frame_in_requested_direction; reference_may_be_unlabelled",
        },
        "unresolved_finding_mapping": {
            "evidence_fields": ["source_label", "annotation_id", "ground_truth_ref"],
            "scope": "review_only_until_reference_and_mapping_conflicts_are_resolved",
        },
    }
    task_slots = {
        "finding_identification": ["observed_finding"],
        "finding_category": ["observed_finding"],
        "evidence_frame_localization": ["observed_finding"],
        "optional_spatial_localization": ["indicated_region_finding"],
        "verified_reference_comparison": ["normal_reference_finding"],
        "observed_anchor_order": ["same_finding_positive_anchors"],
        "temporal_relative_evidence_localization": ["positive_target_finding"],
        "clinical_visual_quality": [],
        "guarded_clinical_sufficiency": [],
    }
    state_slots = {
        "label_without_box": ["observed_finding"],
        "label_with_box": ["observed_finding", "indicated_region_finding"],
        "multiple_labels": ["observed_finding_set"],
        "missing_category": ["observed_finding"],
        "unlabelled_neighbor": [],
        "explicit_normal_reference": ["normal_reference_finding"],
        "multiple_verified_anchors": ["same_finding_positive_anchors"],
        "annotation_assisted_input": ["indicated_region_finding"],
        "unmapped_or_conflicting_reference": ["unresolved_finding_mapping"],
    }
    if set(state_slots) != set(states):
        raise ValueError("New or missing evidence state requires an explicit semantic binding policy")
    tasks = {t["task"]: t for t in original["TASK_SPECIFICATION"]}
    if set(task_slots) != set(tasks):
        raise ValueError("New or missing task requires an explicit semantic binding policy")
    bindings = []

    def attach(owner_kind, owner_key, role_names):
        ids = []
        for role in role_names:
            slot = slots[role]
            for cid in sorted(class_by_id):
                for mapping in maps_by_class[cid]:
                    terminology = mapping["terminology"]
                    constraints = _cb_constraints(mapping.get("required_attribute_constraints_json"))
                    attr_defs = []
                    matching_attrs = attrs_by_term.get((terminology, mapping.get("target_id")), attrs_by_term.get((None, mapping.get("target_id")), []))
                    for a in matching_attrs:
                        attr_defs.append({
                            "attribute": a.get("attribute"), "allowed_value": a.get("value"),
                            "value_kind": a.get("value_kind"), "source_locator": a.get("source_locator"),
                            "applicability": "schema_only; mapping_and_source_attribute_constraints_must_hold",
                            "observed_value": None,
                            "requires_verified_observation": True,
                            "observation_prerequisite": "same_observation_annotation_id + attribute_name + value + units_if_applicable + provenance",
                        })
                    approved = mapping.get("mapping_status") == "approved" and source_status.get(terminology) == "imported"
                    relation = mapping.get("relation")
                    row = {
                        "owner_kind": owner_kind, "owner_key": owner_key, "clinical_slot": role,
                        "dataset_class_id": cid, "normalized_label": class_by_id[cid]["normalized_label"],
                        "terminology": terminology, "target_id": mapping.get("target_id"),
                        "target_term": mapping.get("target_term"), "relation": relation,
                        "mapping_status": mapping["mapping_status"], "mapping_id": mapping.get("mapping_id"),
                        "source_status": source_status.get(terminology, "source_required"),
                        "source_sha256": mapping.get("source_sha256"),
                        "semantic_scope": slot["scope"],
                        "required_evidence_fields_json": _cb_json(slot["evidence_fields"]),
                        "required_attribute_constraints_json": _cb_json(constraints),
                        "attribute_schema_json": _cb_json(attr_defs),
                        "attribute_schema_status": "approved_mapping_observations_still_required" if approved else "mapping_review_or_source_required",
                        "exact_substitution_mapping_approved": bool(approved and relation == "exact"),
                        "unconditional_lexical_substitution": bool(approved and relation == "exact" and not constraints),
                        "clinical_instance_binding_status": "not_evaluated_requires_observation_binding",
                        "observed_attribute_values_available": False,
                        "clinical_truth_established_by_terminology": False,
                    }
                    row["binding_id"] = "contract_binding:" + _cb_hash(row)
                    ids.append(row["binding_id"])
                    bindings.append(row)
        return ids

    def wrap(owner_kind, owner_key, base, role_names, technical):
        return {
            "base_definition": copy.deepcopy(base),
            "technical_scope": technical,
            "semantic_scope": "conditional_schema_binding_not_patient_observation",
            "clinical_slots": list(role_names),
            "terminology_binding_ids": attach(owner_kind, owner_key, role_names),
            "terminology_source_status": copy.deepcopy(source_status),
            "qa_generation_enabled": False,
        }

    enriched = {"EVIDENCE_STATES": copy.deepcopy(states), "TASK_SPECIFICATION": []}
    for task in original["TASK_SPECIFICATION"]:
        item = copy.deepcopy(task)
        item["terminology_contract"] = wrap("task", task["task"], task, task_slots[task["task"]],
            "original_task_reference_and_required_flags_unchanged")
        if task["task"] == "finding_category":
            item["terminology_contract"]["category_policy"] = "dataset clinical_group remains source-backed; terminology category is separate and cannot repair a missing source category"
        if task["task"] in {"clinical_visual_quality", "guarded_clinical_sufficiency"}:
            item["terminology_contract"]["clinical_review_required"] = True
        enriched["TASK_SPECIFICATION"].append(item)
    enriched["EVIDENCE_STATE_MEANINGS"] = {}
    for state in states:
        item = wrap("evidence_state", state, original["EVIDENCE_STATE_MEANINGS"][state], state_slots[state],
                    "evidence_availability_and_provenance_state; not_a_SNOMED_clinical_concept")
        item["positive_finding_asserted_by_state"] = False
        if state == "unlabelled_neighbor":
            item.update(unknown_is_negative=False, unknown_is_normal=False,
                        temporal_reference_allowed=True,
                        reference_fields=["video_key", "frame_index"],
                        positive_target_requires_different_verified_frame=True)
        if state == "missing_category":
            item["terminology_does_not_fill_source_clinical_group"] = True
        if state == "annotation_assisted_input":
            item["assistance_declaration_required"] = True
        enriched["EVIDENCE_STATE_MEANINGS"][state] = item
    enriched["VARIANT_SPECIFICATION"] = {}
    for name, spec in original["VARIANT_SPECIFICATION"].items():
        role_names = ["indicated_region_finding"] if spec["requires_box"] else ["observed_finding"]
        if spec["family"] == "temporal":
            role_names = ["same_finding_positive_anchors", "positive_target_finding"]
        elif spec["family"] == "comparison":
            role_names = ["observed_finding", "normal_reference_finding"]
        elif spec["family"] == "combined":
            role_names = ["observed_finding", "indicated_region_finding", "positive_target_finding"]
        item = copy.deepcopy(spec)
        item["terminology_contract"] = wrap("variant", name, spec, role_names,
            "presentation_geometry_and_assistance_are_technical_predicates; variant_does_not_establish_finding_presence")
        item["terminology_contract"]["clinical_quality_target_available"] = False
        item["terminology_contract"]["progression_inference_allowed"] = False
        enriched["VARIANT_SPECIFICATION"][name] = item
    enriched["TASK_EVIDENCE_POLICY"] = {}
    for task_name, codes in original["TASK_EVIDENCE_POLICY"].items():
        if task_name not in tasks or len(codes) != len(states) or set(codes) - {"A", "N", "R"}:
            raise ValueError("Invalid base task/evidence policy")
        enriched["TASK_EVIDENCE_POLICY"][task_name] = {}
        for state, code in zip(states, codes):
            role_names = [] if code == "N" else task_slots[task_name]
            # Unlabelled reference has no medical assertion. Only the distinct
            # positively annotated target of a relative-location task is bound.
            if state == "unlabelled_neighbor":
                role_names = ["positive_target_finding"] if task_name == "temporal_relative_evidence_localization" and code == "A" else []
            item = wrap("task_evidence_policy", task_name + "/" + state, code, role_names,
                        "A_N_R_is_original_evidence_policy; terminology_never_upgrades_eligibility")
            item["expected_policy_code"] = code
            item["policy_status"] = {"A": "applicable_only_if_original_guards_hold", "N": "not_applicable", "R": "review_required"}[code]
            enriched["TASK_EVIDENCE_POLICY"][task_name][state] = item
    enriched["S8_HINT_SPECIFICATION"] = {}
    for name, value in original["S8_HINT_SPECIFICATION"].items():
        reqs, note = value
        roles = []
        if "REQ_FINDING" in reqs:
            roles.append("indicated_region_finding" if "region" in name else "observed_finding")
        if "REQ_TEMPORAL" in reqs:
            roles.append("positive_target_finding")
        if "REQ_COMPARISON" in reqs:
            roles.append("normal_reference_finding")
        item = wrap("s8_hint", name, value, roles,
                    "preparation_compatibility_hint; not_a_validated_QA_task_or_observed_attribute")
        item.update(requirements=copy.deepcopy(reqs), original_note=note,
                    qa_eligibility_established=False,
                    attribute_values_require_independent_annotations=True)
        enriched["S8_HINT_SPECIFICATION"][name] = item
    if _cb_hash({k: original_contracts[k] for k in _CONTRACT_NAMES}) != base_hash:
        raise AssertionError("Original contract mutated during enrichment")
    # Sorting and hashing make annotations reproducible for a fixed source bundle.
    bindings.sort(key=lambda r: r["binding_id"])
    audit = {
        "base_contract_sha256": base_hash,
        "base_contract_unchanged": True,
        "base_contract": _cb_clean(original),
        "binding_count": len(bindings),
        "class_count": len(class_rows),
        "bindings_sha256": _cb_hash(bindings),
        "qa_generated": 0,
        "clinical_observations_created": 0,
        "mrcm_validation_executed": False,
        "terminology_source_status": copy.deepcopy(source_status),
    }
    return {"enriched_contracts": enriched, "binding_rows": bindings, "audit": audit}


def matrix_from_enriched_contract(enriched_contracts, base_matrix):
    """Annotate an already-computed original coverage matrix; preserve verdicts.

    Pass COVERAGE_CONTRACT computed before replacing the original dictionaries.
    This avoids calling a helper whose globals now have a different schema and
    preserves every original variant-specific exception exactly.
    """
    rows = _cb_records(base_matrix)
    task_index = {t["task"]: t for t in enriched_contracts["TASK_SPECIFICATION"]}
    result = []
    for row in rows:
        task, state, variant = row["task"], row["evidence_state"], row["variant"]
        if task not in task_index or variant not in enriched_contracts["VARIANT_SPECIFICATION"]:
            raise ValueError("Coverage matrix references an unknown task or variant")
        policy = enriched_contracts["TASK_EVIDENCE_POLICY"][task][state]
        out = copy.deepcopy(row)
        out.update(
            original_policy_code=policy["expected_policy_code"],
            terminology_binding_count=len(policy["terminology_binding_ids"]),
            terminology_binding_ids_json=_cb_json(policy["terminology_binding_ids"]),
            terminology_source_status_json=_cb_json(policy["terminology_source_status"]),
            terminology_changes_expected_status=False,
            observed_status="not_evaluated",
            qa_generation_executed=False,
        )
        result.append(out)
    return result

"""Source-derived semantic compositions for a terminology-stage evidence contract.

This module declares typed compositions and their evidence prerequisites. It does
not generate QA, change A/N/R policies, infer new image annotations, or execute a
clinical reasoning engine. Mapping approval is distinct from observation support.
"""


_SC_IS_A = None  # Resolved from each imported Snapshot's metadata registry.
_SC_INFERRED = None


def _sc_clean(value):
    if isinstance(value, float) and math.isnan(value):
        return None
    if isinstance(value, dict):
        return {str(k): _sc_clean(v) for k, v in value.items()}
    if isinstance(value, (tuple, list)):
        return [_sc_clean(v) for v in value]
    return value


def _sc_json(value):
    return json.dumps(_sc_clean(value), sort_keys=True, ensure_ascii=False,
                      separators=(",", ":"), allow_nan=False)


def _sc_hash(value):
    return hashlib.sha256(_sc_json(value).encode()).hexdigest()


def _sc_records(value):
    if value is None:
        return []
    return _sc_clean(value.to_dict("records") if hasattr(value, "to_dict") else list(value))


def _sc_constraints(value):
    if value is None or value == "":
        return []
    if isinstance(value, str):
        value = json.loads(value)
    if isinstance(value, dict):
        value = [value] if "attribute" in value else [dict(attribute=k, value=v) for k, v in value.items()]
    if not isinstance(value, list) or any(not isinstance(r, dict) or not r.get("attribute") or "value" not in r for r in value):
        raise ValueError("Qualified mappings require explicit attribute/value constraints")
    return value


def _sc_policy_codes(contract):
    return {task: {state: p["expected_policy_code"] for state, p in states.items()}
            for task, states in contract["TASK_EVIDENCE_POLICY"].items()}


def build_semantic_compositions(enriched_contract, crosswalk, attributes, observations,
                                source_schema, snomed_subset=None):
    """Return (contract_copy, compositions, evidence_capabilities, audit).

    Tables can be DataFrames or record lists. source_schema is CORPUS.schemas.
    snomed_subset accepts concepts, terms, relationships, is_a_closure,
    seed_attribute_relationships, and an optional source manifest. Relationship
    groups, types, source/destination IDs and effective times remain attached.

    Future independently verified clinical attributes must be present in a
    declared `verified_attributes`/`verified_attributes_json` source field and
    each value must carry matching annotation_id, verification_status='verified'
    and evidence_ref. An ontology's allowed values never satisfy that contract.
    """
    original_policy = _sc_policy_codes(enriched_contract)
    original_hash = _sc_hash(original_policy)
    result = copy.deepcopy(enriched_contract)
    maps, attrs, obs = map(_sc_records, (crosswalk, attributes, observations))
    schema_fields = set()
    for schema in source_schema.values():
        schema_fields.update(schema.get("columns", []))
    logical_fields = schema_fields | {x[:-5] for x in schema_fields if x.endswith("_json")}
    by_class, attr_by_term = defaultdict(list), defaultdict(list)
    for observation in obs:
        if not observation.get("dataset_class_id") or not observation.get("annotation_id"):
            raise ValueError("Observations require source class and annotation identity")
        by_class[observation["dataset_class_id"]].append(observation)
    for a in attrs:
        if a.get("term_id") and a.get("attribute"):
            attr_by_term[(a.get("terminology") or "MST", a["term_id"])].append(a)
    map_by_class = defaultdict(list)
    for mapping in maps:
        cid = mapping.get("dataset_class_id")
        if cid not in by_class:
            raise ValueError("Crosswalk class has no annotation evidence: " + str(cid))
        if mapping.get("normalized_label") not in {r.get("normalized_label") for r in by_class[cid]}:
            raise ValueError("Crosswalk and annotation class labels differ")
        map_by_class[cid].append(mapping)
    capabilities = []
    capability_index = {}

    def cap(cid, name, count, status, prerequisite, detail=None):
        row = dict(dataset_class_id=cid, normalized_label=by_class[cid][0]["normalized_label"],
                   capability=name, supported_observation_count=int(count), status=status,
                   prerequisite=prerequisite, detail_json=_sc_json(detail or {}),
                   establishes_model_input_coverage=False,
                   observation_source="verified_Phase_2_annotation_inventory")
        row["capability_id"] = "evidence_capability:" + _sc_hash(row)
        capabilities.append(row); capability_index[(cid, name)] = row
        return row

    verified_attribute_index = defaultdict(list)
    for cid, rows in sorted(by_class.items()):
        has_label_schema = {"source_label", "annotation_id"}.issubset(logical_fields)
        labelled = [r for r in rows if r.get("source_label") and r.get("ground_truth_ref")]
        cap(cid, "positive_finding_annotation", len(labelled),
            "available" if has_label_schema and labelled else "source_evidence_required",
            "source_label + annotation_id + ground_truth_ref on the same verified observation")
        categories = {r.get("dataset_category") for r in rows if r.get("dataset_category")}
        category_ok = "clinical_group" in logical_fields and len(categories) == 1 and all(r.get("dataset_category") for r in rows)
        cap(cid, "original_dataset_category", len(rows) if category_ok else 0,
            "available" if category_ok else "missing_or_conflicting_source_category",
            "unambiguous original clinical_group; terminology-derived groups cannot fill this field",
            {"dataset_categories": sorted(categories)})
        bbox_rows = [r for r in rows if r.get("has_bbox") is True]
        bbox_ok = "bbox_xyxy" in logical_fields and bool(bbox_rows)
        cap(cid, "source_geometry_reference", len(bbox_rows) if bbox_ok else 0,
            "raw_reference_available" if bbox_ok else "source_geometry_required",
            "load same-class raw bbox from the verified source table and validate displayed coordinates; has_bbox alone is not coordinates")
        frames_by_video = defaultdict(set)
        if {"video_key", "frame_index"}.issubset(logical_fields):
            for r in labelled:
                frame = r.get("frame_index")
                if r.get("video_key") and isinstance(frame, (int, float)) and not isinstance(frame, bool) and math.isfinite(frame) and frame >= 0 and frame == int(frame):
                    frames_by_video[r["video_key"]].add(int(frame))
        anchor_count = sum(len(v) for v in frames_by_video.values() if len(v) >= 2)
        cap(cid, "same_video_positive_anchors", anchor_count,
            "source_anchors_available" if anchor_count else "multiple_same_video_anchors_required",
            "at least two verified distinct same-finding frames in one video; actual displayed context still requires checking")
        if "verified_attributes" in logical_fields:
            for r in rows:
                entries = r.get("verified_attributes") or []
                if isinstance(entries, str):
                    entries = json.loads(entries)
                if not isinstance(entries, list):
                    raise ValueError("verified_attributes must be a list of observation records")
                for a in entries:
                    if (isinstance(a, dict) and a.get("attribute") and a.get("value") is not None
                        and a.get("annotation_id") == r["annotation_id"]
                        and a.get("verification_status") == "verified" and a.get("evidence_ref")):
                        verified_attribute_index[(cid, a["attribute"])].append(a)
        for name in ("shape", "size_mm", "severity", "grade"):
            matched = [a for (class_id, attr), entries in verified_attribute_index.items()
                       if class_id == cid and attr.casefold() == name.casefold() for a in entries]
            cap(cid, "verified_clinical_attribute:" + name, len(matched),
                "available" if matched else "not_available_in_verified_observations",
                "explicitly verified same-observation attribute value, units when applicable, and evidence reference",
                {"field_declared": name in logical_fields,
                 "verified_attribute_container_declared": "verified_attributes" in logical_fields,
                 "ontology_value_is_not_observation": True})
    subset = snomed_subset or {}
    relationships = _sc_records(subset.get("relationships"))
    seed_attributes = _sc_records(subset.get("seed_attribute_relationships"))
    metadata_ids = subset.get("manifest", {}).get("metadata_ids", {})
    _SC_IS_A = metadata_ids.get("SNOMED_IS_A")
    _SC_INFERRED = metadata_ids.get("SNOMED_INFERRED")
    if relationships or seed_attributes:
        if not all(isinstance(value, str) and value.isdigit()
                   for value in (_SC_IS_A, _SC_INFERRED)):
            raise ValueError("SNOMED compositions require metadata IDs discovered from the same RF2 Snapshot")
    concept_ids = {r["id"] for r in _sc_records(subset.get("concepts")) if r.get("id")}
    relations_by_source = defaultdict(list)
    seen_relation_ids = set()
    for edge in relationships + seed_attributes:
        if edge.get("active", "1") not in {"1", 1, True}:
            continue
        if edge.get("characteristicTypeId") != _SC_INFERRED:
            raise ValueError("Semantic compositions require imported inferred SNOMED relationships")
        if not all(edge.get(k) for k in ("id", "sourceId", "destinationId", "typeId")):
            raise ValueError("SNOMED relationship is missing its source identity")
        if edge["sourceId"] not in concept_ids or edge["destinationId"] not in concept_ids or (edge["typeId"] != _SC_IS_A and edge["typeId"] not in concept_ids):
            raise ValueError("SNOMED composition relationship lacks retained concept dependencies")
        if edge["id"] in seen_relation_ids:
            continue
        seen_relation_ids.add(edge["id"])
        relations_by_source[edge["sourceId"]].append(edge)
    for edges in relations_by_source.values():
        edges.sort(key=lambda r: str(r["id"]))
    compositions = []
    relevant_tasks = {"finding_identification", "finding_category", "evidence_frame_localization",
                      "optional_spatial_localization", "observed_anchor_order",
                      "temporal_relative_evidence_localization"}
    task_ops = {
        "finding_identification": ["project_verified_finding"],
        "finding_category": ["retain_original_dataset_category_separately"],
        "evidence_frame_localization": ["project_displayed_positive_source_ids"],
        "optional_spatial_localization": ["join_same_class_raw_geometry", "check_displayed_coordinate_system"],
        "observed_anchor_order": ["join_positive_same_finding_anchors", "filter_same_video", "compare_observed_frame_indices"],
        "temporal_relative_evidence_localization": ["join_distinct_reference_frame", "filter_same_video", "compare_requested_before_after", "project_positive_target_source_ids"],
    }

    def add(mapping, kind, semantic_ops, source_refs, prerequisites, evidence_status,
            derived_attrs=None, allowed=None, target_tasks=None, scope="terminology_schema_and_verified_source_labels"):
        cid = mapping["dataset_class_id"]
        approved = mapping.get("mapping_status") == "approved"
        mapping_status = mapping.get("mapping_status", "unmapped")
        status = "declaration_ready_for_later_instantiation" if approved and evidence_status == "source_supported" else (
            "mapping_review_required" if mapping_status == "candidate_review_required" else
            "mapping_rejected" if mapping_status == "rejected" else
            "terminology_source_required" if mapping_status == "source_required" else
            "evidence_required" if approved else "mapping_required")
        for task in sorted(target_tasks or relevant_tasks):
            if task not in result["TASK_EVIDENCE_POLICY"]:
                continue
            row = {
                "dataset_class_id": cid, "normalized_label": mapping["normalized_label"],
                "task": task, "composition_kind": kind, "terminology": mapping["terminology"],
                "target_id": mapping.get("target_id"), "target_term": mapping.get("target_term"),
                "mapping_id": mapping.get("mapping_id"), "mapping_status": mapping_status,
                "mapping_relation": mapping.get("relation"), "source_sha256": mapping.get("source_sha256"),
                "semantic_scope": scope,
                "operator_composition_json": _sc_json(["read_verified_annotation_identity", "bind_source_terminology_mapping"] + semantic_ops + task_ops[task]),
                "source_relations_json": _sc_json(source_refs),
                "required_evidence_json": _sc_json(prerequisites),
                "derived_attribute_constraints_json": _sc_json(derived_attrs or []),
                "allowed_attribute_values_json": _sc_json(allowed or []),
                "evidence_status": evidence_status,
                "declaration_status": status,
                "execution_status": "not_executed",
                "executable_now": False,
                "creates_new_annotation": False,
                "replaces_original_clinical_group": False,
                "establishes_qa_eligibility": False,
                "model_input_visibility_checked": False,
            }
            if task == "finding_category":
                row["target_scope"] = "auxiliary_terminology_category_only; cannot_replace_original_source_backed_category_task"
            else:
                row["target_scope"] = "original_task_evidence_guards_still_required"
            row["composition_id"] = "semantic_composition:" + _sc_hash(row)
            compositions.append(row)

    for mapping in sorted(maps, key=_sc_json):
        cid, term = mapping["dataset_class_id"], mapping.get("target_id")
        constraints = _sc_constraints(mapping.get("required_attribute_constraints_json"))
        if mapping.get("mapping_status") in {"source_required", "unmapped"} or not term:
            if mapping.get("terminology") == "SNOMED_CT":
                add(mapping, "snomed_hierarchy_pending_source_or_mapping", [], [],
                    ["import versioned SNOMED concepts and inferred relationships", "approve source-class mapping"],
                    "terminology_source_or_mapping_required", target_tasks={"finding_category"},
                    scope="unresolved_schema_placeholder; no_medical_relationship_asserted")
            continue
        available_defs = attr_by_term.get((mapping.get("terminology"), term), [])
        if constraints:
            matched_defs, unmatched = [], []
            for c in constraints:
                found = [a for a in available_defs if a.get("attribute") == c["attribute"] and a.get("value") == c["value"]]
                if found:
                    matched_defs.extend(found)
                else:
                    unmatched.append(c)
            source_refs = [dict(term_id=a.get("term_id"), attribute=a.get("attribute"), value=a.get("value"),
                                source_locator=a.get("source_locator"), source_sha256=a.get("source_sha256")) for a in matched_defs]
            source_supported = not unmatched and capability_index[(cid, "positive_finding_annotation")]["status"] == "available"
            add(mapping, "qualified_source_label", ["compose_source_term_with_reviewed_label_qualifiers"],
                source_refs, ["approved reviewed mapping explicitly equates this source label to the qualified expression",
                              "original source_label and annotation identity are retained",
                              "every qualifier matches an imported term/attribute/value relation",
                              "derived qualifiers are labelled as mapping derivations, not independent annotations"],
                "source_supported" if source_supported else "attribute_relation_source_required",
                derived_attrs=constraints)
        by_attribute = defaultdict(list)
        for a in available_defs:
            by_attribute[a["attribute"]].append(a)
        for attribute, definitions in sorted(by_attribute.items()):
            observed = verified_attribute_index.get((cid, attribute), [])
            needs_unit = any(a.get("value_kind") == "measurement_placeholder" for a in definitions)
            enumerated = {str(a.get("value")).strip().casefold() for a in definitions if a.get("value_kind") == "enumerated"}
            expected_units = {unit for a in definitions for unit in re.findall(r"\b(mm|cm)\b", str(a.get("value", "")).casefold())}
            usable = []
            for a in observed:
                value = a.get("value")
                if needs_unit:
                    if isinstance(value, bool) or not isinstance(value, (int, float)) or not math.isfinite(value) or not a.get("unit"):
                        continue
                    if expected_units and str(a["unit"]).strip().casefold() not in expected_units:
                        continue
                elif enumerated and str(value).strip().casefold() not in enumerated:
                    continue
                elif not enumerated and any(d.get("value_kind") == "classification_reference" for d in definitions):
                    # The referenced classification has not been imported here.
                    continue
                usable.append(a)
            # A source-label qualifier may support qualification, but is not an
            # independently recorded measurement/shape/severity observation.
            unit_groups = defaultdict(set)
            for a in usable:
                unit_groups[str(a.get("unit", "")).strip().casefold()].add(a["annotation_id"])
            comparison_supported = any(len(ids) >= 2 for ids in unit_groups.values())
            capability_name = "attribute_comparison:" + attribute
            cap(cid, capability_name, len(usable),
                "two_or_more_verified_values_available" if comparison_supported else "verified_attribute_values_required",
                "two distinct verified observations with the same attribute and compatible units; do not substitute ontology definitions",
                {"attribute": attribute, "unit_required": needs_unit,
                 "source_attribute_definition_count": len(definitions)})
            source_refs = [dict(term_id=a.get("term_id"), attribute=a.get("attribute"), value=a.get("value"),
                                value_kind=a.get("value_kind"), source_locator=a.get("source_locator"),
                                source_sha256=a.get("source_sha256")) for a in definitions]
            add(mapping, "verified_attribute_comparison", ["join_independently_verified_attribute_observations", "compare_recorded_attribute_values"],
                source_refs, ["two distinct observations carry this attribute with independently verified values",
                              "attribute observation has matching annotation_id, verification_status and evidence_ref",
                              "measurement units must be recorded and compatible when applicable",
                              "attribute values must be within imported schema and any mapping constraints",
                              "this is a proposed extension; existing task operator must be explicitly versioned before execution"],
                "source_supported" if comparison_supported else "verified_attribute_values_required",
                allowed=source_refs,
                target_tasks={"finding_identification", "observed_anchor_order", "temporal_relative_evidence_localization"})
        if mapping.get("terminology") == "SNOMED_CT":
            actual_edges = relations_by_source.get(term, [])
            if not actual_edges:
                add(mapping, "snomed_relations_pending_import", [], [],
                    ["retain actual source-linked inferred edges for the mapped concept"], "source_relationships_required",
                    target_tasks={"finding_category"}, scope="unresolved_relationship_import; no_absence_or_incompatibility_inferred")
            for edge in actual_edges:
                # Keep the entire imported relationship row, including role group
                # and module/version identifiers. Never collapse attribute groups.
                is_a = edge["typeId"] == _SC_IS_A
                group_edges = [edge] if is_a else [r for r in actual_edges if r["typeId"] != _SC_IS_A and str(r.get("relationshipGroup", "0")) == str(edge.get("relationshipGroup", "0"))]
                add(mapping, "snomed_is_a_classification" if is_a else "snomed_concept_attribute_relation_group",
                    ["follow_imported_inferred_is_a"] if is_a else ["retain_imported_concept_definition_relationship_group"],
                    group_edges, ["approved source-class mapping", "retained active concept dependencies",
                             "SNOMED classification remains separate from original clinical_group",
                             "concept definition is not a new observation of anatomy, morphology or severity in this image"],
                    "source_supported", target_tasks={"finding_category"},
                    scope="concept_hierarchy_only" if is_a else "concept_definition_only; not_an_observed_image_attribute")
    # Deduplicate identical source declarations, not distinct annotation contexts.
    unique = {row["composition_id"]: row for row in compositions}
    compositions = [unique[k] for k in sorted(unique)]
    by_task = defaultdict(list)
    for row in compositions:
        by_task[row["task"]].append(row)
    for task, states in result["TASK_EVIDENCE_POLICY"].items():
        for state, policy in states.items():
            code = policy["expected_policy_code"]
            references = []
            if code != "N":
                for row in by_task.get(task, []):
                    if state == "unlabelled_neighbor" and task != "temporal_relative_evidence_localization":
                        continue
                    references.append({
                        "composition_id": row["composition_id"],
                        "composition_kind": row["composition_kind"],
                        "dataset_class_id": row["dataset_class_id"],
                        "declaration_status": row["declaration_status"],
                        "original_policy_code": code,
                        "execution_status": "not_executed",
                        "requires_original_evidence_guards": True,
                        "reference_scope": "separate_positive_target_only; unlabelled_reference_has_no_finding_assertion" if state == "unlabelled_neighbor" else "original_task_and_evidence_state_scope",
                    })
            policy["semantic_compositions"] = references
            policy["semantic_composition_policy"] = {
                "changes_expected_policy_code": False,
                "promotes_review_to_applicable": False,
                "creates_clinical_observations": False,
                "execution_status": "not_executed",
            }
    if _sc_policy_codes(result) != original_policy or _sc_hash(_sc_policy_codes(enriched_contract)) != original_hash:
        raise AssertionError("Semantic enrichment changed the original task/evidence policy")
    capabilities = sorted({r["capability_id"]: r for r in capabilities}.values(), key=lambda r: r["capability_id"])
    audit = {
        "base_policy_sha256": original_hash, "base_policy_unchanged": True,
        "composition_count": len(compositions), "evidence_capability_count": len(capabilities),
        "compositions_sha256": _sc_hash(compositions),
        "execution_status": "not_executed", "qa_generated": 0,
        "clinical_observations_created": 0, "mrcm_validation_executed": False,
        "snomed_relationships_imported": len(seen_relation_ids),
        "snomed_manifest": _sc_clean(subset.get("manifest", {})),
        "available_source_fields": sorted(schema_fields),
    }
    return result, compositions, capabilities, audit

CONTRACT_BINDING_RESULT = enrich_terminology_contract(BASE_CONTRACT, CLASSES, CROSSWALK, ATTRIBUTE_CATALOG, SOURCE_STATUS)
ENRICHED_CONTRACT = CONTRACT_BINDING_RESULT["enriched_contracts"]
SNOMED_COMPOSITION_SOURCE = ({**MAPPING_RESULT["snomed_subset"], "manifest": SNOMED_SNAPSHOT["manifest"]}
                             if SNOMED_SNAPSHOT is not None else None)
ENRICHED_CONTRACT, composition_rows, capability_rows, COMPOSITION_AUDIT = build_semantic_compositions(
    ENRICHED_CONTRACT, CROSSWALK, ATTRIBUTE_CATALOG, OBSERVATIONS,
    CORPUS.schemas, SNOMED_COMPOSITION_SOURCE,
)
SEMANTIC_COMPOSITIONS = pd.DataFrame(composition_rows)
EVIDENCE_CAPABILITIES = pd.DataFrame(capability_rows)
TASK_SPECIFICATION = ENRICHED_CONTRACT["TASK_SPECIFICATION"]
EVIDENCE_STATE_MEANINGS = ENRICHED_CONTRACT["EVIDENCE_STATE_MEANINGS"]
VARIANT_SPECIFICATION = ENRICHED_CONTRACT["VARIANT_SPECIFICATION"]
TASK_EVIDENCE_POLICY = ENRICHED_CONTRACT["TASK_EVIDENCE_POLICY"]
S8_HINT_SPECIFICATION = ENRICHED_CONTRACT["S8_HINT_SPECIFICATION"]
CONTRACT_TERMINOLOGY_BINDINGS = pd.DataFrame(CONTRACT_BINDING_RESULT["binding_rows"])
COVERAGE_CONTRACT = pd.DataFrame(matrix_from_enriched_contract(ENRICHED_CONTRACT, BASE_COVERAGE_CONTRACT))
assert COVERAGE_CONTRACT["expected_status"].equals(BASE_COVERAGE_CONTRACT["expected_status"])
print("Preserved", len(COVERAGE_CONTRACT), "expected coverage decisions; terminology schema bindings:", len(CONTRACT_TERMINOLOGY_BINDINGS))
display(CONTRACT_TERMINOLOGY_BINDINGS[["owner_kind", "owner_key", "clinical_slot", "normalized_label", "terminology", "target_term", "mapping_status"]].head(30))
display(pd.DataFrame([{"evidence_state": state, "clinical_slots": entry["clinical_slots"], "bindings": len(entry["terminology_binding_ids"])} for state, entry in EVIDENCE_STATE_MEANINGS.items()]))
display(EVIDENCE_CAPABILITIES[["normalized_label", "capability", "supported_observation_count", "status"]])
print("Semantic compositions declared, not executed:", len(SEMANTIC_COMPOSITIONS))


Preserved 891 expected coverage decisions; terminology schema bindings: 4335


,owner_kind,owner_key,clinical_slot,normalized_label,terminology,target_term,mapping_status
0,task_evidence_policy,evidence_frame_localization/label_with_box,observed_finding,erythema,SNOMED_CT,Erythema,approved
1,s8_hint,verified_reference_image_comparison,normal_reference_finding,erythema,SNOMED_CT,Erythema,approved
2,task_evidence_policy,optional_spatial_localization/unmapped_or_conf...,indicated_region_finding,blood fresh,CEST,,source_required
3,task_evidence_policy,evidence_frame_localization/annotation_assiste...,observed_finding,blood hematin,SNOMED_CT,Hematin,approved
4,variant,temporal_collage,same_finding_positive_anchors,ampulla of vater,MST,Major papilla,approved
5,s8_hint,indicated_region_interpretation,indicated_region_finding,ulcer,SNOMED_CT,Ulcer,approved
6,variant,comparison,observed_finding,lymphangiectasia,MST,Lymphangiectasia,approved
7,variant,progression,positive_target_finding,lymphangiectasia,CEST,,source_required
8,task_evidence_policy,optional_spatial_localization/annotation_assis...,indicated_region_finding,ulcer,MST,Ulcer,approved
9,task_evidence_policy,temporal_relative_evidence_localization/unlabe...,positive_target_finding,foreign body,MST,Foreign body,approved


,evidence_state,clinical_slots,bindings
0,label_without_box,[observed_finding],51
1,label_with_box,"[observed_finding, indicated_region_finding]",102
2,multiple_labels,[observed_finding_set],51
3,missing_category,[observed_finding],51
4,unlabelled_neighbor,[],0
5,explicit_normal_reference,[normal_reference_finding],51
6,multiple_verified_anchors,[same_finding_positive_anchors],51
7,annotation_assisted_input,[indicated_region_finding],51
8,unmapped_or_conflicting_reference,[unresolved_finding_mapping],51


,normalized_label,capability,supported_observation_count,status
0,erythema,verified_clinical_attribute:size_mm,0,not_available_in_verified_observations
1,pylorus,original_dataset_category,1529,available
2,blood hematin,verified_clinical_attribute:grade,0,not_available_in_verified_observations
3,polyp,attribute_comparison:Size,0,verified_attribute_values_required
4,ulcer,source_geometry_reference,854,raw_reference_available
...,...,...,...,...
126,blood hematin,attribute_comparison:Type,0,verified_attribute_values_required
127,blood hematin,verified_clinical_attribute:shape,0,not_available_in_verified_observations
128,blood hematin,verified_clinical_attribute:severity,0,not_available_in_verified_observations
129,angiectasia,verified_clinical_attribute:grade,0,not_available_in_verified_observations


Semantic compositions declared, not executed: 101


In [11]:
ulcer_rows = CROSSWALK.loc[
    CROSSWALK["normalized_label"].eq("ulcer")
]
snomed_ulcer = ulcer_rows.loc[
    ulcer_rows["terminology"].eq("SNOMED_CT")
]

display(ulcer_rows[[
    "terminology", "target_term", "target_type",
    "relation", "mapping_status"
]])

for semantic_tag, relation in {
    "disorder": "exact",
    "morphologic abnormality": "related",
}.items():
    rows = snomed_ulcer.loc[
        snomed_ulcer["target_type"].eq(semantic_tag)
    ]

    if not (
        len(rows) == 1
        and rows["mapping_status"].eq("approved").all()
        and rows["relation"].eq(relation).all()
    ):
        display(RAG_MAPPING_AUDIT.loc[
            RAG_MAPPING_AUDIT["normalized_label"].eq("ulcer")
        ])
        raise ValueError(
            f"Verificarea ulcer a eșuat pentru {semantic_tag}."
        )

previous_term_output = TERM_OUTPUT

RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ")
RUN_DIR = REPORT_DIR = (
    WORK / "outputs/phase3/sector1/runs" / RUN_ID
)
TERM_OUTPUT = RUN_DIR / "terminology"
TERM_OUTPUT.mkdir(parents=True, exist_ok=False)

for name in (
    "snomed_rf2_component_inventory.json",
    "snomed_metadata_manifest.json",
    "snomed_description_audit.json",
):
    source = previous_term_output / name
    if source.is_file():
        shutil.copy2(source, TERM_OUTPUT / name)

write_json(
    RUN_DIR / "rag_source_provenance.json",
    RAG_SOURCE_PROVENANCE,
)
write_json(TERM_OUTPUT / "mapping_rerun_provenance.json", {
    "previous_terminology_directory": str(previous_term_output),
    "reused_imported_snapshots": True,
    "rag_source_sha256": RAG_SOURCE_PROVENANCE["vm_sha256"],
})

,terminology,target_term,target_type,relation,mapping_status
46,CEST,,,unmapped,source_required
47,MST,Ulcer,diagnosis,related,approved
48,MST,Ulcer,finding,exact,approved
49,SNOMED_CT,Ulcer,disorder,exact,approved
50,SNOMED_CT,Ulcer,morphologic abnormality,related,approved


## 10. Save terminology tables, contract bindings and source coverage

Write Parquet and JSON lookup tables, the enriched contract, the expected coverage matrix, source fingerprints, parse issues and editable input templates. The local destination is the current run's `outputs/phase3/sector1/runs/<run_id>/terminology` directory. In Colab, copy the completed terminology bundle to the same relative location under the S8 storage root and verify every copied file hash.

The report distinguishes source availability, automatically accepted mappings, unresolved associations and native-only classes. `snomed_retrieval_audit` records the actual search seeds and outcomes; `rag_mapping_audit` records concept-level reasons, and `rag_synonym_audit` records source-description acceptance. `distinct_class_vocabulary` preserves all contributing provenance in grouped entries. `ready_for_generation` remains false because QA generation is outside this stage. No old QA files are overwritten or deleted, and nothing is published to GitHub by this notebook.


In [12]:
TABLES_TO_SAVE = {
    "rag_mapping_audit": RAG_MAPPING_AUDIT,
    "rag_synonym_audit": RAG_SYNONYM_AUDIT,
    "distinct_class_vocabulary": DISTINCT_CLASS_VOCABULARY,
    "snomed_retrieval_audit": SNOMED_RETRIEVAL_AUDIT,
    "phase2_classes": CLASSES,
    "phase2_annotation_provenance": OBSERVATIONS,
    "phase2_inventory_issues": pd.DataFrame(PHASE2_INVENTORY["issues"]),
    "terminology_terms": TERM_CATALOG,
    "terminology_attributes": ATTRIBUTE_CATALOG,
    "source_attested_synonyms": SOURCE_SYNONYMS,
    "class_crosswalk": CROSSWALK,
    "mst_proposal_audit": MAPPING_RESULT["mst_proposal_audit"],
    "approved_class_synonyms": APPROVED_CLASS_SYNONYMS,
    "mst_applicability": SOURCE_BUNDLES["MST"]["applicability"],
    "mst_source_cells": SOURCE_BUNDLES["MST"]["raw_cells"],
    "mst_parse_issues": SOURCE_BUNDLES["MST"]["issues"],
    "mst_label_alignment": SOURCE_BUNDLES["MST"]["label_alignment"],
    "mst_labels_missing_words": SOURCE_BUNDLES["MST"]["labels_missing_words"],
    "mst_labels_unresolved": SOURCE_BUNDLES["MST"]["labels_unresolved"],
    "mst_attribute_issues": SOURCE_BUNDLES["MST"]["attribute_issues"],
    "coverage_contract": COVERAGE_CONTRACT,
    "contract_terminology_bindings": CONTRACT_TERMINOLOGY_BINDINGS,
    "semantic_compositions": SEMANTIC_COMPOSITIONS,
    "phase2_evidence_capabilities": EVIDENCE_CAPABILITIES,
}
if SNOMED_SNAPSHOT is not None:
    TABLES_TO_SAVE["snomed_metadata_audit"] = SNOMED_SNAPSHOT["metadata_audit"]
    TABLES_TO_SAVE["snomed_description_issues"] = SNOMED_SNAPSHOT["description_issues"]
    TABLES_TO_SAVE["snomed_candidates"] = MAPPING_RESULT["snomed_candidates"]
    for name, table in MAPPING_RESULT["snomed_subset"].items():
        TABLES_TO_SAVE["snomed_subset_" + name] = table
for name, table in TABLES_TO_SAVE.items():
    terminology_write_table(TERM_OUTPUT, name, table)
write_json(TERM_OUTPUT / "source_manifest.json", SOURCE_MANIFESTS)
shutil.copy2(RAG_SOURCE_PROVENANCE["local_path"], TERM_OUTPUT / "rag_evidence_package.json")
shutil.copy2(RAG_SOURCE_PROVENANCE["manifest_local_path"], TERM_OUTPUT / "rag_evidence_manifest.json")
write_json(TERM_OUTPUT / "rag_source_provenance.json", RAG_SOURCE_PROVENANCE)
# Preserve the original verified package/manifest; save the effective policy separately.
write_json(TERM_OUTPUT / "rag_effective_evidence_package.json", RAG_RUNTIME_EVIDENCE_PACKAGE)
write_json(TERM_OUTPUT / "rag_sources.json", RAG_EVIDENCE_PACKAGE["sources"])
write_json(TERM_OUTPUT / "rag_passages.json", RAG_EVIDENCE_PACKAGE["passages"])
write_json(TERM_OUTPUT / "rag_class_profiles.json", RAG_RUNTIME_EVIDENCE_PACKAGE["profiles"])
write_json(TERM_OUTPUT / "rag_mapping_decisions.json", RAG_RESULT["reviews"])
write_json(TERM_OUTPUT / "rag_review_summary.json", RAG_RESULT["summary"])
write_json(TERM_OUTPUT / "source_staging_manifest.json", SOURCE_STAGING)
write_json(TERM_OUTPUT / "source_acquisition_manifest.json", SOURCE_ACQUISITION)
write_json(TERM_OUTPUT / "ontology_manifest.json", ONTOLOGY_STORAGE_MANIFEST)
write_json(TERM_OUTPUT / "phase2_source_snapshot.json", CORPUS.source_snapshot)
write_json(TERM_OUTPUT / "phase2_metadata_validation.json", CORPUS.validation_report)
write_json(TERM_OUTPUT / "phase2_inventory_summary.json", PHASE2_INVENTORY["summary"])
write_json(TERM_OUTPUT / "task_contract_enriched.json", {
    "schema_version": "mmvqa-terminology-contract-v1", "base_coverage_spec_version": COVERAGE_SPEC_VERSION,
    **ENRICHED_CONTRACT, "TEMPORAL_RELATIVE_COMPLETENESS_SPECIFICATION": TEMPORAL_RELATIVE_COMPLETENESS_SPECIFICATION,
    "execution_status": CONTRACT_EXECUTION_STATUS,
})
write_json(TERM_OUTPUT / "contract_binding_audit.json", CONTRACT_BINDING_RESULT["audit"])
write_json(TERM_OUTPUT / "semantic_composition_audit.json", COMPOSITION_AUDIT)
SAFE_TERMINOLOGY_CONFIG = {**TERMINOLOGY_CONFIG, "downloads": {
    key: sanitize_acquisition_spec(spec) for key, spec in TERMINOLOGY_CONFIG["downloads"].items()
}}
write_json(TERM_OUTPUT / "configuration.json", {"source": CONFIG, "terminology": SAFE_TERMINOLOGY_CONFIG, "rag_github": RAG_GITHUB_CONFIG, "gi_github": GI_GITHUB_CONFIG})
write_json(TERM_OUTPUT / "templates" / "cest_package.template.json", empty_cest_package_template())
write_json(TERM_OUTPUT / "templates" / "reviewed_class_mappings.template.json", terminology_review_template(CROSSWALK))
(TERM_OUTPUT / "sources").mkdir(parents=True, exist_ok=True)
shutil.copy2(GI_SOURCE_PROVENANCE["local_path"], TERM_OUTPUT / "sources" / "capsule_vascular_nomenclature_2019.json")
shutil.copy2(GI_SOURCE_PROVENANCE["manifest_local_path"], TERM_OUTPUT / "gi_supplement_manifest.json")
write_json(TERM_OUTPUT / "gi_source_provenance.json", GI_SOURCE_PROVENANCE)
shutil.copy2(MST_PATH, TERM_OUTPUT / "sources" / "mst30_tables.xls")
(TERM_OUTPUT / "MST_COPYRIGHT.txt").write_text(MST_COPYRIGHT_NOTICE + "\n", encoding="utf-8")
if "CEST" in SOURCE_BUNDLES:
    (TERM_OUTPUT / "CEST_COPYRIGHT.txt").write_text(SOURCE_MANIFESTS["CEST"]["copyright_notice"] + "\n", encoding="utf-8")
if MAPPING_REVIEWS is not None:
    write_json(TERM_OUTPUT / "mapping_reviews_applied.json", MAPPING_REVIEWS)
mapping_counts = CROSSWALK.groupby(["terminology", "mapping_status"]).size().reset_index(name="rows")
source_complete = all(SOURCE_STATUS.get(name) == "imported" for name in ("MST", "CEST", "SNOMED_CT"))
TERMINOLOGY_SUMMARY = {
    "stage": "terminology_mapping", "code_version": CODE_VERSION,
    "notebook_base_commit": CONFIG["notebook_base_commit"],
    "phase2_source_commit": RESOLVED_COMMIT,
    "phase2_class_count": len(CLASSES), "phase2_annotation_count": len(OBSERVATIONS),
    "source_status": SOURCE_STATUS, "required_sources_imported": source_complete,
    "source_artifacts_verified": len(SOURCE_STAGING),
    "source_integrity_policy": "Drive_SHA256_reference_then_verified_VM_copy",
    "source_acquisition_counts": dict(Counter(record["acquisition"] for record in SOURCE_ACQUISITION.values())),
    "ontology_manifest_path": str(ONTOLOGY_MANIFEST_PATH),
    "snomed_language_refset_ids": SNOMED_SNAPSHOT["manifest"]["language_refset_ids"] if SNOMED_SNAPSHOT is not None else [],
    "snomed_metadata_ids": SNOMED_SNAPSHOT["manifest"]["metadata_ids"] if SNOMED_SNAPSHOT is not None else {},
    "snomed_metadata_discovery_policy_version": SNOMED_SNAPSHOT["manifest"]["metadata_discovery_policy_version"] if SNOMED_SNAPSHOT is not None else None,
    "gi_source_ref": GI_GITHUB_CONFIG["ref"],
    "gi_source_sha256": GI_SOURCE_PROVENANCE["source_sha256"],
    "gi_source_integrity": "VM_SHA256_matches_GitHub_manifest",
    "snomed_source_description_validation_status": SNOMED_SNAPSHOT["manifest"]["source_description_validation_status"] if SNOMED_SNAPSHOT is not None else "source_required",
    "snomed_description_issue_count": SNOMED_SNAPSHOT["manifest"]["description_issue_count"] if SNOMED_SNAPSHOT is not None else 0,
    "snomed_ambiguous_concept_count": SNOMED_SNAPSHOT["manifest"]["ambiguous_concept_count"] if SNOMED_SNAPSHOT is not None else 0,
    "terminology_term_count": len(TERM_CATALOG), "attribute_schema_rows": len(ATTRIBUTE_CATALOG),
    "mapping_status_counts": mapping_counts.to_dict("records"),
    "approved_class_synonym_rows": len(APPROVED_CLASS_SYNONYMS),
    "distinct_class_term_language_entries": len(DISTINCT_CLASS_VOCABULARY),
    "distinct_synonym_surface_forms": int(APPROVED_CLASS_SYNONYMS["term"].map(rag_lexical_key).nunique()),
    "synonym_policy": "active_source_accepted_descriptions_of_approved_exact_concepts",
    "rag_mapping_policy_version": RAG_EVIDENCE_PACKAGE.get("policy_version"),
    "automatic_mapping_review": RAG_RESULT["summary"],
    "rag_source_repository": RAG_GITHUB_CONFIG["repository"],
    "rag_source_ref": RAG_GITHUB_CONFIG["ref"],
    "rag_source_integrity": "VM_SHA256_matches_GitHub_manifest",
    "manual_review_required": False,
    "explicit_review_override_applied": REVIEW_PATH is not None,
    "unresolved_or_missing_mapping_rows": int(CROSSWALK["mapping_status"].isin(["candidate_review_required", "source_required", "unmapped"]).sum()),
    "mst_parse_issue_count": len(SOURCE_BUNDLES["MST"]["issues"]),
    "mst_label_alignment_counts": dict(Counter(SOURCE_BUNDLES["MST"]["label_alignment"]["link_status"])),
    "mst_labels_missing_words_count": len(SOURCE_BUNDLES["MST"]["labels_missing_words"]),
    "mst_labels_unresolved_count": len(SOURCE_BUNDLES["MST"]["labels_unresolved"]),
    "coverage_contract_rows": len(COVERAGE_CONTRACT),
    "coverage_observed_status": "not_evaluated",
    "contract_binding_rows": len(CONTRACT_TERMINOLOGY_BINDINGS),
    "semantic_composition_declarations": len(SEMANTIC_COMPOSITIONS),
    "semantic_compositions_executed": False,
    "qa_generated": 0, "llm_called": False, "mrcm_validation_performed": False,
    "clinical_observations_created": 0, "ready_for_generation": False,
    "stage_status": "sources_imported_automatic_mapping_review_complete" if source_complete else "automatic_mapping_review_complete_partial_sources",
}
write_json(TERM_OUTPUT / "summary.json", TERMINOLOGY_SUMMARY)
file_manifest = []
for path in sorted(TERM_OUTPUT.rglob("*")):
    if path.is_file() and path.name != "files_manifest.json":
        size, digest = file_digest(path)
        file_manifest.append({"relative_path": path.relative_to(TERM_OUTPUT).as_posix(), "bytes": size, "sha256": digest})
write_json(TERM_OUTPUT / "files_manifest.json", file_manifest)
saved_directory = TERM_OUTPUT
if TERMINOLOGY_CONFIG["save_to_drive"]:
    destination = STORAGE_ROOT / "outputs" / "phase3" / "sector1" / "runs" / RUN_ID / "terminology"
    if destination.resolve() != TERM_OUTPUT.resolve():
        shutil.copytree(TERM_OUTPUT, destination, dirs_exist_ok=False)
        for record in file_manifest:
            verify_file(destination / record["relative_path"], record["bytes"], record["sha256"])
        verify_file(destination / "files_manifest.json", *file_digest(TERM_OUTPUT / "files_manifest.json"))
        saved_directory = destination
display(mapping_counts)
print(json.dumps(TERMINOLOGY_SUMMARY, indent=2, ensure_ascii=False))
print("Saved terminology bundle:", saved_directory)
print("No QA generation, estimation or full candidate-corpus audit was executed.")


,terminology,mapping_status,rows
0,CEST,source_required,14
1,MST,approved,13
2,MST,rejected,1
3,MST,unmapped,2
4,SBCE_VASCULAR_DELPHI_2019,approved,1
5,SNOMED_CT,approved,18
6,SNOMED_CT,unmapped,2


{
  "stage": "terminology_mapping",
  "code_version": "phase3-sector1-terminology-v1.4.6",
  "notebook_base_commit": "35ffcb93020f84fde4bab7896c1a75e6ea322f7e",
  "phase2_source_commit": "fd5c5c1ab660bc1518d18b1500882ab458e6419f",
  "phase2_class_count": 14,
  "phase2_annotation_count": 47239,
  "source_status": {
    "MST": "imported",
    "CEST": "source_required",
    "SNOMED_CT": "imported",
    "SBCE_VASCULAR_DELPHI_2019": "imported"
  },
  "required_sources_imported": false,
  "source_artifacts_verified": 2,
  "source_integrity_policy": "Drive_SHA256_reference_then_verified_VM_copy",
  "source_acquisition_counts": {
    "existing_storage": 2,
    "not_attempted": 2
  },
  "ontology_manifest_path": "/content/drive/MyDrive/MMVQA_Clinical/inputs/terminology/ontology_manifest.json",
  "snomed_language_refset_ids": [
    "19491000087109",
    "20581000087109",
    "229851000087100",
    "229861000087102",
    "900000000000508004",
    "900000000000509007"
  ],
  "snomed_metadata_ids":

## Changes relative to the previous notebook

| Previous content | Current treatment |
|---|---|
| Published S8 source verification and metadata validation | Retained; image preview/staging removed from this terminology-only run. |
| S8-to-MMVQA field traceability | Replaced by per-class and annotation-level terminology provenance. |
| Task definitions and independent coverage specification | Retained and enriched with terminology bindings; expected matrix preserved. |
| Generator functions and hard-coded question templates | Removed. |
| Rule Validation with Independent Reference Examples | Removed. |
| Estimation Report Before Final Generation | Removed. |
| Generation and Full Audit of the Candidate QA Corpus | Removed. |
| Saved old QA outputs and execution counts | Cleared. |
| Fixed MST SHA and hard-coded language refsets | Replaced with Drive source manifests, verified VM staging and RF2 refset discovery. |
| Manual first-time placement of source files | Missing inputs download to Drive from configured external sources; existing inputs skip external requests. |

MST downloads automatically from its official public source when absent from Drive. To run with all sources, configure the authorized SNOMED RF2 URL and CEST transcription/document URLs once in section 1 (prefer Colab Secret names for protected URLs and authentication), or reuse existing Drive files. SHA-256 values and available language refsets are derived from the input bytes and retained in `ontology_manifest.json`. Section 1 now fetches and verifies the RAG evidence package from GitHub main; section 8 applies it automatically and exports its sources, passages, decisions and per-candidate audit. No manual medical review file is needed. Accepted decisions remain tied to source versions, are labelled AI-assisted, and do not certify image-level truth. The original task/evidence rules and expected coverage matrix remain unchanged.
